# 第二阶段 S2-P19：C0、C1 与 I0 的 SAC 训练代码

面向想读懂近期三条训练路线的同学。顺序是：确认冻结版本 → 看状态、动作与奖励 → 看网络和双 Q 更新 → 看真实训练循环、验证与断点。代码格是正式源码的完整函数或类，不是演示用的假训练循环。

**版本边界：**C0/C1 的原科学提交是 `47da2399`，10 万步后续跑的运行提交是 `48c7c4fe`，其中修复了回放束数表与满环断点；I0 的原科学提交也是 `47da2399`，运行提交是 `62e33c1b`，加入日志与预算/验证修复。下面始终按运行清单使用运行提交。三个任务均已结案；本笔记本不提交工作台请求。

**运行方式：**首格的 `ROUTE` 默认为 `C0`，改为 `C1` 可看同一代码架构但不同示范配置。读 I0 时改成 `I0`，**重启内核**再 Run All，防止两个冻结包同名模块混在内存中。Run All 只解包冻结源码、逐字节核验并定义函数和类；不会调用训练、验证或正式评价。


In [1]:
from pathlib import Path, PurePosixPath
import hashlib, importlib, io, json, subprocess, sys, zipfile
from IPython.core.magic import register_cell_magic

ROUTE = "C0"  # 可改为 C1；阅读 I0 请改为 I0 后重启内核
assert ROUTE in {"C0", "C1", "I0"}
ROOT = Path(r"D:/paper")
OUT = ROOT / "outputs/training_notebooks_20261012/stage2"
OUT.mkdir(parents=True, exist_ok=True)
C_COMMIT = "48c7c4feac71a6da9090d2274cbce73c9cb78e2f"
I_COMMIT = "62e33c1baa066c2a577a351cd18ecb487f4e11e8"
COMMIT = I_COMMIT if ROUTE == "I0" else C_COMMIT
SNAPSHOT = OUT / "snapshots" / COMMIT
marker = SNAPSHOT / ".source_commit"
source_root = (SNAPSHOT / "src").resolve()
for name, module in tuple(sys.modules.items()):
    if name == "satellite_rl" or name.startswith("satellite_rl."):
        module_file = getattr(module, "__file__", None)
        if module_file is None or not Path(module_file).resolve().is_relative_to(source_root):
            raise RuntimeError("内核已加载另一版本 satellite_rl；请重启内核后 Run All")

# 每次都从 Git 读取完整归档。marker 只标识目录，不能证明其中的文件没有被改动。
archive = subprocess.check_output(["git", "archive", "--format=zip", COMMIT, "src/"], cwd=ROOT)
expected_files = {}
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    for item in z.infolist():
        relative = PurePosixPath(item.filename)
        if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "src":
            raise ValueError("unsafe archive member")
        if item.is_dir():
            continue
        key = relative.as_posix()
        if key in expected_files:
            raise ValueError(f"duplicate archive member: {key}")
        expected_files[key] = z.read(item)
if not expected_files:
    raise RuntimeError("冻结源码归档为空")
if not SNAPSHOT.exists():
    SNAPSHOT.mkdir(parents=True)
    for relative, payload in expected_files.items():
        target = SNAPSHOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
    marker.write_text(COMMIT, encoding="ascii")
if not marker.is_file() or marker.read_text(encoding="ascii") != COMMIT:
    raise RuntimeError("冻结源码目录缺少正确 marker；不会覆盖或删除已有文件")
for relative, payload in expected_files.items():
    target = SNAPSHOT / relative
    if not target.is_file() or target.read_bytes() != payload:
        raise RuntimeError(f"冻结源码文件与 Git 不一致：{relative}")
expected_py = {name for name in expected_files if name.endswith(".py")}
actual_py = {path.relative_to(SNAPSHOT).as_posix() for path in SNAPSHOT.rglob("*.py")}
if actual_py != expected_py:
    raise RuntimeError(f"冻结目录中的 Python 文件集合不一致；额外={sorted(actual_py - expected_py)}, 缺失={sorted(expected_py - actual_py)}")
sys.path.insert(0, str(source_root))
import satellite_rl
assert Path(satellite_rl.__file__).resolve().is_relative_to(source_root)
print("路线", ROUTE, "冻结源码", COMMIT[:12], "包路径", satellite_rl.__file__)


路线 C0 冻结源码 48c7c4feac71 包路径 D:\paper\outputs\training_notebooks_20261012\stage2\snapshots\48c7c4feac71a6da9090d2274cbce73c9cb78e2f\src\satellite_rl\__init__.py


## 先看三份实际训练配置

| 路线 | 训练动作 | 网络/更新 | 额外来源 | 已完成交互 |
| --- | --- | --- | --- | --- |
| C0 | 逐束选频率起点、槽数、功率档位 | Transformer 逐束编码；**factorized 策略 + additive 双 Q**；固定 α=0.01、γ=1 | 无教师；在线回放 | 400000 逐束步 |
| C1 | 与 C0 相同 | 与 C0 同结构、同种子 51001、同 400000 步上限 | 教师转移占每次更新批量的一半，在线交互仍单独计数 | 400000 逐束步 |
| I0 | 贪心完整初解上选候选修改或 STOP | 候选打分 actor + 双 Q；固定 α=0.01、γ=1 | 非学习贪心初解；每次候选做完整物理核验 | 397112 宏步，触及 12500 回合上限 |

C0/C1 都用 `terminal_si_delta` 奖励、batch=256、回放容量 20000、预热 2000 步；I0 用 SI 改进奖励、batch=128、最多 16 候选/32 决策/32 物理探测。三份训练场景各 8064 景，固定验证各 72 景。**实际配置和数据入口**在下格读取的每个工作台归档 `mirror/job/config.json` 与 `mirror/job/job.json` 的 `scene_sets`；后者保存每景路径和 SHA。运行清单 `outputs/experiments/sac_routes_s2p19_20261007/run_manifest.json` 给出任务 ID、源码与配置 SHA。改 `ROUTE` 只切换阅读版本，不会取来场景或重放已完成训练。


In [2]:
manifest = json.loads((ROOT / "outputs/experiments/sac_routes_s2p19_20261007/run_manifest.json").read_text(encoding="utf-8"))
archive_root = ROOT / "outputs/experiments/sac_routes_s2p19_20261007/remote_archive"
frozen_configs = {}
for arm in ("C0", "C1", "I0"):
    task = manifest["tasks"][arm]
    matches = list(archive_root.glob(f"*/{task['run_id']}/*/mirror/job/config.json"))
    assert len(matches) == 1, (arm, matches)
    config_path = matches[0]
    assert hashlib.sha256(config_path.read_bytes()).hexdigest() == task["config_sha256"]
    job_path = config_path.with_name("job.json")
    job = json.loads(job_path.read_text(encoding="utf-8"))
    frozen_configs[arm] = json.loads(config_path.read_text(encoding="utf-8"))
    counts = {name: group["count"] for name, group in job["scene_sets"].items()}
    print(arm, task["run_id"], "配置SHA已核", "场景", counts)


C0 train_sac_20261007_152717_479365fb 配置SHA已核 场景 {'train': 8064, 'validation': 72}
C1 train_sac_20261007_153047_1faf6d10 配置SHA已核 场景 {'train': 8064, 'validation': 72}


I0 train_sac_20261007_174121_7d0b687c 配置SHA已核 场景 {'train': 8064, 'validation': 72}


`workbench/environment.py` 与 `agents/workbench_networks.py` 在原科学提交、C0/C1 运行提交和 I0 运行提交之间逐字节相等。C0/C1 的 `workbench_algorithms.py` 在续跑版修了回放记录；I0 的 `sac_improvement/environment.py` 在运行版改过日志/保护细节。因此下文各格仍按本路线的**完整运行提交**取代码，不把几个提交的类拼在一起。


## 源码格如何工作

下一格注册 `%%project_module`。每个源码格会先把格内文字与冻结 Git 文件的对应行逐字节比较，再在该模块自己的命名空间定义函数或类。未选路线的格子只跳过执行，仍能直接阅读。其它辅助函数由同一冻结源码树正常导入，因此源码格不需要重抄整个项目。切换路线必须重启内核。


In [3]:
SOURCE_SPECS = {
  "C_ENV": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/environment.py",
    "module": "satellite_rl.workbench.environment",
    "start": 141,
    "end": 824,
    "sha256": "d42db0052c883a57747f3d7a16604dfbea6c8f93d40f55765840a262844c3456"
  },
  "C_ENCODER": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/agents/workbench_networks.py",
    "module": "satellite_rl.agents.workbench_networks",
    "start": 15,
    "end": 387,
    "sha256": "c7baf2336ab5785dd33508ceedb07e6db9984ee7f0cbc92d6e886569616249e7"
  },
  "C_NET": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/agents/workbench_networks.py",
    "module": "satellite_rl.agents.workbench_networks",
    "start": 390,
    "end": 398,
    "sha256": "4be393eee6f2ecbf7ba1cba3477c6b1a8768b730172f8e0cd57a086baa19fcd5"
  },
  "C_TEACHER": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/s2p19_demonstrations.py",
    "module": "satellite_rl.workbench.s2p19_demonstrations",
    "start": 45,
    "end": 105,
    "sha256": "f3d91598e5a2ae13dae50170f079e10451e8e707c5b2b6b39c86c198976bb96e"
  },
  "C_DEMO": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/s2p19_demonstrations.py",
    "module": "satellite_rl.workbench.s2p19_demonstrations",
    "start": 264,
    "end": 326,
    "sha256": "38db2ea240ee9e4d87bf7952dbb9ba49d4c0903a8c996540a312034a89398033"
  },
  "C_REPLAY": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/agents/workbench_algorithms.py",
    "module": "satellite_rl.agents.workbench_algorithms",
    "start": 169,
    "end": 458,
    "sha256": "fe776bbe8ca4b4f25d7b2e60da29e06ad2e69a24113178fb177bd0b95c5ccf44"
  },
  "C_AGENT": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/agents/workbench_algorithms.py",
    "module": "satellite_rl.agents.workbench_algorithms",
    "start": 461,
    "end": 1186,
    "sha256": "af2f1c537a75e29b926c6ab42576e853bc11705414318520eb4897c2416c4136"
  },
  "C_BUILD": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/agents/workbench_algorithms.py",
    "module": "satellite_rl.agents.workbench_algorithms",
    "start": 1189,
    "end": 1190,
    "sha256": "0249eec02cfa9fb1a07412aea58bf9f5bdd1360234f2550cb6f80a026f541904"
  },
  "C_SAVE": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/checkpoints.py",
    "module": "satellite_rl.workbench.checkpoints",
    "start": 264,
    "end": 278,
    "sha256": "102d5efd2252bd76af69f4c52ac25da2e4cd826da75b162bced9176ddfc6b7df"
  },
  "C_LOAD": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/checkpoints.py",
    "module": "satellite_rl.workbench.checkpoints",
    "start": 281,
    "end": 309,
    "sha256": "8c4dd84c1d46619e9c82e8a1a5f97e495eb1121d5ce4faebc86d2276a21d2707"
  },
  "C_VALID": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/runner.py",
    "module": "satellite_rl.workbench.runner",
    "start": 583,
    "end": 707,
    "sha256": "820ae06041d860218de5e8b4ebbb61c2fcec5acec62841d9c4694a0437cb64aa"
  },
  "C_TRAIN": {
    "routes": [
      "C0",
      "C1"
    ],
    "commit": "48c7c4feac71a6da9090d2274cbce73c9cb78e2f",
    "path": "src/satellite_rl/workbench/runner.py",
    "module": "satellite_rl.workbench.runner",
    "start": 728,
    "end": 1223,
    "sha256": "7d0f787aa55c5c667548811b7549b70ebc9cfbcd033d3da75a900a3cb2de4618"
  },
  "I_CONFIG": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/config.py",
    "module": "satellite_rl.sac_improvement.config",
    "start": 45,
    "end": 85,
    "sha256": "c9f7bd3a779e169c024b980a5c220c60e3fc1c79f96ee4b3c4b3823ccc8b3915"
  },
  "I_CANDIDATE": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/environment.py",
    "module": "satellite_rl.sac_improvement.environment",
    "start": 35,
    "end": 41,
    "sha256": "7963235f7a2062eb57f3e09013054401ada0d46601dbb9168fcc5aaa559ca517"
  },
  "I_ENV": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/environment.py",
    "module": "satellite_rl.sac_improvement.environment",
    "start": 44,
    "end": 477,
    "sha256": "49c2a1bd91701eee72cd50cb179f49f89d305e523136867ef2f88120781143a7"
  },
  "I_NET": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/agent.py",
    "module": "satellite_rl.sac_improvement.agent",
    "start": 21,
    "end": 34,
    "sha256": "2b4c8be56ace46ba7317007e6ab5764dd684215a8d1ed6968ef781ea5ad9507b"
  },
  "I_PACK": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/agent.py",
    "module": "satellite_rl.sac_improvement.agent",
    "start": 37,
    "end": 53,
    "sha256": "741d3faefe6a6eaf595f6fafab41e68c6c3a2bdd12b768e4d95db3fb7f8fa682"
  },
  "I_REPLAY": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/agent.py",
    "module": "satellite_rl.sac_improvement.agent",
    "start": 66,
    "end": 104,
    "sha256": "53a803d3cc2188548c4dce9fb4c818e1fbccd56509611ddf75793ac792a4e6fc"
  },
  "I_AGENT": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/agent.py",
    "module": "satellite_rl.sac_improvement.agent",
    "start": 107,
    "end": 198,
    "sha256": "81c238e841d35dd6d457f96274c6d4f9ad7f71203e481fe016c5693478760cfa"
  },
  "I_SAVE": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/runner.py",
    "module": "satellite_rl.sac_improvement.runner",
    "start": 147,
    "end": 160,
    "sha256": "d30143a2fe78ab161904506553a18a28c941b533db72280e39e79c341f85979e"
  },
  "I_LOAD": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/runner.py",
    "module": "satellite_rl.sac_improvement.runner",
    "start": 163,
    "end": 172,
    "sha256": "96a650a78b4924fe6fee387a3ef5c142f9512ca07cc709292bc4b8379913b742"
  },
  "I_VALID": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/runner.py",
    "module": "satellite_rl.sac_improvement.runner",
    "start": 497,
    "end": 568,
    "sha256": "ead1beb1d5a85f7f9f105668cdb6c7b91b6fa65b567cc92688ec706025dab456"
  },
  "I_TRAIN": {
    "routes": [
      "I0"
    ],
    "commit": "62e33c1baa066c2a577a351cd18ecb487f4e11e8",
    "path": "src/satellite_rl/sac_improvement/runner.py",
    "module": "satellite_rl.sac_improvement.runner",
    "start": 652,
    "end": 871,
    "sha256": "45937ec16ac3f5d0d27aaedb1026fb54df8fe468f8137c470238a8f210c50fbe"
  }
}

@register_cell_magic
def project_module(line, cell):
    key = line.strip()
    spec = SOURCE_SPECS[key]
    if ROUTE not in spec["routes"]:
        return None
    if COMMIT != spec["commit"]:
        raise RuntimeError("冻结版本不匹配；请修改 ROUTE 并重启内核")
    path = spec["path"]
    git_bytes = subprocess.check_output(["git", "show", f"{COMMIT}:{path}"], cwd=ROOT)
    lines = git_bytes.decode("utf-8").splitlines(keepends=True)
    expected = "".join(lines[spec["start"] - 1:spec["end"]])
    if cell != expected or hashlib.sha256(cell.encode()).hexdigest() != spec["sha256"]:
        raise RuntimeError(f"{key}: 代码格与冻结源码不一致")
    module = importlib.import_module(spec["module"])
    exec(compile(cell, f"{COMMIT}:{path}:{spec['start']}", "exec"), module.__dict__)
    print(f"已定义 {key}: {path}:{spec['start']}-{spec['end']}")


## C0 / C1：逐束构造 SAC

C0 根据当前已分配波束选择下一束的频率起点、槽数和功率档位。C1 使用同结构网络，更新时把在线回放和冻结教师转移按 1:1 混合；教师数据不会计入环境交互步数。状态含已分配布局与需求，动作掩码排除不可行组合。奖励由完整终态目标的势函数增量构造。


### 状态、联合可行动作掩码与奖励

先读 `_state`：它把当前波束、频谱占用和需求拼成观测；再读 `action_mask`，看哪些频率、槽数和功率组合可选。`step` 会记录请求动作与实际分配，`_reward` 再按完整终态 SI 的增量给奖励。读长类时可以先跳过物理细节，沿这四个方法走一遍。

下格是冻结源码的完整 `WorkbenchEnv` 定义，来源 `src/satellite_rl/workbench/environment.py:141-824`，提交 `48c7c4feac71`；运行时逐字节核对。


In [4]:
%%project_module C_ENV
class WorkbenchEnv:
    """Strict wrapper. The raw environment remains the source of physical outcomes."""

    def __init__(self, config):
        self.config = validate_config(config)
        self.summary = config_summary(self.config)
        self.region_profile = get_profile(self.config["environment"]["profile"])
        self.environment_metadata = profile_metadata(self.region_profile)
        self.scene_metadata = None
        self.raw_env = None
        self._scene = None
        self._trace = []
        self._last_legacy_state = None
        self._prior = None
        self._overload_seen = False
        self._mask_used_power = Decimal(0)
        self._previous_actual_action = np.zeros(4, dtype=np.float32)
        self._no_business = False

    def reset(self, seed=None, scene=None):
        max_beams = self.config["environment"]["beam_count_max"]
        if scene is None:
            scenario_seed = np.random.SeedSequence().generate_state(1)[0] if seed is None else seed
            with _seed_scope(scenario_seed):
                beam = legacy.BeamInfo(profile=self.region_profile, count=100)
            arrays = {key: np.array(beam.beam_info[key], copy=True) for key in SCENE_KEYS}
            arrays["original_indices"] = np.array(beam.shuffle_indices, copy=True)
            arrays = validate_scene(arrays, self.region_profile, max_beams=max_beams)
            scene_metadata = deepcopy(beam.generation_metadata) or dict(
                kind=self.region_profile.default_scene_kind, measured_traffic=False)
        else:
            arrays = validate_scene(scene, self.region_profile, max_beams=max_beams)
            scene_metadata = dict(kind="imported_scene", measured_traffic=None)
        if len(arrays["rate"]) == 0:
            self._scene = arrays
            self.scene_metadata = dict(scene_metadata, no_business=True)
            self.raw_env = None
            self._no_business = True
            self._trace = []
            self._prior = None
            return np.zeros(self.summary["state_dim"], dtype=np.float32)
        self._no_business = False
        if self.config["network"].get("graph_mode", "none") == "interference_bias":
            from satellite_rl.agents.graph_encoder import validate_physical_contract
            validate_physical_contract(arrays, self.config, {
                "beam_width_deg": legacy.bw,
                "carrier_start_hz": legacy.f_min,
                # The raw PHY uses this literal for every slot bandwidth term.
                "slot_bandwidth_hz": 2.5e7,
                "light_speed_m_s": legacy.c,
                "transmit_gain_dbi": float(arrays["beam_G_Tx_max"][0, 0]),
                "receive_gain_dbi": float(arrays["UE_G_Rx_max"][0, 0]),
                "latitude_bounds_deg": self.region_profile.latitude_bounds_deg,
                "longitude_bounds_deg": self.region_profile.longitude_bounds_deg,
                "satellite_longitude_deg": self.region_profile.satellite_longitude_deg,
                "earth_radius_km": self.region_profile.earth_radius_km,
                "orbit_radius_km": self.region_profile.orbit_radius_km,
            })
        self._scene = arrays
        self.scene_metadata = scene_metadata
        beam = SimpleNamespace(beam_info={key: np.array(arrays[key], copy=True) for key in SCENE_KEYS})
        frequency = legacy.SatelliteFreqState()
        current = legacy.CurrentBeamState(beam.beam_info, profile=self.region_profile)
        preset = self.config["reward"]["preset"]
        raw_variant = preset if preset in ("legacy_original", "legacy_repaired", "fused") else "legacy_repaired"
        budget = self.config["environment"]["power_budget_w"]
        with _budget_scope(budget):
            self.raw_env = legacy.Env(frequency, beam, current, reward_variant=raw_variant,
                                      basic_fraction=self.config["environment"]["basic_fraction"],
                                      profile=self.region_profile)
            self._last_legacy_state = self.raw_env.reset(beam)
        self._trace = []
        self._previous_actual_action = np.zeros(4, dtype=np.float32)
        self._overload_seen = False
        self._mask_used_power = Decimal(0)
        self._prior = self._snapshot()
        return self._state()

    def scene(self):
        self._require_reset()
        return {key: np.array(value, copy=True) for key, value in self._scene.items()}

    def _require_reset(self):
        if self.raw_env is None and not self._no_business:
            raise RuntimeError("请先 reset 环境")

    def _snapshot(self):
        env = self.raw_env
        count = env.beam_idx
        rates = np.maximum(np.asarray(env.beam_bps, dtype=np.float64), 0)
        demands = self._scene["rate"].reshape(-1)[:count]
        ratios = np.minimum(rates / demands, 1) if count else np.zeros(0)
        beta = float(self._scene["rate"].mean())
        all_ratios = np.zeros(env.beam_num, dtype=np.float64)
        all_ratios[:count] = ratios
        return dict(count=count, rates=rates.copy(), demands=demands.copy(), ratios=ratios,
                    si=float(ratios.sum()), mean_si=float(ratios.mean()) if count else 0.0,
                    basic=float(np.minimum(rates / demands / env.basic_fraction, 1).sum()) if count else 0.0,
                    sgm=float(_sgm(rates, demands, beta).sum()), throughput=float(rates.sum()) / 1e9,
                    useful_throughput=float(np.minimum(rates, demands).sum()) / 1e9,
                    jain=_jain(all_ratios), min_si=float(all_ratios.min()),
                    shortfall=float(self.raw_env.beam_num - ratios.sum()))

    def _state(self):
        env, legacy_state = self.raw_env, self._last_legacy_state
        idx = env.beam_idx
        count = env.beam_num
        parts = []
        base = {"beam": legacy_state[:3], "power": legacy_state[3:4],
                "interference": legacy_state[4:104], "occupancy": legacy_state[104:204]}
        ratios = self._snapshot()["ratios"]
        occupancy = env.satellite_freq_state.freq_pool_real
        group = idx % 8
        free = occupancy[group] == 0
        longest = max((len(x) for x in np.split(free, np.where(np.diff(free.astype(int)) != 0)[0] + 1) if len(x) and x[0]), default=0)
        power_budget = self.config["environment"]["power_budget_w"]
        # S2-P18 修正2：原来的 30 W 是 3000W/100束写死的“每束参考功率”，N≠100 时
        # 系统性低估大束数的资源紧张程度（方案18 第 2.4 节）。这里改成随束数变化的
        # 公平份额 budget/N；N=100 时 3000/100 与原来的 30 是同一个浮点数，因此
        # 100 束逐位不变。这是纠正参考量，不是新增信息。
        power_reference_w = power_budget / count
        # 当前组剩余预留义务：与掩码 action_mask 的 remaining_group 同一口径，
        # 即包含当前束在内、本复用组还需要处理多少束。
        count_remaining = count - idx
        group_remaining = len(range(idx, count, 8))
        # 剩余合法槽上限：与掩码 max_slots 同一口径，先数本组未被占用的位置，
        # 再扣掉为同组其余波束各预留的 1 槽；负值表示当前束已无槽可分。
        group_free_slots = float(np.count_nonzero(~(np.asarray(occupancy[group]) > 0)))
        slots_available = max(0.0, group_free_slots - (group_remaining - 1))
        definitions = {
            "progress": [idx / count], "remaining_count": [count_remaining / count],
            "group": [group / 7],
            "power_per_remaining": [(power_budget - env.total_power) / max(1, count_remaining) / 30],
            "served_shortfall": [float((1 - ratios).mean()) if len(ratios) else 0],
            "served_met": [float(np.mean(ratios >= 1)) if len(ratios) else 0],
            "served_min": [float(ratios.min()) if len(ratios) else 0],
            "other_occupancy": occupancy[[g for g in range(8) if g != group]].reshape(-1),
            "longest_free": [longest / 100],
            "previous_actual_action": self._previous_actual_action,
            "group_remaining_obligation": [group_remaining / count],
            "remaining_slot_limit": [slots_available / 100],
        }
        power_left = power_budget - env.total_power
        if power_left < -1e-7 and "power_pressure" in self.config["state"]["features"]:
            raise RuntimeError("剩余功率为负，资源账错误")
        remaining = count_remaining
        definitions.update(
            beam_count_normalized=[count / 200],
            remaining_beams_normalized=[remaining / 200],
            power_pressure=[power_left / (power_left + power_reference_w * remaining) if remaining else 0],
        )
        if self.config["state"]["demand_visibility"] == "known":
            demands = self._scene["rate"].reshape(-1)
            definitions.update(beta=[float(demands.mean()) / 6e8],
                               remaining_demand=[float(demands[idx:].sum()) / (count * 6e8)],
                               remaining_demand_mean=[float(demands[idx:].mean()) / 6e8 if idx < count else 0])
        features = self.config["state"]["features"]
        if "global_context" in features:
            definitions.update(self._structured_context(definitions, ratios))
        elif "natural_global_context" in features:
            context = self._structured_context(definitions, ratios)
            level = self.config["state"]["context_level"]
            definitions["natural_global_context"] = context["global_context"][:3 if level == "compact" else 10]
            if "natural_beam_context" in features:
                beam_rows = context["beam_context"]
                beam_width = context["beam_width"]
                definitions["natural_beam_context"] = (
                    beam_rows[:, :, :beam_width] if beam_rows.ndim == 3
                    else beam_rows[:, :beam_width])
            if "natural_resource_context" in features:
                definitions["natural_resource_context"] = context["resource_context"][:, :204 if level == "full" else 4]
        for fid in self.config["state"]["features"]:
            value = np.asarray(base[fid] if fid in base else definitions[fid], dtype=np.float32).reshape(-1)
            parts.append(value)
        state = np.concatenate(parts)
        if state.size != self.state_width(count) or not np.isfinite(state).all():
            raise ValueError("状态维度或数值无效")
        return state

    def state_width(self, count):
        """本场景实际观测宽度：仅“按实际束数构造”的对照臂随 N 变化。"""
        return natural_state_width(self.config, count)

    def _structured_context(self, definitions, ratios):
        """Build four visibility levels from information available before the action."""
        env = self.raw_env
        idx, count = env.beam_idx, env.beam_num
        level = self.config["state"]["context_level"]
        # S2-P18 手段G：可选按场景实际束数构造逐束字段（context_beam_count="actual"）。
        # 默认档位仍用固定 (10,10,11) 的 100 束布局，所以既有对照逐位不变。
        actual_count = self.config["network"].get("context_beam_count", "fixed_100") == "actual"
        beam_width = 5 if level == "layout" else 11
        # 网络输入宽度必须恒定：按实际束数构造时把逐束实体补零到场景束数上限，
        # 前 count 行为真实束、其余行是占位。占位不产生干扰、动作、奖励或统计，
        # 池化由编码器的有效位屏蔽排除（方案17 问答第 5 节的要求）。
        beam_rows = count if actual_count else 100
        if actual_count:
            beam_rows = int(self.config["environment"]["beam_count_max"])
            if count > beam_rows:
                raise RuntimeError("实际束数超过场景束数上限，无法构造固定宽度观测")
        global_context = np.zeros(10, dtype=np.float32)
        # 注意：这里始终按 11 通道分配，实际写入观测时再按档位截取到 beam_width 列，
        # 与历史实现（layout 档取前 5 列）保持一致。
        beam_context = np.zeros((beam_rows, 11), dtype=np.float32)
        resource_context = np.zeros((8, 204), dtype=np.float32)

        global_context[:3] = (definitions["beta"][0], definitions["progress"][0],
                              definitions["remaining_demand"][0])
        if level in ("summary", "layout", "full"):
            global_context[3:] = (
                definitions["remaining_count"][0], definitions["group"][0],
                definitions["power_per_remaining"][0], definitions["served_shortfall"][0],
                definitions["served_met"][0], definitions["served_min"][0],
                definitions["longest_free"][0],
            )
            if idx == count:
                # There is no next beam at the terminal state. Keep aggregate
                # service/resource records, but zero current-beam-only fields.
                global_context[[4, 5, 9]] = 0
            occupancy = np.asarray(env.satellite_freq_state.freq_pool_real)
            for group in range(8):
                occupied = occupancy[group] > 0
                free = ~occupied
                longest = max((len(x) for x in np.split(free, np.flatnonzero(np.diff(free.astype(int))) + 1)
                               if len(x) and x[0]), default=0)
                historical = [item for item in env.history_beam if item[1] == group and item[3] > 0]
                group_ids = np.arange(group, idx, 8)
                resource_context[group, :4] = (
                    float(occupied.mean()), longest / 100,
                    sum(10 ** (entry[4] / 10) for entry in historical)
                    / self.config["environment"]["power_budget_w"],
                    float(ratios[group_ids].mean()) if len(group_ids) else 0,
                )

        if level in ("layout", "full"):
            locations = self._scene["beam_location"]
            lat_min, lat_max = self.region_profile.latitude_bounds_deg
            lon_min, lon_max = self.region_profile.longitude_bounds_deg
            rows = beam_context.reshape(beam_rows, 11)[:, :beam_width]
            rows[:count, 0] = 2 * (locations[:, 0] - (lat_max + lat_min) / 2) / (lat_max - lat_min)
            rows[:count, 1] = 2 * (locations[:, 1] - (lon_max + lon_min) / 2) / (lon_max - lon_min)
            rate_mean = (legacy.rate_min + legacy.rate_max - legacy.rate_step) / 2 * 1e6
            demands = self._scene["rate"].reshape(-1)
            rows[:count, 2] = demands / rate_mean
            rows[:count, 3] = np.arange(count) % 8 / 7
            rows[:count, 4] = np.arange(count) / (count - 1)

        if level == "full":
            rows = beam_context.reshape(beam_rows, 11)[:, :beam_width]
            rows[:idx, 5] = 1
            rates = np.maximum(np.asarray(env.beam_bps, dtype=np.float64), 0)
            rows[:idx, 9] = rates / 6e8
            rows[:idx, 10] = ratios
            for beam_id, group, frequency, slots, power_db in env.history_beam:
                if not 0 <= beam_id < idx or group != beam_id % 8:
                    raise ValueError("历史波束记录与处理顺序不一致")
                rows[beam_id, 6:9] = (
                    frequency / 99, slots / 10,
                    (10 ** (power_db / 10) if slots > 0 else 0)
                    / max(self.config["environment"]["power_levels_w"]),
                )
            occupancy = np.asarray(env.satellite_freq_state.freq_pool_real, dtype=np.float32)
            resource_context[:, 4:104] = occupancy
            if idx < count:
                # Interference depends on the prospective group's polarization,
                # so the eight groups share two exact historical calculations.
                for polarization in (0, 1):
                    watts = env._calc_group_slot_interference(polarization)
                    db = 10 * np.log10(watts + 1e-30)
                    normalized = np.clip((db - legacy.P_MIN_DB) /
                                         (legacy.P_MAX_DB - legacy.P_MIN_DB), 0, 1)
                    resource_context[polarization::2, 104:204] = normalized
        return {"global_context": global_context,
                "beam_context": beam_context,
                "beam_width": beam_width,
                "resource_context": resource_context}

    def _project(self, requested):
        """Choose the first jointly feasible contiguous allocation from requested start onward."""
        env = self.raw_env
        budget_left = self.config["environment"]["power_budget_w"] - env.total_power
        levels = self.config["environment"]["power_levels_w"]
        group = env.beam_idx % 8
        occupied = env.satellite_freq_state.freq_pool_real[group]
        freq, slots, power_idx = requested
        candidate_powers = [i for i in range(power_idx, -1, -1) if levels[i] <= budget_left]
        for f in list(range(freq, 100)) + list(range(0, freq)):
            free_run = 0
            for k in range(f, min(100, f + slots)):
                if occupied[k] > 0:
                    break
                free_run += 1
            if free_run and candidate_powers:
                return f, free_run, candidate_powers[0]
        # No positive-slot feasible action exists. The wrapper records an explicit
        # no-service outcome without passing a zero requested length to legacy step.
        return freq, 0, power_idx

    def action_mask(self):
        """Exact legal pairs and power levels from the current resource ledger.

        This is independent of the configurable observation layout. Slot-head
        indices 0..9 represent actual lengths 1..10. Future beams are reserved
        one slot in their own group and one minimum configured power level.
        """
        self._require_reset()
        if self._no_business:
            return None
        if self.config["environment"].get("action_mask", "none") == "none":
            return None
        env = self.raw_env
        if env.beam_idx >= env.beam_num or self._trace and self._trace[-1]["done"]:
            return None
        remaining = env.beam_num - env.beam_idx
        remaining_group = len(range(env.beam_idx, env.beam_num, 8))
        occupied = np.asarray(env.satellite_freq_state.freq_pool_real[env.beam_idx % 8]) > 0
        max_slots = int(np.count_nonzero(~occupied)) - (remaining_group - 1)
        free_run = np.zeros(100, dtype=np.int64)
        run = 0
        for position in range(99, -1, -1):
            run = 0 if occupied[position] else run + 1
            free_run[position] = run
        lengths = np.arange(1, 11)
        frequency_slots = lengths[None, :] <= np.minimum(free_run, max_slots)[:, None]
        levels = self.config["environment"]["power_levels_w"]
        budget = float(self.config["environment"]["power_budget_w"])
        # Exact nominal watt ledger: a relative/absolute tolerance could admit a
        # genuinely unaffordable level for small/custom power grids. Actual dBW
        # conversion is conservatively below the selected nominal level at step.
        power_left = Decimal(str(budget)) - self._mask_used_power
        reserve = Decimal(str(levels[0])) * (remaining - 1)
        power = np.array([Decimal(str(level)) + reserve <= power_left for level in levels], dtype=bool)
        if not frequency_slots.any() or not power.any():
            raise RuntimeError(
                "无合法动作：不能在剩余资源下为每束保留一槽和最低功率；"
                f"beam={env.beam_idx}, free_slots={int((~occupied).sum())}, "
                f"remaining_group={remaining_group}, power_left={power_left:g}, "
                f"power_reserve={reserve:g}")
        return {"frequency_slots": frequency_slots, "power": power}

    def step(self, action_indices):
        self._require_reset()
        if self._no_business:
            raise RuntimeError("无业务场景没有可执行动作")
        if self.raw_env.beam_idx >= self.raw_env.beam_num or self._trace and self._trace[-1]["done"]:
            raise RuntimeError("回合已结束，请 reset")
        if not isinstance(action_indices, (tuple, list, np.ndarray)) or len(action_indices) != 3:
            raise ValueError("动作必须有频率起点、槽数索引、功率索引三个整数")
        dims = self.summary["action_dims"]
        values = []
        for i, (value, dim) in enumerate(zip(action_indices, dims)):
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or not 0 <= value < dim:
                raise ValueError(f"动作索引 {i} 超界")
            values.append(int(value))
        freq_idx, slot_idx, power_idx = values
        mask = self.action_mask()
        if mask is not None and (not mask["frequency_slots"][freq_idx, slot_idx]
                                 or not mask["power"][power_idx]):
            raise ValueError("动作不满足联合无效动作掩码；必须在策略采样前屏蔽非法组合")
        levels = self.config["environment"]["power_levels_w"]
        request = dict(group=self.raw_env.beam_idx % 8, freq=freq_idx, slots=slot_idx + 1,
                       power_w=float(levels[power_idx]), indices=values)
        before = self._prior
        mode = self.config["environment"]["action_mode"]
        projected = self._project((freq_idx, slot_idx + 1, power_idx)) if mode == "feasible_projection" else (freq_idx, slot_idx + 1, power_idx)
        p_freq, p_slots, p_power_idx = projected
        power_w = levels[p_power_idx]
        no_feasible = mode == "feasible_projection" and p_slots == 0
        power_db = 10 * np.log10(power_w)
        if mode == "feasible_projection" or mask is not None:
            power_db -= 1e-12  # Avoid floating-point round-up of an exactly feasible watt budget.
        action = [request["group"], p_freq, 1 if no_feasible else p_slots, power_db]
        with _budget_scope(self.config["environment"]["power_budget_w"]):
            if no_feasible:
                occupied = self.raw_env.satellite_freq_state.freq_pool_real
                previous_occupancy = float(occupied[request["group"], p_freq])
                previous_variant = self.raw_env.reward_variant
                occupied[request["group"], p_freq] = max(previous_occupancy, 1)
                self.raw_env.reward_variant = "legacy_repaired" if previous_variant == "legacy_original" else previous_variant
                try:
                    legacy_state, legacy_reward, done, raw_info = self.raw_env.step(action)
                finally:
                    occupied[request["group"], p_freq] = previous_occupancy
                    self.raw_env.reward_variant = previous_variant
                self.raw_env.occupy -= 1
                raw_info["occupy"] = 0
                if self.config["reward"]["preset"] in ("legacy_original", "legacy_repaired"):
                    legacy_reward += 1
            else:
                legacy_state, legacy_reward, done, raw_info = self.raw_env.step(action)
        if mask is not None:
            if (int(action[1]) != freq_idx or int(action[2]) != slot_idx + 1
                    or int(raw_info["out"]) or int(raw_info["occupy"]) or self.raw_env.overload):
                raise RuntimeError("合法动作执行后出现裁剪或超功率；停止并保留现场")
            if not math.isclose(float(raw_info["actual_power"]), float(levels[power_idx]),
                                rel_tol=1e-9, abs_tol=1e-9):
                raise RuntimeError("掩码动作的实际功率与所选功率档位不一致")
            self._mask_used_power += Decimal(str(levels[power_idx]))
        self._last_legacy_state = legacy_state
        after = self._snapshot()
        self._prior = after
        actual = dict(group=int(action[0]), freq=int(action[1]), slots=int(action[2]),
                      power_w=float(raw_info["actual_power"]), charged_power_w=float(raw_info["power"]),
                      selected_power_w=float(power_w), indices=[p_freq, (p_slots - 1) if p_slots else None, p_power_idx])
        self._previous_actual_action = np.asarray(
            [actual["freq"] / 99, actual["slots"] / 10, actual["power_w"] / 50, 1],
            dtype=np.float32)
        result = self._reward(before, after, raw_info, legacy_reward, request, actual, done)
        state = self._state()
        info = _json(raw_info)
        # raw_info.current_satisfaction is capped and raw_info.sgm_rate is a
        # reward term. Neither can reconstruct physical capacity or service SGM.
        current_rate = self.raw_env.beam_bps[-1]
        current_demand = self._scene["rate"].reshape(-1)[int(raw_info["beam_id"])]
        current_sgm = _sgm([current_rate], [current_demand],
                           float(self._scene["rate"].mean()))[0]
        info["beam_service_step"] = beam_service_fields(current_rate, current_demand, current_sgm)
        info.update(reward_terms=result[1], reward_preset=self.config["reward"]["preset"],
                    requested_action=request, actual_action=actual, legacy_reward=legacy_reward,
                    projection_applied=mode == "feasible_projection" and projected != (freq_idx, slot_idx + 1, power_idx),
                    reward_total=result[0])
        info["action_mask"] = self.config["environment"].get("action_mask", "none")
        if mask is not None:
            info["legal_frequency_slot_pairs"] = int(mask["frequency_slots"].sum())
            info["legal_power_levels"] = int(mask["power"].sum())
            info["legal_action_count"] = info["legal_frequency_slot_pairs"] * info["legal_power_levels"]
        self._trace.append(dict(beam_id=int(raw_info["beam_id"]), requested_action=request,
                                actual_action=actual, reward=result[0], done=bool(done),
                                out=int(raw_info["out"]), occupy=int(raw_info["occupy"])))
        return state, result[0], bool(done), info

    def _throughput_reference_gbps(self):
        """ΔΣmin(C,D) 的归一化参考量（方案18 第 3 节手段A）。

        默认取“全场景平均每束需求”除以 1e9。选择理由：该参考量就是逐束满足率的
        分母自身的平均，于是归一化后的项恰好等于“本步新增的有效承载相当于多少束
        的平均需求”，与 N 无关、与场景负载同尺度，权重可以直接读成"每多做满一束
        业务值多少分"。也可切换为场景总需求（总量口径）或固定数值 Gbps。
        """
        shaping = self.config["reward"].get("shaping", {})
        mode = shaping.get("throughput_reference", "mean_demand")
        if isinstance(mode, (int, float)) and not isinstance(mode, bool):
            if mode <= 0:
                raise ValueError("吞吐归一化参考量必须是正数")
            return float(mode)
        demands = self._scene["rate"].reshape(-1)
        if mode == "mean_demand":
            return float(demands.mean()) / 1e9
        if mode == "total_demand":
            return float(demands.sum()) / 1e9
        raise ValueError(f"未知吞吐归一化参考量: {mode!r}")

    def _reward(self, before, after, info, legacy_reward, requested, actual, done):
        preset = self.config["reward"]["preset"]
        if preset == "terminal_si_delta":
            # Both potentials use every target beam. Unserved beams contribute
            # zero and earlier beams are recomputed after later interference.
            # With gamma=1 the undiscounted episode sum telescopes exactly.
            count = int(self.raw_env.beam_num)
            delta = 100.0 * (float(after["si"]) - float(before["si"])) / count
            return delta, {"terminal_si_delta": dict(
                raw=delta, scaled=delta, weighted=delta, timing="step", preset=preset)}
        if preset != "custom":
            if preset == "fused":
                components = {"basic_delta": after["basic"] - before["basic"],
                              "si_delta": after["si"] - before["si"],
                              "sgm_delta": after["sgm"] - before["sgm"],
                              "power_cost": -_float(info["actual_power"]) / self.config["environment"]["power_budget_w"]}
                weights = {"basic_delta": 2.0, "si_delta": 1.0,
                           "sgm_delta": 0.2, "power_cost": 1.0}
            else:
                increment = _float(info["satisfaction_increment"])
                sat = 1 if increment >= 1 else -1
                current_sgm = _float(info["sgm_rate"])
                current_rate = max(0.0, _float(self.raw_env.beam_bps[-1]))
                requirement = _float(info["required_rate"])
                gap = abs(current_rate - requirement)
                sgm_penalty = max(0.0, gap - 5e7) / 1e9 if gap > 5e7 else 0.0
                components = {"legacy_threshold": float(sat), "current_sgm": current_sgm,
                              "legacy_sgm_penalty": -sgm_penalty,
                              "occupy_penalty": -_float(info["occupy"]) / max(1, _float(info["origin_slots"]))}
                weights = {key: 1.0 for key in components}
            total = float(legacy_reward)
            # Preserve the raw environment's exact reward, including any numerical residual.
            residual = total - sum(value * weights[key] for key, value in components.items())
            if abs(residual) > 1e-10:
                components["legacy_residual"] = residual
                weights["legacy_residual"] = 1.0
            if preset == "legacy_repaired_throughput":
                self._add_throughput_terms(components, weights, before, after, info, actual)
                total += sum(value * weights[key] for key, value in components.items()
                             if key in ("throughput_delta", "spectrum_occupancy"))
            total += self._shape_reward(components, weights, before, after)
            return total, self._components(components, weights, preset)
        budget = self.config["environment"]["power_budget_w"]
        this_rate = max(0.0, _float(self.raw_env.beam_bps[-1]))
        demand = _float(info["required_rate"])
        prior_rates = before["rates"]
        changed_prior = after["rates"][:len(prior_rates)]
        interference_loss = max(0.0, float((prior_rates - changed_prior).sum())) / 1e9
        excess = max(0.0, this_rate - demand) / 1e9
        raw = {
            "si_delta": after["si"] - before["si"],
            "basic_delta": after["basic"] - before["basic"],
            "sgm_delta": after["sgm"] - before["sgm"],
            "current_si": min(this_rate / demand, 1),
            "current_sgm": float(_sgm([this_rate], [demand], float(self._scene["rate"].mean()))[0]),
            "throughput_delta": after["throughput"] - before["throughput"],
            "useful_throughput_delta": after["useful_throughput"] - before["useful_throughput"],
            "jain_delta": after["jain"] - before["jain"],
            "min_si_delta": after["min_si"] - before["min_si"],
            "shortfall_delta": before["shortfall"] - after["shortfall"],
            "excess_supply": -excess,
            "power_cost": -_float(info["actual_power"]) / budget,
            "slot_cost": -actual["slots"] / 10,
            "interference_loss": -interference_loss,
            "out_penalty": -_float(info["out"]) / requested["slots"],
            "occupy_penalty": -_float(info["occupy"]) / requested["slots"],
            "clip_penalty": -(requested["slots"] - actual["slots"]) / requested["slots"],
            "overload_penalty": -max(0, self.raw_env.total_power - budget) / budget if self.raw_env.overload and not self._overload_seen else 0,
            "zero_slot_penalty": -1 if actual["slots"] == 0 else 0,
            "terminal_si": after["si"] / self.raw_env.beam_num if done else 0,
            "terminal_basic": after["basic"] / self.raw_env.beam_num if done else 0,
            "terminal_sgm": after["sgm"] / self.raw_env.beam_num if done else 0,
            "terminal_throughput": after["throughput"] if done else 0,
            "terminal_jain": _jain(self._all_ratios()) if done else 0,
            "terminal_min_si": float(self._all_ratios().min()) if done else 0,
            "incomplete_penalty": -(self.raw_env.beam_num - self.raw_env.beam_idx) / self.raw_env.beam_num if done else 0,
        }
        self._overload_seen = self.raw_env.overload
        terms, total = {}, 0.0
        for entry in self.config["reward"]["terms"]:
            value = float(raw[entry["id"]])
            scaled = value / entry["scale"]
            weighted = entry["weight"] * scaled
            terms[entry["id"]] = dict(raw=value, scaled=scaled, weighted=weighted, timing=entry["timing"])
            total += weighted
        total += self._shape_reward(components=None, weights=None, before=before, after=after,
                                    extra=terms)
        return float(total), terms

    @staticmethod
    def _components(components, weights, preset):
        return {key: dict(raw=value, scaled=value, weighted=value * weights[key], timing="step",
                          preset=preset, enable="legacy_repaired")
                for key, value in components.items()}

    def _add_throughput_terms(self, components, weights, before, after, info, actual):
        """S2-P18 手段A：在保留原四项的前提下追加“用起频谱/多做业务量”的正项。

        默认权重为 0，只有把 preset 设为 ``legacy_repaired_throughput`` 并显式给出
        正权重时才生效，因此 ``legacy_repaired`` 的返回奖励逐位不变。两项都只读取
        本步已经发生的结果，不改变物理、动作可行性或终态统计：

        - ``throughput_delta``：本步有效承载增量 ΔΣmin(C,D)，除以参考量。参考量默认
          取“全场景平均每束需求”，见 :meth:`_throughput_reference_gbps` 的说明。
        - ``spectrum_occupancy``：本步实际占用槽数 / 当前组可用槽数。分母取
          ``该组100个位置 − 同组其余波束各预留1槽``，与动作掩码的可行槽口径一致；
          这样“把空槽用起来”与“给后续束留出义务”在同一个尺度上比较。
        """
        shaping = self.config["reward"].get("shaping", {})
        w_throughput = float(shaping.get("w_throughput", 0))
        w_spectrum = float(shaping.get("w_spectrum", 0))
        if w_throughput:
            reference = self._throughput_reference_gbps()
            components["throughput_delta"] = (
                after["useful_throughput"] - before["useful_throughput"]) / reference
            weights["throughput_delta"] = w_throughput
        if w_spectrum:
            group = int(info["group"])
            remaining_group = len(range(self.raw_env.beam_idx, self.raw_env.beam_num, 8))
            available = 100.0 - (remaining_group - 1)
            if available <= 0:
                raise RuntimeError("当前组可用槽上限非正，资源账错误")
            components["spectrum_occupancy"] = float(actual["slots"]) / available
            weights["spectrum_occupancy"] = w_spectrum

    def _shape_reward(self, components, weights, before, after, extra=None):
        """S2-P18 手段F：跨束势能型奖励整形（potential-based shaping）。

        势函数（只依赖状态，不依赖动作）：

            Φ(s) = mean_i min(C_i / D_i, 1)

        其中 i 取该场景全部目标波束，未服务束的 C_i 记 0；C_i 取每步干扰更新后的
        逐束链路容量 ``raw_env.beam_bps``，D_i 为场景需求。均值的分母固定为场景束数
        N，所以 Φ 的量纲是“平均需求满足率”，与 N 无关。

        整形项为 ``w_shape × (γ·Φ(s') − Φ(s))``，γ 取算法实际使用的折扣。这是标准的
        势能型整形（Ng、Harada 与 Russell，1999）：对任意折扣与任意 MDP，回报中该
        差分沿轨迹求和后只剩端点势函数 ``γ^(T-1)·Φ(s_T) − Φ(s_0)``，与策略无关，
        因此最优策略集合不变。实现层面该性质由“只读状态、不读动作、只按相邻状态
        差分相加”保证，并由 ``tests/test_s2p18_arms.py`` 的确定性小例核对。
        """
        weight = float(self.config["reward"].get("shaping", {}).get("w_shape", 0))
        if not weight:
            return 0.0
        gamma = float(self.train_gamma())
        term = gamma * float(after["mean_si"]) - float(before["mean_si"])
        if extra is not None:
            extra["cross_beam_potential"] = dict(raw=term, scaled=term, weighted=weight * term,
                                                 timing="step")
        else:
            components["cross_beam_potential"] = term
            weights["cross_beam_potential"] = weight
        return weight * term

    def train_gamma(self):
        """当前回合实际使用的折扣；由 runner 按束数日程写入。"""
        return float(getattr(self, "_effective_gamma", self.config["training"]["gamma"]))

    def _all_ratios(self):
        rates = np.zeros(self.raw_env.beam_num, dtype=np.float64)
        rates[:len(self.raw_env.beam_bps)] = np.maximum(self.raw_env.beam_bps, 0)
        return np.minimum(rates / self._scene["rate"].reshape(-1), 1)

    def beam_results(self):
        self._require_reset()
        if self._no_business:
            return []
        env = self.raw_env
        rates = np.zeros(env.beam_num)
        rates[:len(env.beam_bps)] = np.maximum(env.beam_bps, 0)
        demands = self._scene["rate"].reshape(-1)
        beta = float(demands.mean())
        sgm = _sgm(rates, demands, beta)
        actions = {int(row[0]): row for row in env.history_beam}
        result = []
        for i in range(env.beam_num):
            row = actions.get(i)
            result.append(dict(beam_id=i, original_index=int(self._scene["original_indices"][i]),
                               lat=float(self._scene["beam_location"][i, 0]),
                               lon=float(self._scene["beam_location"][i, 1]),
                               **beam_service_fields(rates[i], demands[i], sgm[i]),
                               group=int(row[1]) if row else i % 8,
                               freq=int(row[2]) if row else None,
                               slots=int(row[3]) if row else 0,
                               power_w=float(10 ** (row[4] / 10)) if row and row[3] > 0 else 0.0,
                               basic_met=bool(rates[i] >= env.basic_fraction * demands[i])))
        return result

    def metrics(self, beams=None):
        self._require_reset()
        if self._no_business:
            return dict(beam_count=0, processed_count=0, complete=True,
                        termination_reason="no_business", final_si_total=None,
                        final_si_mean=None, final_sgm_total=None,
                        final_sgm_mean=None, basic_met_count=None,
                        zero_capacity_count=None, power_w=0.0,
                        power_budget_w=float(self.config["environment"]["power_budget_w"]))
        env = self.raw_env
        beams = self.beam_results() if beams is None else beams
        rates = np.array([b["rate_bps"] for b in beams])
        ratios = np.array([b["si"] for b in beams])
        sgm = np.array([b["sgm"] for b in beams])
        requested_slots = sum(t["requested_action"]["slots"] for t in self._trace)
        actual_slots = sum(t["actual_action"]["slots"] for t in self._trace)
        zero_slots = sum(t["actual_action"]["slots"] == 0 for t in self._trace)
        clipped = sum(t["requested_action"]["slots"] != t["actual_action"]["slots"]
                      or t["requested_action"]["freq"] != t["actual_action"]["freq"]
                      or not math.isclose(t["requested_action"]["power_w"], t["actual_action"]["power_w"], rel_tol=1e-9)
                      for t in self._trace)
        throughput = float(rates.sum())
        return dict(beam_count=env.beam_num, processed_count=env.beam_idx,
                    complete=env.beam_idx == env.beam_num, overload=bool(env.overload),
                    final_si_total=float(ratios.sum()), final_si_mean=float(ratios.mean()),
                    final_sgm_total=float(sgm.sum()), final_sgm_mean=float(sgm.mean()),
                    basic_met_count=sum(b["basic_met"] for b in beams),
                    zero_capacity_count=int(np.sum(rates <= 0)),
                    jain=float(_jain(ratios)), throughput_bps=throughput,
                    power_w=float(env.total_power), power_accounted_w=float(env.total_power),
                    radiated_power_w=sum(t["actual_action"]["power_w"] for t in self._trace),
                    power_budget_w=float(self.config["environment"]["power_budget_w"]),
                    requested_slots=requested_slots, actual_slots=actual_slots, allocated_slots=actual_slots,
                    occupancy_fraction=float(np.mean(env.satellite_freq_state.freq_pool_real)),
                    spectral_efficiency=throughput / (actual_slots * 2.5e7) if actual_slots else 0.0,
                    interference_beams=int(env.interfere_num),
                    in_range_interference_beams=int(env.interfere_in_range_num),
                    zero_slot_count=zero_slots,
                    zero_slot_rate=zero_slots / env.beam_idx if env.beam_idx else 0.0,
                    out_slots=sum(t["out"] for t in self._trace),
                    occupied_slots=sum(t["occupy"] for t in self._trace),
                    clipped_count=clipped,
                    termination_reason="overload" if env.overload else "complete" if env.beam_idx == env.beam_num else "in_progress")


已定义 C_ENV: src/satellite_rl/workbench/environment.py:141-824


### 逐束状态编码

输入是一批按真实束数补零的逐束观测，输出是供策略和 Q 网络共用的场景特征。`_resolve_lengths` 保留真实束数，`forward` 只汇聚有效波束；如果把补零当真实波束，注意力和平均值都会偏。

下格是冻结源码的完整 `Encoder` 定义，来源 `src/satellite_rl/agents/workbench_networks.py:15-387`，提交 `48c7c4feac71`；运行时逐字节核对。


In [5]:
%%project_module C_ENCODER
class Encoder(nn.Module):
    def __init__(self, config, state_dim):
        super().__init__()
        spec = config["network"]
        kind = spec["kind"]
        self.kind = kind
        self.state_dim = state_dim
        self.transformer_layout = spec.get("transformer_layout", "scalar_tokens")
        self.previous_actual_action = "previous_actual_action" in config.get("state", {}).get("features", [])
        self.graph_mode = spec.get("graph_mode", "none")
        # 逐束实体编码的只读标记：只有自然布局的 actual 档会置真，其余一律为假，
        # 这样 forward 的统一分支可以直接查询而无需按 kind 分支判断。
        self.actual_count = False
        self.beam_rows = 0
        self.max_beam_count = 0
        if self.graph_mode != "none":
            from satellite_rl.agents.graph_encoder import GraphEncoder
            self.graph = GraphEncoder(config, state_dim, self.graph_mode)
            self.output_dim = self.graph.output_dim
            self.transformer_layout = "graph_" + self.graph_mode
            return
        activation = _activation(spec["activation"])
        dropout = float(spec["dropout"])
        if kind == "mlp":
            layers = []
            width = state_dim
            for hidden in spec["hidden_sizes"]:
                layers += [nn.Linear(width, hidden), activation()]
                if dropout:
                    layers.append(nn.Dropout(dropout))
                width = hidden
            self.net = nn.Sequential(*layers)
            self.output_dim = width
        elif kind == "cnn":
            channels = spec["cnn_channels"]
            kernel = spec["kernel_size"]
            layers = []
            width = 1
            for channel in channels:
                layers += [nn.Conv1d(width, channel, kernel, padding=kernel // 2), activation()]
                if dropout:
                    layers.append(nn.Dropout(dropout))
                width = channel
            bins = min(16, state_dim)
            self.net = nn.Sequential(*layers, nn.AdaptiveAvgPool1d(bins), nn.Flatten())
            self.output_dim = width * bins
        elif kind == "transformer":
            embed = spec["embedding_dim"]
            self.readout_mode = spec.get("readout_mode", "first_token")
            self.beam_count_conditioning = spec.get("beam_count_conditioning", "none")
            self.context_beam_count = spec.get("context_beam_count", "fixed_100")
            if self.readout_mode not in ("first_token", "attention_pooling"):
                raise ValueError(f"Unsupported readout mode: {self.readout_mode}")
            if self.beam_count_conditioning not in ("none", "film"):
                raise ValueError(f"Unsupported beam-count conditioning: {self.beam_count_conditioning}")
            if self.context_beam_count not in ("fixed_100", "actual"):
                raise ValueError(f"Unsupported context beam count mode: {self.context_beam_count}")
            if self.transformer_layout == "scalar_tokens":
                self.input_projection = nn.Linear(1, embed)
                self.position = nn.Parameter(torch.zeros(1, state_dim, embed))
            elif self.transformer_layout in ("tsac_13_tokens", "tsac_14_context", "tsac_14_variable_context", "tsac_32_context", "tsac_natural_context"):
                expected_dim = {"tsac_13_tokens": 204, "tsac_14_context": 207,
                                "tsac_14_variable_context": (
                                    213 if config.get("state", {}).get("features")
                                    == list(S2P18_VARIABLE_FEATURES) else 210),
                                "tsac_32_context": 2946}.get(self.transformer_layout)
                if self.transformer_layout == "tsac_natural_context":
                    self.natural_level = config["state"]["context_level"]
                    self.natural_shapes = natural_shapes(self.natural_level)
                    expected_dim = natural_dim(self.natural_level) + (
                        4 if self.previous_actual_action else 0)
                    self.actual_count = (self.context_beam_count == "actual"
                                         and "natural_beam_context" in self.natural_shapes)
                    if self.actual_count:
                        # 逐束实体编码（S2-P18 手段G）：逐束字段只含真实实体，观测宽度
                        # 随场景实际束数 N 变化，不填充到 100；配置期的状态维数上界用
                        # environment.beam_count_max，实际维数由场景束数决定。
                        self.max_beam_count = int(config["environment"].get("beam_count_max", 100))
                        rows = self.max_beam_count
                        expected_dim += ((rows - 100)
                                         * self.natural_shapes["natural_beam_context"][-1])
                    else:
                        rows = self.natural_shapes.get("natural_beam_context", [10, 10, 5])[0]
                        self.max_beam_count = 100
                else:
                    self.actual_count = False
                    rows = 10
                    self.beam_rows = 100
                    self.max_beam_count = 100
                if state_dim != expected_dim:
                    raise ValueError(f"T-SAC {self.transformer_layout} layout requires the ordered {expected_dim}-dimensional state")
                self.beam_rows = rows
                self.geo_embed = nn.Linear(2, embed)
                self.rate_embed = nn.Linear(1, embed)
                self.power_embed = nn.Linear(1, embed)
                self.slot_embed = nn.Linear(20, embed)
                self.pos_embed = nn.Parameter(torch.zeros(1, 13, embed))
            else:
                raise ValueError(f"Unsupported transformer layout: {self.transformer_layout}")
            layer = nn.TransformerEncoderLayer(embed, spec["num_heads"],
                                               spec["feedforward_dim"], dropout,
                                               activation=torch.tanh if spec["activation"] == "tanh"
                                               else spec["activation"],
                                               batch_first=self.transformer_layout == "scalar_tokens")
            self.net = nn.TransformerEncoder(layer, spec["num_layers"], enable_nested_tensor=False)
            if self.transformer_layout in ("tsac_14_context", "tsac_14_variable_context"):
                # Keep the 13 original token and encoder initializations in their
                # original RNG order. The extra projection cannot shift the heads.
                with torch.random.fork_rng(devices=[]):
                    self.context_embed = nn.Linear(6 if self.transformer_layout == "tsac_14_variable_context" else 3, embed)
                self.context_pos_embed = nn.Parameter(torch.zeros(1, 1, embed))
            elif self.transformer_layout == "tsac_32_context":
                # Four information levels share exactly the same parameters.
                # Keep the original 13 token and encoder initialization order.
                with torch.random.fork_rng(devices=[]):
                    self.global_embed = nn.Linear(10, embed)
                    self.beam_embed = nn.Linear(110, embed)
                    self.resource_embed = nn.Linear(204, embed)
                self.extra_pos_embed = nn.Parameter(torch.zeros(1, 19, embed))
            elif self.transformer_layout == "tsac_natural_context":
                # Preserve the common 13-token and encoder initialization. Only
                # actual visible fields receive a projection and position.
                with torch.random.fork_rng(devices=[]):
                    if "natural_global_context" in self.natural_shapes:
                        self.global_embed = nn.Linear(self.natural_shapes["natural_global_context"][0], embed)
                    if "natural_beam_context" in self.natural_shapes:
                        if self.actual_count:
                            # 逐束实体编码：每个实体行共享一个投影，参数形状只与“每实体
                            # 通道数”有关，行数不进参数，因此同一份权重可用于任意 N。
                            self.beam_embed = nn.Linear(
                                self.natural_shapes["natural_beam_context"][-1], embed)
                            self.beam_embed_features = self.natural_shapes["natural_beam_context"][-1]
                        else:
                            # 固定 100 束档位：状态里逐束块是 (100, 每段通道数)，
                            # 拼接后按展平宽度做一次投影（与历史行为一致）。
                            self.beam_embed_features = 100 * self.natural_shapes["natural_beam_context"][-1]
                            self.beam_embed = nn.Linear(self.beam_embed_features, embed)
                    if "natural_resource_context" in self.natural_shapes:
                        self.resource_embed = nn.Linear(self.natural_shapes["natural_resource_context"][-1], embed)
                if self.actual_count:
                    # 位置参数按束数上界一次分配；每个场景只用前 N 行，不因 N 大小
                    # 改变参数形状，所以同一份权重可用于任意 N 而无需重新初始化。
                    pos_slots = (self.natural_shapes["natural_beam_context"][-1] * self.max_beam_count
                                 + self.natural_shapes["natural_resource_context"][0]
                                 if "natural_resource_context" in self.natural_shapes
                                 else self.natural_shapes["natural_beam_context"][-1] * self.max_beam_count)
                else:
                    pos_slots = natural_token_count(self.natural_level) - 13
                self.pos_slots = pos_slots
                if pos_slots:
                    self.extra_pos_embed = nn.Parameter(torch.zeros(1, pos_slots, embed))
                if self.actual_count and self.readout_mode == "attention_pooling":
                    # 注意力池化读出：一个可学习的可加性打分向量（无额外参数形状），
                    # 对逐束实体令牌做掩码加权平均。该参数只在池化模式下创建，
                    # 因此 first_token 模式的参数与初始化顺序逐位不变。
                    with torch.random.fork_rng(devices=[]):
                        self.pool_query = nn.Linear(embed, 1)
                if self.beam_count_conditioning == "film":
                    # N 条件调制（FiLM）：把场景实际束数作为条件，对读出结果做
                    # 逐通道缩放与平移，使同一套权重按规模调整内部表示。
                    with torch.random.fork_rng(devices=[]):
                        self.film = nn.Linear(1, 2 * embed)
                        nn.init.zeros_(self.film.weight)
                        nn.init.zeros_(self.film.bias)
                if self.previous_actual_action:
                    with torch.random.fork_rng(devices=[]):
                        self.previous_action_embed = nn.Linear(4, embed)
                    self.previous_action_pos = nn.Parameter(torch.zeros(1, 1, embed))
            self.output_dim = embed
        else:
            raise ValueError(f"Unsupported network: {kind}")

    def _resolve_lengths(self, states, lengths):
        """逐束实体编码的“有效令牌长度”与单样本实际状态宽度。

        返回 ``(lengths, actual_width)``：``lengths`` 是长度张量，``actual_width`` 是
        单个样本未填充时的状态宽度。``lengths`` 为整数时表示单样本（不填充）。
        """
        if lengths is None:
            return None, self.state_dim
        if torch.is_tensor(lengths):
            return lengths.reshape(-1), int(lengths.max().item())
        return torch.as_tensor([int(lengths)], device=states.device), int(lengths)

    def _beam_count(self, states, actual_width):
        """从状态里取出“当前是第几束”（即处理序号 idx+1）。

        逐束实体的第 5 个通道是 ``i/(N−1)``；已处理束在实体里是连续前缀
        （layout 档第 5 通道、full 档第 6 通道为 1），它们的个数就是当前处理序号
        idx。行数是补零后的固定行数，真实束数由 ``lengths`` 给出，因此这里同时
        接受长度张量（批）与整数（单样本）。该标量同时用于 N 条件调制与规模统计。
        """
        if not self.actual_count:
            return None
        width = self.natural_shapes["natural_beam_context"][-1]
        served_channel = 5 if width == 11 else 4
        offset = 204
        if "natural_global_context" in self.natural_shapes:
            offset += self.natural_shapes["natural_global_context"][0]
        trailing = 4 * self.previous_actual_action
        if "natural_resource_context" in self.natural_shapes:
            trailing += 8 * self.natural_shapes["natural_resource_context"][0]
        rows = (states.shape[1] - offset - trailing) // width
        if torch.is_tensor(actual_width):
            count = actual_width.reshape(-1).to(states.device)
        elif isinstance(actual_width, int) and actual_width < rows:
            count = torch.full((states.shape[0],), int(actual_width),
                               device=states.device, dtype=torch.long)
        else:
            count = torch.full((states.shape[0],), rows, device=states.device, dtype=torch.long)
        served = states[:, offset + served_channel:offset + rows * width:width] > 0.5
        index = served.to(torch.int64).sum(dim=1)
        position = torch.round(
            states[:, offset + 4:offset + rows * width:width]
            * torch.clamp(count - 1, min=1).reshape(-1, 1).to(states.dtype)).long()
        gathered = position.gather(1, index.clamp(max=rows - 1).reshape(-1, 1)).reshape(-1)
        return (gathered.to(states.dtype) + 1.0).clamp(min=1.0, max=float(self.max_beam_count))

    def forward(self, states, lengths=None):
        if states.ndim != 2:
            raise ValueError(f"State must be [batch,width], got {tuple(states.shape)}")
        resolved, actual_width = self._resolve_lengths(states, lengths)
        if self.actual_count:
            # 逐束实体档位：观测宽度固定为「按场景束数上限补零」的宽度，而
            # ``lengths`` 给出的是每个样本的真实束数，两者不是同一个量。
            if states.shape[1] != self.state_dim:
                raise ValueError(f"State shape must be [batch,{self.state_dim}], got {tuple(states.shape)}")
        else:
            if actual_width != states.shape[1]:
                raise ValueError(f"State width {states.shape[1]} differs from declared {actual_width}")
            if states.shape[1] != self.state_dim:
                raise ValueError(f"State shape must be [batch,{self.state_dim}], got {tuple(states.shape)}")
        if self.graph_mode != "none":
            return self.graph(states)
        if self.kind == "mlp":
            return self.net(states)
        if self.kind == "cnn":
            return self.net(states.unsqueeze(1))
        if self.transformer_layout == "scalar_tokens":
            return self.net(self.input_projection(states.unsqueeze(-1)) + self.position).mean(1)
        batch = states.shape[0]
        row = states[:, 4:104].reshape(batch, 10, 10)
        occupancy = states[:, 104:204].reshape(batch, 10, 10)
        slot_features = torch.cat((row, occupancy), dim=-1)
        tokens = torch.cat((
            self.geo_embed(states[:, :2]).unsqueeze(1),
            self.rate_embed(states[:, 2:3]).unsqueeze(1),
            self.power_embed(states[:, 3:4]).unsqueeze(1),
            self.slot_embed(slot_features),
        ), dim=1) + self.pos_embed
        padding_mask = None
        entity_start = 13
        entity_count = 0
        if self.transformer_layout in ("tsac_14_context", "tsac_14_variable_context"):
            context_width = 6 if self.transformer_layout == "tsac_14_variable_context" else 3
            context = self.context_embed(states[:, 204:204 + context_width]).unsqueeze(1) + self.context_pos_embed
            tokens = torch.cat((tokens, context), dim=1)
        elif self.transformer_layout == "tsac_32_context":
            context = torch.cat((
                self.global_embed(states[:, 204:214]).unsqueeze(1),
                self.beam_embed(states[:, 214:1314].reshape(batch, 10, 110)),
                self.resource_embed(states[:, 1314:2946].reshape(batch, 8, 204)),
            ), dim=1) + self.extra_pos_embed
            tokens = torch.cat((tokens, context), dim=1)
        elif self.transformer_layout == "tsac_natural_context":
            offset = 204
            parts = []
            extra_count = 0
            entity_tokens = None
            if "natural_global_context" in self.natural_shapes:
                width = self.natural_shapes["natural_global_context"][0]
                parts.append(self.global_embed(states[:, offset:offset + width]).unsqueeze(1))
                offset += width
                extra_count += 1
            if "natural_beam_context" in self.natural_shapes:
                # 观测里的逐束字段是 [实体数, 通道数]：通道数取本档声明宽度，
                # 行数在逐束编码档位下由状态宽度反推（不填充到 100）。
                width = self.natural_shapes["natural_beam_context"][-1]
                trailing = 4 * self.previous_actual_action
                if "natural_resource_context" in self.natural_shapes:
                    resource = self.natural_shapes["natural_resource_context"]
                    trailing += resource[0] * resource[-1]
                available = states.shape[1] - offset - trailing
                if available < 0:
                    raise ValueError("逐束字段宽度与状态维数不一致")
                if self.actual_count:
                    # 逐束实体编码档位：行数由状态宽度反推，先按“每实体通道数”
                    # 切成单实体令牌，再对每个实体做同一个投影。
                    rows, remainder = divmod(available, width)
                    if remainder:
                        raise ValueError("逐束字段宽度与状态维数不一致")
                    if not 1 <= rows <= self.max_beam_count:
                        raise ValueError(f"实际束数 {rows} 超出 1–{self.max_beam_count}")
                    entity_count = rows
                    block = states[:, offset:offset + rows * width].reshape(batch, rows, width)
                    entity_tokens = self.beam_embed(block)
                else:
                    # 固定档位：逐束信息按展平块整体投影（与历史实现一致）。
                    if available != self.beam_embed_features:
                        raise ValueError("逐束字段宽度与状态维数不一致")
                    entity_count = 10
                    block = states[:, offset:offset + available]
                    parts.append(self.beam_embed(block).unsqueeze(1))
                    extra_count += 1
                offset += available
            if "natural_resource_context" in self.natural_shapes:
                width = 8 * self.natural_shapes["natural_resource_context"][-1]
                parts.append(self.resource_embed(states[:, offset:offset + width].reshape(batch, 8, width // 8)))
                offset += width
                extra_count += 8
            if parts:
                extra = torch.cat(parts, dim=1) + self.extra_pos_embed[:, :extra_count]
                tokens = torch.cat((tokens, extra), dim=1)
            if self.actual_count and entity_tokens is not None:
                # 合并顺序：13 个基础令牌 + 1 个全局 + N 个逐束实体 + 8 个资源组。
                # 实体块不带位置参数：位置参数会把实体顺序写进注意力结果，而逐束
                # 实体本身无序，加了会破坏置换一致性。
                entity_start = tokens.shape[1]
                tokens = torch.cat((tokens, entity_tokens), dim=1)
            if self.previous_actual_action:
                previous = (self.previous_action_embed(states[:, -4:]).unsqueeze(1) +
                            self.previous_action_pos)
                tokens = torch.cat((tokens, previous), dim=1)
            if self.actual_count:
                # 单样本调用（无批次填充）时所有实体都有效：等价于“每样本有效令牌长度
                # 等于实体数”。批量采样时由调用方传入真实长度，用于屏蔽批内填充。
                lengths = (torch.full((states.shape[0],), entity_count, device=states.device,
                                      dtype=torch.long) if resolved is None
                           else (resolved if torch.is_tensor(resolved) else resolved.reshape(-1)))
                entity_positions = torch.arange(tokens.shape[1], device=states.device)[None, :]
                padding_mask = ~((entity_positions >= entity_start)
                                 & (entity_positions < entity_start + entity_count)
                                 & (entity_positions - entity_start < lengths[:, None]))
                # 用零替换被屏蔽的令牌，避免位置参数读出非有限值。
                tokens = tokens.masked_fill(padding_mask.unsqueeze(-1), 0.0)
        sequence = tokens.permute(1, 0, 2)
        # 上一次前向实际喂给骨干的令牌数与实体数（只读诊断，供前端/测试核对，
        # 不影响任何计算）。
        self.last_token_count = int(sequence.shape[0])
        self.last_entity_count = int(entity_count)
        if padding_mask is None:
            encoded = self.net(sequence)
        else:
            # 直接调用底层 forward：某些 PyTorch 版本在注册了 forward pre-hook
            # （例如结构追踪）时无法用关键字传 src_key_padding_mask，这里绕开
            # hook 派发，行为与 TransformerEncoder.forward 完全一致。
            encoded = type(self.net).forward(
                self.net, sequence, None, padding_mask.to(torch.bool))
        encoded = encoded.permute(1, 0, 2)
        result = encoded[:, 0]
        if self.transformer_layout == "tsac_natural_context" and self.actual_count and entity_count:
            if self.readout_mode == "attention_pooling":
                # 注意力池化读出：对“逐束实体令牌”做掩码加权平均，权重由每个实体自身
                # 的可学习打分决定（本档每实体 5 个通道）。因为打分逐实体独立、聚合是
                # 对实体的求和，读出对实体顺序严格不变（等变性由构造保证）；公开的
                # 基础令牌不参与池化，只由 first_token 通道提供全局上下文。
                entity = tokens[:, entity_start:entity_start + entity_count]
                query = self.pool_query.weight.reshape(-1)
                scale = float(query.shape[0] * max(entity_count, 1)) ** 0.5
                scores = (entity * query).sum(-1) / scale
                valid = ((torch.arange(entity_count, device=states.device)[None, :]
                          < lengths[:, None]) if padding_mask is not None
                         else torch.ones(1, entity_count, dtype=torch.bool, device=states.device))
                scores = scores.masked_fill(~valid, float("-inf"))
                weights = scores.softmax(dim=-1)
                if not bool(torch.isfinite(weights).all()):
                    raise ValueError("注意力池化的有效实体为空，无法计算读出")
                result = torch.einsum("bn,bnd->bd", weights, entity)
            if self.beam_count_conditioning == "film":
                count = self._beam_count(states, resolved if resolved is not None else actual_width).reshape(-1, 1)
                scale, shift = self.film(count / float(self.max_beam_count)).chunk(2, dim=-1)
                result = result * (1.0 + scale) + shift
        return result


已定义 C_ENCODER: src/satellite_rl/agents/workbench_networks.py:15-387


### 实际使用的策略与 Q 网络

C0/C1 的冻结配置都选 `factorized` 策略和 `additive` Q，所以实际使用这个多头网络。它对频率、槽数、功率各出一组值；两个 Q 实例由训练器分别创建，再取较小者压住乐观估计。

下格是冻结源码的完整 `MultiHeadNetwork` 定义，来源 `src/satellite_rl/agents/workbench_networks.py:390-398`，提交 `48c7c4feac71`；运行时逐字节核对。


In [6]:
%%project_module C_NET
class MultiHeadNetwork(nn.Module):
    def __init__(self, config, state_dim, action_dims):
        super().__init__()
        self.encoder = Encoder(config, state_dim)
        self.heads = nn.ModuleList([nn.Linear(self.encoder.output_dim, int(dim)) for dim in action_dims])

    def forward(self, states, lengths=None):
        x = self.encoder(states, lengths=lengths)
        return tuple(head(x) for head in self.heads)


已定义 C_NET: src/satellite_rl/agents/workbench_networks.py:390-398


### C1 教师轨迹生成

输入是一景冻结训练场景，输出是贪心教师的逐束转移。它是 C1 的离线数据准备，不会增加 C1 的在线 `env_step`；C0 没有示范源。

下格是冻结源码的完整 `generate_greedy_trajectory` 定义，来源 `src/satellite_rl/workbench/s2p19_demonstrations.py:45-105`，提交 `48c7c4feac71`；运行时逐字节核对。


In [7]:
%%project_module C_TEACHER
def generate_greedy_trajectory(config, scene_path, scene_sha256, target_path, *, stop=None):
    """Write one complete teacher trajectory atomically; never truncate a shard.

    ``stop`` is an optional no-argument callback returning true to interrupt.
    The caller owns durable per-scene progress and may rerun this idempotently.
    """
    from satellite_rl.workbench.baselines import BaselinePolicy
    from satellite_rl.workbench.environment import WorkbenchEnv

    scene_path, target_path = Path(scene_path), Path(target_path)
    if _sha256(scene_path) != scene_sha256:
        raise ValueError("teacher scene SHA differs from frozen source")
    with np.load(scene_path, allow_pickle=False) as data:
        scene = {key: np.array(data[key], copy=True) for key in data.files}
    env = WorkbenchEnv(config)
    state = env.reset(scene=scene)
    count = int(env.raw_env.beam_num)
    if not 50 <= count <= 200:
        raise ValueError("S2-P19 teacher requires 50..200 target beams")
    policy = BaselinePolicy("greedy", env)
    rows = {key: [] for key in ("state", "action", "reward", "next_state", "done",
                                "mask_fl", "mask_power", "next_fl", "next_power")}
    done = False
    while not done:
        if stop is not None and stop():
            raise InterruptedError("teacher preparation stopped before complete trajectory")
        mask = env.action_mask()
        action = policy.act(state, action_mask=mask)
        next_state, reward, done, _ = env.step(action)
        next_mask = None if done else env.action_mask()
        rows["state"].append(np.asarray(state, dtype=np.float32).copy())
        rows["action"].append(action)
        rows["reward"].append(float(reward))
        rows["next_state"].append(np.asarray(next_state, dtype=np.float32).copy())
        rows["done"].append(done)
        rows["mask_fl"].append(mask["frequency_slots"].copy())
        rows["mask_power"].append(mask["power"].copy())
        rows["next_fl"].append(np.zeros_like(mask["frequency_slots"]) if done else
                               next_mask["frequency_slots"].copy())
        rows["next_power"].append(np.zeros_like(mask["power"]) if done else
                                  next_mask["power"].copy())
        state = next_state
    if len(rows["reward"]) != count:
        raise ValueError("teacher trajectory did not assign every target beam")
    metrics = env.metrics()
    if not np.isclose(sum(rows["reward"]), 100 * metrics["final_si_mean"], atol=1e-5):
        raise ValueError("teacher reward does not telescope to terminal all-beam SI")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    temp = target_path.with_name(target_path.name + ".incomplete")
    try:
        with temp.open("wb") as stream:
            np.savez_compressed(stream, **{key: np.asarray(value) for key, value in rows.items()},
                                entity_count=np.asarray(count, dtype=np.int32),
                                scene_sha256=np.asarray(scene_sha256),
                                terminal_si_mean=np.asarray(metrics["final_si_mean"], dtype=np.float64))
        temp.replace(target_path)
    finally:
        temp.unlink(missing_ok=True)
    return {"path": target_path.name, "sha256": _sha256(target_path),
            "scene_sha256": scene_sha256, "beam_count": count,
            "transition_count": count, "bucket": _bucket(count)}


已定义 C_TEACHER: src/satellite_rl/workbench/s2p19_demonstrations.py:45-105


### C1 示范数据装载与采样

该类把教师转移和来源 SHA 一起装入并抽样。先看构造时如何核 manifest，再看 `sample` 如何取批；C1 更新时从这里取一半样本。

下格是冻结源码的完整 `DemonstrationDataset` 定义，来源 `src/satellite_rl/workbench/s2p19_demonstrations.py:264-326`，提交 `48c7c4feac71`；运行时逐字节核对。


In [8]:
%%project_module C_DEMO
class DemonstrationDataset:
    """Lazy, SHA-verified shard sampler with bounded memory and replayable RNG."""

    def __init__(self, manifest_path, expected_sha256, config, source_set_sha256, seed):
        manifest = validate_manifest(manifest_path, expected_sha256, config, source_set_sha256)
        self.root = Path(manifest_path).parent
        self.records = manifest["records"]
        self.arrays = {key: np.load(self.root / row["path"], mmap_mode="r", allow_pickle=False)
                       for key, row in manifest["packed"].items()}
        self.by_bucket = {key: [] for key in ("anchor_100", "lower", "middle", "upper")}
        self.entity_counts = np.empty(sum(row["transition_count"] for row in self.records), dtype=np.int16)
        offset = 0
        for row in self.records:
            count = int(row["transition_count"])
            self.by_bucket[row["bucket"]].extend(range(offset, offset + count))
            self.entity_counts[offset:offset + count] = int(row["beam_count"])
            offset += count
        self.rng = random.Random(seed)
        self.draw_count = 0
        self.manifest_sha256 = expected_sha256

    def sample(self, count, device, action_dims):
        if count % 4:
            raise ValueError("demonstration sample must split evenly over four N buckets")
        from satellite_rl.agents.workbench_algorithms import pad_states
        from satellite_rl.agents.action_masking import mask_batch
        indices = []
        for bucket in self.by_bucket.values():
            indices.extend(self.rng.choices(bucket, k=count // 4))
        self.rng.shuffle(indices)
        counts = self.entity_counts[indices].tolist()
        s, lengths = pad_states(self.arrays["state"][indices], device,
                                padded_width=self.arrays["state"].shape[1], entity_counts=counts)
        ns, next_lengths = pad_states(self.arrays["next_state"][indices], device,
                                      padded_width=self.arrays["next_state"].shape[1], entity_counts=counts)
        current = mask_batch([{"frequency_slots": self.arrays["mask_fl"][i],
                               "power": self.arrays["mask_power"][i]} for i in indices], action_dims, device)
        following = mask_batch([{"frequency_slots": self.arrays["next_fl"][i],
                                 "power": self.arrays["next_power"][i]} for i in indices], action_dims, device)
        self.draw_count += count
        return (s, torch.as_tensor(np.array(self.arrays["action"][indices], copy=True),
                                   dtype=torch.long, device=device),
                torch.as_tensor(np.array(self.arrays["reward"][indices], copy=True),
                                dtype=torch.float32, device=device),
                ns, torch.as_tensor(np.array(self.arrays["done"][indices], copy=True),
                                    dtype=torch.float32, device=device),
                lengths, next_lengths, current, following)

    def state_dict(self):
        return {"manifest_sha256": self.manifest_sha256, "rng": self.rng.getstate(),
                "draw_count": self.draw_count}

    def load_state_dict(self, state):
        if state.get("manifest_sha256") != self.manifest_sha256:
            raise ValueError("demonstration manifest changed on checkpoint resume")
        self.rng.setstate(state["rng"])
        self.draw_count = int(state["draw_count"])

    def close(self):
        """Release memory maps, including on Windows where open files cannot move."""
        for array in self.arrays.values():
            array._mmap.close()
        self.arrays.clear()


已定义 C_DEMO: src/satellite_rl/workbench/s2p19_demonstrations.py:264-326


### 在线回放与恢复修复

在线转移按真实束数、动作掩码与回合进度存入回放池，抽样时恢复成张量。续跑修复针对满环时束数表多出的无效尾零，严格核对旧断点后才恢复。

下格是冻结源码的完整 `Replay` 定义，来源 `src/satellite_rl/agents/workbench_algorithms.py:169-458`，提交 `48c7c4feac71`；运行时逐字节核对。


In [9]:
%%project_module C_REPLAY
class Replay:
    def __init__(self, capacity, seed, bucket_mode="none", progress_strata=1,
                 padded_width=None):
        # 逐束实体档位：观测固定为补零宽度，采样时不需要按批再补齐，长度按每条
        # 转移记录的真实束数给出。None 表示历史行为。
        self.padded_width = int(padded_width) if padded_width else None
        if bucket_mode not in ("none", "fixed_100", "mixed_four_buckets"):
            raise ValueError("Unknown replay bucket mode")
        if not isinstance(progress_strata, int) or isinstance(progress_strata, bool) or not 1 <= progress_strata <= 8:
            raise ValueError("进度分层段数须为1至8的整数")
        self.bucket_mode = bucket_mode
        # S2-P18 手段E：在数量桶之上再加“回合内进度分位”分层。默认 1 段，
        # 即旧采样行为逐位不变（每批只在数量桶之间等分）。
        self.progress_strata = progress_strata
        self.capacity = capacity
        self.rows = (deque(maxlen=capacity) if bucket_mode == "none" else [])
        self.rng = random.Random(seed)
        self.next_position = 0
        self.bucket_indices = ({name: [] for name in
                                (("anchor_100",) if bucket_mode == "fixed_100" else
                                 ("anchor_100", "lower", "middle", "upper"))}
                               if bucket_mode != "none" else None)
        self.bucket_positions = ({name: {} for name in self.bucket_indices}
                                 if self.bucket_indices is not None else None)
        self.slot_buckets = []
        # 每条转移的进度段（0..progress_strata-1）；progress_strata==1 时保持空表，
        # 旧断点的状态字典因此可以原样加载。
        self.slot_segments = []
        self.segment_indices = ({key: [[] for _ in range(progress_strata)] for key in (self.bucket_indices or {})}
                                if self.bucket_indices is not None and progress_strata > 1 else None)
        self.sampled_by_bucket = {key: 0 for key in (self.bucket_indices or {})}
        self.replacement_draws = {key: 0 for key in (self.bucket_indices or {})}
        self.sampled_by_stratum = {}
        self.replacement_draws_by_stratum = {}
        # 逐束实体档位：每条转移的真实束数（与 rows 同步扩展；其他档位保持空表）。
        self.slot_entity_counts = []
        # 逐束实体档位：本条转移的真实束数，由 act() 记录、observe() 使用。
        self.pending_entity_count = None

    def _stratum(self, bucket, segment):
        return f"{bucket}|p{segment}"

    def __len__(self):
        return len(self.rows)

    def add(self, state, action, reward, next_state, done, action_mask=None, next_action_mask=None,
            bucket=None, progress_segment=None, entity_count=None):
        if self.padded_width is not None and entity_count is None:
            # 逐束实体档位：观测按束数上限补零，自然布局里没有可还原束数的标量，
            # 因此真实束数必须在写入时记录，采样时按它屏蔽补零实体。
            raise ValueError("逐束实体档位要求在写入转移时给出真实束数")
        row = (np.asarray(state, dtype=np.float32).copy(),
               tuple(int(x) for x in action), float(reward),
               np.asarray(next_state, dtype=np.float32).copy(), bool(done))
        row = row if action_mask is None else row + (action_mask, next_action_mask)
        if self.padded_width is not None:
            # 逐束实体档位：把真实束数存进附加表（不改动行结构，避免影响既有
            # 掩码字段的解析与旧断点兼容）。
            if self.bucket_mode == "none":
                # deque 满后会从左侧丢弃最旧行；束数表必须同步左移。
                if len(self.slot_entity_counts) != len(self.rows):
                    raise ValueError("逐束实体档位的回放束数表与行数不同步")
                if len(self.rows) == self.capacity:
                    self.slot_entity_counts.pop(0)
                self.slot_entity_counts.append(int(entity_count))
            else:
                # ring 满后只覆盖 next_position，不能按 len(rows)+1 扩出
                # 从未被索引的第 capacity+1 个零元素。
                index = self.next_position
                if index == len(self.slot_entity_counts):
                    self.slot_entity_counts.append(int(entity_count))
                elif 0 <= index < len(self.slot_entity_counts) == len(self.rows):
                    self.slot_entity_counts[index] = int(entity_count)
                else:
                    raise ValueError("逐束实体档位的回放束数表与环位置不同步")
        if self.segment_indices is not None:
            if progress_segment is None or not 0 <= int(progress_segment) < self.progress_strata:
                raise ValueError("进度分层采样要求每条转移给出有效的回合内进度段")
            progress_segment = int(progress_segment)
        elif progress_segment is not None:
            raise ValueError("进度分层采样未启用")
        if self.bucket_mode == "none":
            if bucket is not None:
                raise ValueError("Legacy replay does not accept a bucket")
            if progress_segment is not None:
                raise ValueError("Legacy replay does not stratify by progress")
            self.rows.append(row)
            return
        if bucket not in self.bucket_indices:
            raise ValueError("Transition bucket does not match configured sampling mode")
        index = self.next_position
        if index < len(self.rows):
            old_bucket = self.slot_buckets[index]
            members = self.bucket_indices[old_bucket]
            old_position = self.bucket_positions[old_bucket].pop(index)
            last_index = members.pop()
            if old_position < len(members):
                members[old_position] = last_index
                self.bucket_positions[old_bucket][last_index] = old_position
            if self.segment_indices is not None:
                self._remove_segment(old_bucket, self.slot_segments[index], index)
            self.rows[index] = row
            self.slot_buckets[index] = bucket
        else:
            self.rows.append(row)
            self.slot_buckets.append(bucket)
        self.bucket_positions[bucket][index] = len(self.bucket_indices[bucket])
        self.bucket_indices[bucket].append(index)
        if self.segment_indices is not None:
            self._insert_segment(bucket, progress_segment, index)
        self.next_position = (index + 1) % self.capacity

    def _insert_segment(self, bucket, segment, index):
        if len(self.slot_segments) <= index:
            self.slot_segments.append(segment)
        else:
            self.slot_segments[index] = segment
        self.segment_indices[bucket][segment].append(index)

    def _remove_segment(self, bucket, segment, index):
        members = self.segment_indices[bucket][segment]
        position = members.index(index)
        members.pop(position)

    def sample(self, count, device, action_dims=None):
        if self.bucket_mode == "none":
            rows = self.rng.sample(self.rows, count)
        else:
            strata = self.progress_strata if self.segment_indices is not None else 1
            grid = len(self.bucket_indices) * strata
            if count % grid:
                raise ValueError("Batch size must divide evenly across replay buckets and progress strata")
            per_cell = count // grid
            indices = []
            for bucket in self.bucket_indices:
                drawn_total = 0
                for segment in range(strata):
                    members = (self.segment_indices[bucket][segment] if self.segment_indices is not None
                               else self.bucket_indices[bucket])
                    if not members:
                        raise ValueError(f"Every active replay stratum needs a real transition: "
                                         f"bucket={bucket}, segment={segment}, strata={strata}")
                    chosen = (self.rng.sample(members, per_cell) if len(members) >= per_cell
                              else self.rng.choices(members, k=per_cell))
                    indices.extend(chosen)
                    self.sampled_by_bucket[bucket] += per_cell
                    drawn = per_cell - len(set(chosen)) if len(members) < per_cell else 0
                    drawn_total += drawn
                    if self.segment_indices is not None:
                        key = self._stratum(bucket, segment)
                        self.sampled_by_stratum[key] = self.sampled_by_stratum.get(key, 0) + per_cell
                        if drawn:
                            self.replacement_draws_by_stratum[key] = (
                                self.replacement_draws_by_stratum.get(key, 0) + drawn)
                # 桶级有放回次数保持历史口径：整桶所有分层的合计。
                self.replacement_draws[bucket] += drawn_total
            self.rng.shuffle(indices)
            rows = [self.rows[index] for index in indices]
        states, actions, rewards, next_states, dones = zip(*(row[:5] for row in rows))
        state_tensor, lengths = pad_states(states, device, padded_width=self.padded_width,
                                          entity_counts=([self.slot_entity_counts[index] for index in indices]
                                                         if self.padded_width is not None else None))
        next_state_tensor, next_lengths = pad_states(
            next_states, device, padded_width=self.padded_width,
            entity_counts=([self.slot_entity_counts[index] for index in indices]
                           if self.padded_width is not None else None))
        result = (state_tensor, torch.as_tensor(actions, dtype=torch.long, device=device),
                  torch.as_tensor(rewards, dtype=torch.float32, device=device),
                  next_state_tensor, torch.as_tensor(dones, dtype=torch.float32, device=device),
                  lengths, next_lengths)
        if action_dims is None:
            return result
        if any(len(row) != 7 for row in rows):
            raise ValueError("masked SAC replay row has no action masks")
        return result + (mask_batch([row[5] for row in rows], action_dims, device),
                         mask_batch([row[6] for row in rows], action_dims, device))

    def state_dict(self):
        if self.padded_width is not None and len(self.slot_entity_counts) != len(self.rows):
            raise ValueError("逐束实体档位的回放束数表与行数不同步")
        if self.bucket_mode == "none":
            data = {"rows": list(self.rows), "rng": self.rng.getstate(),
                    "capacity": self.rows.maxlen}
            if self.padded_width is not None:
                data["slot_entity_counts"] = list(self.slot_entity_counts)
            return data
        data = {"rows": list(self.rows), "rng": self.rng.getstate(), "capacity": self.capacity,
                "bucket_mode": self.bucket_mode, "next_position": self.next_position,
                "slot_buckets": list(self.slot_buckets),
                "bucket_indices": {key: list(value) for key, value in self.bucket_indices.items()},
                "sampled_by_bucket": dict(self.sampled_by_bucket),
                "replacement_draws": dict(self.replacement_draws)}
        if self.segment_indices is not None:
            data["progress_strata"] = self.progress_strata
            data["slot_segments"] = list(self.slot_segments)
            data["sampled_by_stratum"] = dict(self.sampled_by_stratum)
            data["replacement_draws_by_stratum"] = dict(self.replacement_draws_by_stratum)
        if self.padded_width is not None:
            data["slot_entity_counts"] = list(self.slot_entity_counts)
        return data

    def load_state_dict(self, data):
        if data["capacity"] != self.capacity:
            raise ValueError("Replay capacity changed")
        if data.get("bucket_mode", "none") != self.bucket_mode:
            raise ValueError("Replay bucket mode changed")
        # 旧断点没有进度分层字段：按原方式加载，只补空的分层计数。
        saved_strata = data.get("progress_strata")
        if saved_strata is not None and int(saved_strata) != self.progress_strata:
            raise ValueError("Replay progress strata changed")
        if self.bucket_mode == "none":
            self.rows = deque(data["rows"], maxlen=self.capacity)
            if self.padded_width is not None:
                saved_counts = data.get("slot_entity_counts")
                if saved_counts is None or len(saved_counts) != len(self.rows):
                    raise ValueError("逐束实体档位的回放缺真实束数记录")
                if any(type(value) is not int or value <= 0 for value in saved_counts):
                    raise ValueError("逐束实体档位的回放真实束数非法")
                self.slot_entity_counts = [int(value) for value in saved_counts]
        else:
            rows, buckets = list(data["rows"]), list(data["slot_buckets"])
            if len(rows) != len(buckets) or len(rows) > self.capacity:
                raise ValueError("Replay ring length mismatch")
            indices = {key: [] for key in self.bucket_indices}
            saved_indices = data["bucket_indices"]
            if set(saved_indices) != set(indices):
                raise ValueError("Replay checkpoint bucket names differ")
            for index, bucket in enumerate(buckets):
                if bucket not in indices:
                    raise ValueError("Unknown checkpoint replay bucket")
                indices[bucket].append(index)
            if any(sorted(value) != sorted(saved_indices[key]) for key, value in indices.items()):
                raise ValueError("Replay bucket index differs from stored rows")
            position = data["next_position"]
            if not isinstance(position, int) or position < 0 or position >= self.capacity or (len(rows) < self.capacity and position != len(rows)):
                raise ValueError("Invalid replay ring position")
            self.rows, self.slot_buckets = rows, buckets
            self.bucket_indices = {key: list(saved_indices[key]) for key in indices}
            self.bucket_positions = {key: {index: offset for offset, index in enumerate(value)}
                                     for key, value in self.bucket_indices.items()}
            self.next_position = position
            self.sampled_by_bucket = dict(data["sampled_by_bucket"])
            self.replacement_draws = dict(data["replacement_draws"])
            if self.padded_width is not None:
                saved_counts = data.get("slot_entity_counts")
                if (saved_counts is not None and len(rows) == self.capacity and
                        len(saved_counts) == len(rows) + 1 and
                        type(saved_counts[-1]) is int and saved_counts[-1] == 0):
                    # 旧满环断点的唯一已知写盘缺陷：多保存一个从未被采样
                    # 或覆盖的尾部零。前 capacity 条束数与行索引一一对应。
                    saved_counts = saved_counts[:-1]
                if saved_counts is None or len(saved_counts) != len(rows):
                    raise ValueError("逐束实体档位的回放缺真实束数记录")
                if any(type(value) is not int or value <= 0 for value in saved_counts):
                    raise ValueError("逐束实体档位的回放真实束数非法")
                self.slot_entity_counts = [int(value) for value in saved_counts]
            if self.segment_indices is not None:
                saved_segments = data.get("slot_segments")
                if saved_segments is not None:
                    if len(saved_segments) != len(rows):
                        raise ValueError("Replay progress segment ring length mismatch")
                    self.slot_segments = [int(value) for value in saved_segments]
                    self.segment_indices = {key: [[] for _ in range(self.progress_strata)]
                                            for key in self.bucket_indices}
                    for index, (bucket, segment) in enumerate(zip(buckets, self.slot_segments)):
                        if not 0 <= segment < self.progress_strata:
                            raise ValueError("Checkpoint replay progress segment out of range")
                        self.segment_indices[bucket][segment].append(index)
                    self.sampled_by_stratum = dict(data.get("sampled_by_stratum", {}))
                    self.replacement_draws_by_stratum = dict(data.get("replacement_draws_by_stratum", {}))
                else:
                    # 旧断点：按桶内顺序均匀回填进度段；这一映射只影响“旧断点继续训练”
                    # 的分层配额，不改变已保存的转移本身。
                    self.slot_segments = [0] * len(rows)
                    self.segment_indices = {key: [[] for _ in range(self.progress_strata)]
                                            for key in self.bucket_indices}
                    for index, bucket in enumerate(buckets):
                        members = self.bucket_indices[bucket]
                        segment = members.index(index) * self.progress_strata // max(1, len(members))
                        self.slot_segments[index] = segment
                        self.segment_indices[bucket][segment].append(index)
        self.rng.setstate(data["rng"])

    @property
    def capacity(self):
        return self.rows.maxlen if self.bucket_mode == "none" else self._capacity

    @capacity.setter
    def capacity(self, value):
        self._capacity = value


已定义 C_REPLAY: src/satellite_rl/agents/workbench_algorithms.py:169-458


### 双 Q、actor、熵及混合回放更新

先读 `__init__`：它创建 actor、两个 Q、两个目标 Q 和回放池。再读 `act` 的掩码选择、`_sample_masked_replay` 的 C1 半数教师混合、`_update_masked_sac` 的目标 Q 与 actor 更新，最后看 `state_dict` 怎样保留优化器和回放状态。这个类很长，按这条路径读即可。

下格是冻结源码的完整 `WorkbenchAgent` 定义，来源 `src/satellite_rl/agents/workbench_algorithms.py:461-1186`，提交 `48c7c4feac71`；运行时逐字节核对。


In [10]:
%%project_module C_AGENT
class WorkbenchAgent:
    def __init__(self, config, state_dim, action_dims, device="cpu"):
        self.config = config
        self.train = config["training"]
        self.device = torch.device(device)
        self.algorithm = config["algorithm"]
        self.action_mask_mode = config.get("environment", {}).get("action_mask", "none")
        if self.action_mask_mode not in ("none", "joint_feasible"):
            raise ValueError(f"Unsupported action mask mode: {self.action_mask_mode}")
        if self.action_mask_mode != "none" and self.algorithm != "sac":
            raise ValueError("joint_feasible action mask currently supports SAC only")
        if self.algorithm == "ppo" and config["network"]["dropout"]:
            raise ValueError("PPO does not support dropout: action and update probabilities must match")
        self.action_dims = tuple(action_dims)
        self.progress_strata = int(self.train.get("progress_strata", 1))
        self.variable_width = (config["network"].get("context_beam_count", "fixed_100") == "actual")
        if self.variable_width and (config["network"].get("kind") != "transformer" or
                                    config["network"].get("temporal_mode", "none") != "none"):
            raise ValueError("逐束实体编码（按实际束数）暂不支持跨波束记忆模式")
        if self.variable_width and (self.algorithm != "sac" or
                                    self.action_mask_mode != "joint_feasible"):
            raise ValueError("本轮逐束实体编码仅支持 SAC＋联合可行动作掩码")
        self.temporal_mode = config["network"].get("temporal_mode", "none")
        self.temporal_window = int(config["network"].get("temporal_window", 16))
        self.temporal_enabled = self.temporal_mode in ("gru", "causal_transformer")
        if self.temporal_mode not in ("none", "previous_action", "gru", "causal_transformer"):
            raise ValueError("Unsupported temporal mode")
        if self.temporal_enabled and (self.algorithm != "sac" or
                                      self.action_mask_mode != "joint_feasible" or
                                      not 1 <= self.temporal_window <= 16 or state_dim != 211):
            raise ValueError("Temporal SAC requires joint_feasible, 211 fields and a 1–16 step window")
        self.online_history = deque(maxlen=self.temporal_window) if self.temporal_enabled else None
        self.alpha_mode = self.train.get("alpha_mode", "adaptive")
        if self.alpha_mode not in ("adaptive", "fixed"):
            raise ValueError("Unsupported SAC alpha mode")
        if self.alpha_mode == "fixed" and self.algorithm != "sac":
            raise ValueError("Fixed alpha requires SAC")
        self.sac_policy_structure = config["network"].get("sac_policy_structure", "factorized")
        self.sac_critic_structure = config["network"].get("sac_critic_structure", "additive")
        if self.sac_policy_structure not in ("factorized", "conditional"):
            raise ValueError("Unsupported SAC policy structure")
        if self.sac_critic_structure not in ("additive", "joint"):
            raise ValueError("Unsupported SAC critic structure")
        if (self.sac_policy_structure != "factorized" or self.sac_critic_structure != "additive"):
            if self.algorithm != "sac" or self.action_mask_mode != "joint_feasible":
                raise ValueError("New SAC structures require SAC with joint_feasible action masking")
            if self.action_dims != (100, 10, 10):
                raise ValueError("New SAC structures require action dimensions (100, 10, 10)")
        self.target_entropy_ratio = (entropy_ratio(self.train["target_entropy"], self.action_dims)
                                     if self.action_mask_mode == "joint_feasible" else None)
        self.rng = random.Random(self.train["seed"] + 31)
        self.update_step = 0
        self.last_action_info = {}
        if self.algorithm == "sac":
            actor_class = (TemporalMultiHeadNetwork if self.temporal_enabled else
                           ConditionalPolicyNetwork if self.sac_policy_structure == "conditional"
                           else MultiHeadNetwork)
            critic_class = (TemporalMultiHeadNetwork if self.temporal_enabled else
                            JointQNetwork if self.sac_critic_structure == "joint" else MultiHeadNetwork)
            self.actor = actor_class(config, state_dim, action_dims).to(self.device)
            self.critic1 = critic_class(config, state_dim, action_dims).to(self.device)
            self.critic2 = critic_class(config, state_dim, action_dims).to(self.device)
            self.target1 = copy.deepcopy(self.critic1).to(self.device).eval()
            self.target2 = copy.deepcopy(self.critic2).to(self.device).eval()
            self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=self.train["actor_lr"])
            self.critic1_opt = torch.optim.Adam(self.critic1.parameters(), lr=self.train["critic_lr"])
            self.critic2_opt = torch.optim.Adam(self.critic2.parameters(), lr=self.train["critic_lr"])
            alpha_value = (self.train["fixed_alpha"] if self.alpha_mode == "fixed"
                           else self.train["initial_alpha"])
            self.log_alpha = nn.Parameter(torch.tensor(np.log(alpha_value),
                                                       dtype=torch.float32, device=self.device),
                                          requires_grad=self.alpha_mode == "adaptive")
            self.alpha_opt = torch.optim.Adam([self.log_alpha], lr=self.train["alpha_lr"])
            self.replay = (SequenceReplay(self.train["replay_capacity"], self.train["seed"] + 41,
                                          self.temporal_window) if self.temporal_enabled else
                           Replay(self.train["replay_capacity"], self.train["seed"] + 41,
                                  self.train.get("beam_count_mode", "none"),
                                  self.progress_strata,
                                  padded_width=state_dim if self.variable_width else None))
            self.demonstrations = None
            demonstration_config = self.train.get("demonstrations", {})
            if demonstration_config.get("mode", "none") == "half":
                if self.action_mask_mode != "joint_feasible" or not self.variable_width:
                    raise ValueError("S2-P19 demonstrations require masked variable-N SAC")
                from satellite_rl.workbench.s2p19_demonstrations import DemonstrationDataset
                self.demonstrations = DemonstrationDataset(
                    demonstration_config["manifest_path"],
                    demonstration_config["manifest_sha256"], config,
                    demonstration_config["source_set_sha256"], self.train["seed"] + 47)
        elif self.algorithm == "ppo":
            self.actor = MultiHeadNetwork(config, state_dim, action_dims).to(self.device)
            self.value = ScalarNetwork(config, state_dim).to(self.device)
            self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=self.train["actor_lr"])
            self.value_opt = torch.optim.Adam(self.value.parameters(), lr=self.train["critic_lr"])
            self.rollout = []
        elif self.algorithm == "dqn":
            self.online = MultiHeadNetwork(config, state_dim, action_dims).to(self.device)
            self.target = copy.deepcopy(self.online).to(self.device).eval()
            self.optimizer = torch.optim.Adam(self.online.parameters(), lr=self.train["critic_lr"])
            self.replay = Replay(self.train["replay_capacity"], self.train["seed"] + 41)
        else:
            raise ValueError(f"Unsupported algorithm: {self.algorithm}")

    def policy_network(self):
        return self.online if self.algorithm == "dqn" else self.actor

    def reset_episode(self):
        if self.temporal_enabled:
            self.online_history.clear()

    def _net(self, module, states, lengths=None):
        """统一网络调用：逐束实体编码模式下传入本批有效宽度与填充掩码。

        其他布局的 ``Encoder.forward`` 只接受状态张量，因此这里只在变宽模式下追加
        第二、三个参数，旧路径的调用与返回值逐位不变。
        """
        if not self.variable_width:
            return module(states)
        return module(states, lengths=lengths)

    def _sac_actor_logits(self, states, lengths=None):
        """Match action-time probabilities even when the actor has dropout."""
        if self.temporal_enabled:
            # Temporal modes require dropout=0. cuDNN GRU backward cannot run
            # after eval-mode forward; train/eval probabilities are identical.
            return self.actor(states)
        was_training = self.actor.training
        self.actor.eval()
        try:
            return self._net(self.actor, states, lengths)
        finally:
            self.actor.train(was_training)

    def act(self, state, deterministic=False, env_step=None, action_mask=None,
            beam_entity_count=None):
        """按当前观测选择一个动作。

        ``beam_entity_count``：逐束实体档位下本场景的真实束数。观测按场景束数上限
        补零，因此必须显式给出真实束数，编码器才会只对真实实体做注意力池化、并把
        补零行排除在读出与 N 条件调制之外（占位不参与池化与统计）。
        """
        if self.action_mask_mode == "joint_feasible":
            action_mask = copy_action_mask(action_mask, self.action_dims)
        elif action_mask is not None:
            raise ValueError("action_mask supplied while masking is disabled")
        net = self.policy_network()
        was_training = net.training
        net.eval()
        with torch.no_grad():
            x = _state_tensor(state, self.device).reshape(1, -1)
            lengths = None
            if self.variable_width:
                if beam_entity_count is None:
                    raise ValueError("逐束实体档位必须在选择动作时给出真实束数")
                lengths = int(beam_entity_count)
                # 逐束实体档位：记下本条转移的真实束数，observe() 写入回放时使用
                # （自然布局的观测里没有可还原束数的标量）。
                self.pending_entity_count = lengths
            if self.temporal_enabled:
                self.online_history.append(np.asarray(state, dtype=np.float32).copy())
                sequence = _state_tensor(list(self.online_history), self.device).unsqueeze(0)
                x = (sequence, torch.as_tensor([len(self.online_history)], device=self.device))
            heads = net(x) if lengths is None else self._net(net, x, lengths)
            if self.algorithm == "dqn":
                epsilon = 0.0 if deterministic else self.train["eps_end"]
                if not deterministic and env_step is not None:
                    epsilon += (self.train["eps_start"] - self.train["eps_end"]) * max(
                        0.0, 1.0 - env_step / self.train["eps_decay_steps"])
                action = tuple(int(torch.argmax(q, dim=1).item()) if deterministic else
                               (self.rng.randrange(dim) if self.rng.random() < epsilon
                                else int(torch.argmax(q, dim=1).item()))
                               for q, dim in zip(heads, self.action_dims))
                self.last_action_info = {"epsilon": epsilon, "q_mean": float(_gather(
                    heads, torch.tensor([action], device=self.device)).item())}
            elif self.action_mask_mode == "joint_feasible":
                frequency_slots, power = mask_batch([action_mask], self.action_dims, self.device)
                if self.sac_policy_structure == "conditional":
                    (fl, conditional_power, log_fl, log_p, entropy_fl, entropy_p,
                     entropy_f, entropy_slots) = conditional_masked_policy(
                        heads, frequency_slots, power)
                    joint = fl.unsqueeze(-1) * conditional_power
                    if deterministic:
                        index = joint.flatten(1).argmax(1)
                        frequency = index // (self.action_dims[1] * self.action_dims[2])
                        slots = index // self.action_dims[2] % self.action_dims[1]
                        power_index = index % self.action_dims[2]
                    else:
                        fl_index = torch.distributions.Categorical(probs=fl.flatten(1)).sample()
                        frequency = fl_index // self.action_dims[1]
                        slots = fl_index % self.action_dims[1]
                        power_index = torch.distributions.Categorical(
                            probs=conditional_power[0, frequency.item(), slots.item()].reshape(1, -1)).sample()
                    action = (int(frequency.item()), int(slots.item()), int(power_index.item()))
                    self.last_action_info = {
                        "log_prob": float((log_fl[0, action[0], action[1]] +
                                           log_p[0, action[0], action[1], action[2]]).item()),
                        "entropy": float((entropy_fl + entropy_p).item()),
                        "entropy_frequency": float(entropy_f.item()),
                        "entropy_slots": float(entropy_slots.item()),
                        "entropy_power": float(entropy_p.item()),
                        "target_entropy": float(feasible_target_entropy(
                            frequency_slots, power, self.target_entropy_ratio).item()),
                    }
                else:
                    (fl, p, log_fl, log_p, entropy_fl, entropy_p,
                     entropy_f, entropy_slots) = masked_policy(heads, frequency_slots, power)
                    flat_fl = fl.flatten(1)
                    fl_index = (flat_fl.argmax(1) if deterministic else
                                torch.distributions.Categorical(probs=flat_fl).sample())
                    power_index = (p.argmax(1) if deterministic else
                                   torch.distributions.Categorical(probs=p).sample())
                    frequency = fl_index // self.action_dims[1]
                    slots = fl_index % self.action_dims[1]
                    action = (int(frequency.item()), int(slots.item()), int(power_index.item()))
                    self.last_action_info = {
                        "log_prob": float((log_fl[0, action[0], action[1]] +
                                           log_p[0, action[2]]).item()),
                        "entropy": float((entropy_fl + entropy_p).item()),
                        "entropy_frequency": float(entropy_f.item()),
                        "entropy_slots": float(entropy_slots.item()),
                        "entropy_power": float(entropy_p.item()),
                        "target_entropy": float(feasible_target_entropy(
                            frequency_slots, power, self.target_entropy_ratio).item()),
                    }
            else:
                dists = [torch.distributions.Categorical(logits=logits) for logits in heads]
                action_t = [torch.argmax(logits, 1) if deterministic else dist.sample()
                            for logits, dist in zip(heads, dists)]
                action = tuple(int(a.item()) for a in action_t)
                self.last_action_info = {
                    "log_prob": float(sum(dist.log_prob(a) for dist, a in zip(dists, action_t)).item()),
                    "entropy": float(sum(dist.entropy() for dist in dists).item()),
                    **{f"entropy_{name}": float(dist.entropy().item()) for name, dist in zip(HEADS, dists)},
                }
                if self.algorithm == "ppo":
                    value_was_training = self.value.training
                    self.value.eval()
                    self.last_action_info["value"] = float(self._net(self.value, x).item())
                    self.value.train(value_was_training)
                    self.last_action_info["action"] = action
        net.train(was_training)
        return action

    def observe(self, state, action, reward, next_state, done,
                action_mask=None, next_action_mask=None, actual_action=None, replay_bucket=None,
                progress_segment=None, entity_count=None):
        if entity_count is None and self.variable_width:
            entity_count = getattr(self, "pending_entity_count", None)
        if self.action_mask_mode == "joint_feasible":
            action_mask = copy_action_mask(action_mask, self.action_dims)
            if next_action_mask is None:
                if not done:
                    raise ValueError("nonterminal masked SAC transition requires next_action_mask")
            else:
                if done:
                    raise ValueError("terminal masked SAC transition must have no next_action_mask")
                next_action_mask = copy_action_mask(next_action_mask, self.action_dims)
            chosen = tuple(int(x) for x in action)
            if (len(chosen) != 3 or any(x < 0 or x >= dim for x, dim in zip(chosen, self.action_dims)) or
                    not action_mask["frequency_slots"][chosen[0], chosen[1]] or
                    not action_mask["power"][chosen[2]]):
                raise ValueError("observed action is forbidden by action_mask")
            if self.temporal_enabled:
                self.replay.add(state, chosen, reward, next_state, done,
                                action_mask, next_action_mask, actual_action)
            else:
                self.replay.add(state, chosen, reward, next_state, done, action_mask, next_action_mask,
                                bucket=replay_bucket, progress_segment=progress_segment,
                                entity_count=entity_count)
            return
        if action_mask is not None or next_action_mask is not None:
            raise ValueError("action_mask supplied while masking is disabled")
        if self.algorithm == "ppo":
            if tuple(action) != tuple(self.last_action_info.get("action", ())):
                raise ValueError("PPO observed action differs from the sampled action and log-probability")
            self.rollout.append((np.asarray(state, dtype=np.float32).copy(), tuple(action),
                                 float(reward), np.asarray(next_state, dtype=np.float32).copy(), bool(done),
                                 self.last_action_info["log_prob"], self.last_action_info["value"]))
        else:
            self.replay.add(state, action, reward, next_state, done, bucket=replay_bucket,
                            progress_segment=progress_segment)

    def ready(self, env_step, done=False):
        if self.algorithm == "ppo":
            return len(self.rollout) >= self.train["rollout_steps"] or (done and bool(self.rollout))
        buckets_ready = (not self.replay.bucket_indices or
                         all(self.replay.bucket_indices.values()))
        strata_ready = (self.replay.segment_indices is None or
                        all(all(cell for cell in bucket) for bucket in self.replay.segment_indices.values()))
        online_minimum = (self.train["batch_size"] // 2 if getattr(self, "demonstrations", None)
                          is not None else self.train["batch_size"])
        return (env_step >= self.train["warmup_steps"] and buckets_ready and strata_ready and
                len(self.replay) >= online_minimum and
                env_step % self.train["update_every"] == 0)

    def _sample_masked_replay(self):
        """Return a real 50:50 C1 batch; offline rows never increment env_step."""
        if self.demonstrations is None:
            return self.replay.sample(self.train["batch_size"], self.device, self.action_dims)
        half = self.train["batch_size"] // 2
        online = self.replay.sample(half, self.device, self.action_dims)
        teacher = self.demonstrations.sample(half, self.device, self.action_dims)
        return tuple((torch.cat((a[0], b[0])), torch.cat((a[1], b[1])))
                     if index in (7, 8) else torch.cat((a, b))
                     for index, (a, b) in enumerate(zip(online, teacher)))

    def gamma_for(self, beam_count=None):
        """S2-P18 手段D：当前回合实际使用的折扣。"""
        return resolve_gamma(self.train, beam_count)

    def update(self):
        if self.algorithm == "sac":
            return self._update_sac()
        if self.algorithm == "ppo":
            return self._update_ppo()
        return self._update_dqn()

    def _update_sac(self):
        if self.action_mask_mode == "joint_feasible":
            return self._update_masked_sac()
        s, a, r, ns, d, lengths, next_lengths = self.replay.sample(
            self.train["batch_size"], self.device)
        gamma = self.train["gamma"]
        alpha = self.log_alpha.exp().detach()
        with torch.no_grad():
            logits = self._sac_actor_logits(ns)
            probabilities = [q.softmax(-1) for q in logits]
            joint_prob = probabilities[0][:, :, None, None] * probabilities[1][:, None, :, None] * probabilities[2][:, None, None, :]
            min_q = torch.minimum(_joint_q(self.target1(ns)), _joint_q(self.target2(ns)))
            expected_q = (joint_prob * min_q).sum((1, 2, 3))
            entropy = sum(-(p * q.log_softmax(-1)).sum(1) for p, q in zip(probabilities, logits))
            target = r + gamma * (1 - d) * (expected_q + alpha * entropy)
        q1 = _gather(self.critic1(s), a)
        q2 = _gather(self.critic2(s), a)
        loss1, loss2 = F.mse_loss(q1, target), F.mse_loss(q2, target)
        _finite_loss(loss1 + loss2)
        self.critic1_opt.zero_grad(set_to_none=True)
        loss1.backward()
        grad1 = _grad_norm(self.critic1, self.train["gradient_clip"])
        self.critic1_opt.step()
        self.critic2_opt.zero_grad(set_to_none=True)
        loss2.backward()
        grad2 = _grad_norm(self.critic2, self.train["gradient_clip"])
        self.critic2_opt.step()
        logits = self._sac_actor_logits(s)
        probabilities = [q.softmax(-1) for q in logits]
        joint_prob = probabilities[0][:, :, None, None] * probabilities[1][:, None, :, None] * probabilities[2][:, None, None, :]
        with torch.no_grad():
            min_q_actor = torch.minimum(_joint_q(self.critic1(s)), _joint_q(self.critic2(s)))
        expected_q_actor = (joint_prob * min_q_actor).sum((1, 2, 3))
        entropies = [-(p * q.log_softmax(-1)).sum(1) for p, q in zip(probabilities, logits)]
        entropy_actor = sum(entropies)
        actor_loss = -(expected_q_actor + alpha * entropy_actor).mean()
        _finite_loss(actor_loss)
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        actor_grad = _grad_norm(self.actor, self.train["gradient_clip"])
        self.actor_opt.step()
        if self.alpha_mode == "adaptive":
            alpha_loss = (self.log_alpha * (entropy_actor.detach().mean() - self.train["target_entropy"]))
            _finite_loss(alpha_loss)
            self.alpha_opt.zero_grad(set_to_none=True)
            alpha_loss.backward()
            self.alpha_opt.step()
        else:
            alpha_loss = torch.zeros_like(self.log_alpha.detach())
        self.update_step += 1
        tau = self.train["tau"]
        with torch.no_grad():
            for target_net, source_net in ((self.target1, self.critic1), (self.target2, self.critic2)):
                for target_p, source_p in zip(target_net.parameters(), source_net.parameters()):
                    target_p.lerp_(source_p, tau)
        return _diagnostics({"actor_loss": actor_loss, "critic_1_loss": loss1,
                             "critic_2_loss": loss2, "alpha_loss": alpha_loss,
                             "alpha": self.log_alpha.exp(), "entropy": entropy_actor.mean(),
                             **{f"entropy_{name}": value.mean() for name, value in zip(HEADS, entropies)},
                             "q_mean": (q1.mean() + q2.mean()) / 2,
                             "q_min": torch.minimum(q1, q2).min(),
                             "q_max": torch.maximum(q1, q2).max(), "target_mean": target.mean(),
                             "target_std": target.std(unbiased=False),
                             "td_abs_mean": ((q1 - target).abs().mean() + (q2 - target).abs().mean()) / 2,
                             "actor_grad_norm": actor_grad, "critic_1_grad_norm": grad1,
                             "critic_2_grad_norm": grad2,
                             **({"alpha_update_applied": 0} if self.alpha_mode == "fixed" else {})})

    def _update_masked_sac(self):
        if self.temporal_enabled:
            return self._update_temporal_sac()
        if self.sac_policy_structure != "factorized" or self.sac_critic_structure != "additive":
            return self._update_joint_action_sac()
        s, a, r, ns, d, lengths, next_lengths, current_masks, next_masks = self._sample_masked_replay()
        current_fl, current_power = current_masks
        next_fl, next_power = next_masks
        alpha = self.log_alpha.exp().detach()
        with torch.no_grad():
            target = r.clone()
            active = d < 0.5
            if active.any():
                (next_prob_fl, next_prob_power, _, _, next_entropy_fl,
                 next_entropy_power, _, _) = masked_policy(
                    self._sac_actor_logits(ns[active], next_lengths[active]),
                    next_fl[active], next_power[active])
                next_min_q = torch.minimum(_joint_q(self._net(self.target1, ns[active], next_lengths[active])),
                                           _joint_q(self._net(self.target2, ns[active], next_lengths[active])))
                next_expected_q = torch.einsum("bfl,bp,bflp->b", next_prob_fl,
                                               next_prob_power, next_min_q)
                target[active] += self.train["gamma"] * (
                    next_expected_q + alpha * (next_entropy_fl + next_entropy_power))
        q1 = _gather(self._net(self.critic1, s, lengths), a)
        q2 = _gather(self._net(self.critic2, s, lengths), a)
        loss1, loss2 = F.mse_loss(q1, target), F.mse_loss(q2, target)
        _finite_loss(loss1 + loss2)
        self.critic1_opt.zero_grad(set_to_none=True)
        loss1.backward()
        grad1 = _grad_norm(self.critic1, self.train["gradient_clip"])
        self.critic1_opt.step()
        self.critic2_opt.zero_grad(set_to_none=True)
        loss2.backward()
        grad2 = _grad_norm(self.critic2, self.train["gradient_clip"])
        self.critic2_opt.step()
        (prob_fl, prob_power, _, _, entropy_fl, entropy_power,
         entropy_frequency, entropy_slots) = masked_policy(
             self._sac_actor_logits(s, lengths), current_fl, current_power)
        with torch.no_grad():
            min_q_actor = torch.minimum(_joint_q(self._net(self.critic1, s, lengths)),
                                        _joint_q(self._net(self.critic2, s, lengths)))
        expected_q_actor = torch.einsum("bfl,bp,bflp->b", prob_fl, prob_power, min_q_actor)
        entropy_actor = entropy_fl + entropy_power
        actor_loss = -(expected_q_actor + alpha * entropy_actor).mean()
        _finite_loss(actor_loss)
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        actor_grad = _grad_norm(self.actor, self.train["gradient_clip"])
        self.actor_opt.step()
        target_entropy = feasible_target_entropy(current_fl, current_power,
                                                  self.target_entropy_ratio)
        if self.alpha_mode == "adaptive":
            alpha_loss = self.log_alpha * (entropy_actor.detach() - target_entropy).mean()
            _finite_loss(alpha_loss)
            self.alpha_opt.zero_grad(set_to_none=True)
            alpha_loss.backward()
            self.alpha_opt.step()
        else:
            alpha_loss = torch.zeros_like(self.log_alpha.detach())
        self.update_step += 1
        tau = self.train["tau"]
        with torch.no_grad():
            for target_net, source_net in ((self.target1, self.critic1), (self.target2, self.critic2)):
                for target_p, source_p in zip(target_net.parameters(), source_net.parameters()):
                    target_p.lerp_(source_p, tau)
        return _diagnostics({"actor_loss": actor_loss, "critic_1_loss": loss1,
                             "critic_2_loss": loss2, "alpha_loss": alpha_loss,
                             "alpha": self.log_alpha.exp(), "entropy": entropy_actor.mean(),
                             "entropy_frequency": entropy_frequency.mean(),
                             "entropy_slots": entropy_slots.mean(),
                             "entropy_power": entropy_power.mean(),
                             "target_entropy": target_entropy.mean(),
                             "legal_frequency_slot_count": current_fl.sum((1, 2)).float().mean(),
                             "legal_power_count": current_power.sum(1).float().mean(),
                             "q_mean": (q1.mean() + q2.mean()) / 2,
                             "q_min": torch.minimum(q1, q2).min(),
                             "q_max": torch.maximum(q1, q2).max(), "target_mean": target.mean(),
                             "target_std": target.std(unbiased=False),
                             "td_abs_mean": ((q1 - target).abs().mean() + (q2 - target).abs().mean()) / 2,
                             "actor_grad_norm": actor_grad, "critic_1_grad_norm": grad1,
                             "critic_2_grad_norm": grad2,
                             **({"alpha_update_applied": 0} if self.alpha_mode == "fixed" else {})})

    def _masked_joint_distribution(self, states, frequency_slots, power, lengths=None):
        logits = self._sac_actor_logits(states, lengths)
        if self.sac_policy_structure == "conditional":
            parts = conditional_masked_policy(logits, frequency_slots, power)
            fl, conditional_power = parts[:2]
            joint = fl.unsqueeze(-1) * conditional_power
        else:
            parts = masked_policy(logits, frequency_slots, power)
            fl, power_probability = parts[:2]
            joint = fl.unsqueeze(-1) * power_probability[:, None, None, :]
        return joint, parts[4] + parts[5], parts[4], parts[5], parts[6], parts[7]

    def _critic_table(self, critic, states, lengths=None):
        values = self._net(critic, states, lengths)
        return values if self.sac_critic_structure == "joint" else _joint_q(values)

    def _update_temporal_sac(self):
        return self._update_joint_action_sac()

    def _update_joint_action_sac(self):
        batch = self.replay.sample(self.train["batch_size"], self.device, self.action_dims)
        if len(batch) == 7:
            # 跨波束记忆使用 SequenceReplay：没有变宽状态的长度字段。
            s, a, r, ns, d, current_masks, next_masks = batch
            lengths = next_lengths = None
        else:
            s, a, r, ns, d, lengths, next_lengths, current_masks, next_masks = batch
        current_fl, current_power = current_masks
        next_fl, next_power = next_masks
        alpha = self.log_alpha.exp().detach()
        with torch.no_grad():
            target = r.clone()
            active = d < 0.5
            if active.any():
                active_next = ((ns[0][active], ns[1][active]) if self.temporal_enabled
                               else ns[active])
                active_lengths = None if self.temporal_enabled else next_lengths[active]
                next_prob, next_entropy, *_ = self._masked_joint_distribution(
                    active_next, next_fl[active], next_power[active], active_lengths)
                next_min_q = torch.minimum(self._critic_table(self.target1, active_next, active_lengths),
                                           self._critic_table(self.target2, active_next, active_lengths))
                target[active] += self.train["gamma"] * (
                    (next_prob * next_min_q).sum((1, 2, 3)) + alpha * next_entropy)
        q1_table = self._critic_table(self.critic1, s, lengths)
        q2_table = self._critic_table(self.critic2, s, lengths)
        batch_index = torch.arange(a.shape[0], device=self.device)
        q1 = q1_table[batch_index, a[:, 0], a[:, 1], a[:, 2]]
        q2 = q2_table[batch_index, a[:, 0], a[:, 1], a[:, 2]]
        loss1, loss2 = F.mse_loss(q1, target), F.mse_loss(q2, target)
        _finite_loss(loss1 + loss2)
        self.critic1_opt.zero_grad(set_to_none=True)
        loss1.backward()
        grad1 = _grad_norm(self.critic1, self.train["gradient_clip"])
        self.critic1_opt.step()
        self.critic2_opt.zero_grad(set_to_none=True)
        loss2.backward()
        grad2 = _grad_norm(self.critic2, self.train["gradient_clip"])
        self.critic2_opt.step()
        probability, entropy, entropy_fl, entropy_power, entropy_frequency, entropy_slots = (
            self._masked_joint_distribution(s, current_fl, current_power, lengths))
        with torch.no_grad():
            min_q = torch.minimum(self._critic_table(self.critic1, s, lengths),
                                  self._critic_table(self.critic2, s, lengths))
        actor_loss = -((probability * min_q).sum((1, 2, 3)) + alpha * entropy).mean()
        _finite_loss(actor_loss)
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        actor_grad = _grad_norm(self.actor, self.train["gradient_clip"])
        self.actor_opt.step()
        target_entropy = feasible_target_entropy(current_fl, current_power,
                                                  self.target_entropy_ratio)
        if self.alpha_mode == "adaptive":
            alpha_loss = self.log_alpha * (entropy.detach() - target_entropy).mean()
            _finite_loss(alpha_loss)
            self.alpha_opt.zero_grad(set_to_none=True)
            alpha_loss.backward()
            self.alpha_opt.step()
        else:
            alpha_loss = torch.zeros_like(self.log_alpha.detach())
        self.update_step += 1
        with torch.no_grad():
            for target_net, source_net in ((self.target1, self.critic1), (self.target2, self.critic2)):
                for target_p, source_p in zip(target_net.parameters(), source_net.parameters()):
                    target_p.lerp_(source_p, self.train["tau"])
        temporal_diagnostics = ({"effective_td_targets": a.shape[0],
                                 "context_steps_mean": s[1].float().mean()}
                                if self.temporal_enabled else {})
        return _diagnostics({"actor_loss": actor_loss, "critic_1_loss": loss1,
                             "critic_2_loss": loss2, "alpha_loss": alpha_loss,
                             "alpha": self.log_alpha.exp(), "entropy": entropy.mean(),
                             "entropy_frequency": entropy_frequency.mean(),
                             "entropy_slots": entropy_slots.mean(),
                             "entropy_power": entropy_power.mean(),
                             "target_entropy": target_entropy.mean(),
                             "legal_frequency_slot_count": current_fl.sum((1, 2)).float().mean(),
                             "legal_power_count": current_power.sum(1).float().mean(),
                             "q_mean": (q1.mean() + q2.mean()) / 2,
                             "q_min": torch.minimum(q1, q2).min(),
                             "q_max": torch.maximum(q1, q2).max(), "target_mean": target.mean(),
                             "target_std": target.std(unbiased=False),
                             "td_abs_mean": ((q1 - target).abs().mean() + (q2 - target).abs().mean()) / 2,
                             "actor_grad_norm": actor_grad, "critic_1_grad_norm": grad1,
                             "critic_2_grad_norm": grad2,
                             **temporal_diagnostics,
                             **({"alpha_update_applied": 0} if self.alpha_mode == "fixed" else {})})

    def _update_dqn(self):
        s, a, r, ns, d, lengths, next_lengths = self.replay.sample(self.train["batch_size"], self.device)
        current = _gather(self._net(self.online, s, lengths), a)
        with torch.no_grad():
            next_actions = torch.stack([q.argmax(1) for q in self._net(self.online, ns, next_lengths)], 1)
            next_q = _gather(self._net(self.target, ns, next_lengths), next_actions)
            target = r + self.train["gamma"] * (1 - d) * next_q
        loss = F.smooth_l1_loss(current, target)
        _finite_loss(loss)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad = _grad_norm(self.online, self.train["gradient_clip"])
        self.optimizer.step()
        self.update_step += 1
        target_updated = self.update_step % self.train["target_update_interval"] == 0
        if target_updated:
            self.target.load_state_dict(self.online.state_dict())
        return _diagnostics({"dqn_loss": loss, "q_mean": current.mean(),
                             "q_min": current.min(), "q_max": current.max(),
                             "target_mean": target.mean(), "target_std": target.std(unbiased=False),
                             "td_abs_mean": (current - target).abs().mean(),
                             "critic_grad_norm": grad, "target_updated": int(target_updated)})

    def _update_ppo(self):
        rows = self.rollout
        states, actions, rewards, next_states, dones, old_logp, old_values = zip(*rows)
        with torch.no_grad():
            width_n = np.asarray(next_states[-1], dtype=np.float32).reshape(-1).shape[0]
            last_value = 0.0 if dones[-1] else float(self._net(
                self.value, _state_tensor(next_states[-1], self.device).reshape(1, -1), width_n).item())
        advantages = generalized_advantages(rewards, old_values, dones, last_value,
                                            self.train["gamma"], self.train["gae_lambda"])
        returns = advantages + np.asarray(old_values, dtype=np.float32)
        s = _state_tensor(states, self.device)
        a = torch.as_tensor(actions, dtype=torch.long, device=self.device)
        logp0 = torch.as_tensor(old_logp, dtype=torch.float32, device=self.device)
        adv = torch.as_tensor(advantages, device=self.device)
        adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)
        ret = torch.as_tensor(returns, device=self.device)
        metrics = {}
        for _ in range(self.train["ppo_epochs"]):
            dists = [torch.distributions.Categorical(logits=logits) for logits in self._net(self.actor, s)]
            logp = sum(dist.log_prob(a[:, i]) for i, dist in enumerate(dists))
            entropy_heads = [dist.entropy() for dist in dists]
            entropy = sum(entropy_heads)
            ratio = torch.exp(logp - logp0)
            clipped = torch.clamp(ratio, 1 - self.train["clip_ratio"], 1 + self.train["clip_ratio"])
            actor_loss = -(torch.minimum(ratio * adv, clipped * adv)).mean() - self.train["entropy_coef"] * entropy.mean()
            value_pred = self._net(self.value, s)
            value_loss = F.mse_loss(value_pred, ret) * self.train["value_coef"]
            _finite_loss(actor_loss + value_loss)
            self.actor_opt.zero_grad(set_to_none=True)
            actor_loss.backward()
            actor_grad = _grad_norm(self.actor, self.train["gradient_clip"])
            self.actor_opt.step()
            self.value_opt.zero_grad(set_to_none=True)
            value_loss.backward()
            value_grad = _grad_norm(self.value, self.train["gradient_clip"])
            self.value_opt.step()
            with torch.no_grad():
                residual = ret - value_pred
                explained = 1 - torch.var(residual, unbiased=False) / (torch.var(ret, unbiased=False) + 1e-8)
                metrics = _diagnostics({"actor_loss": actor_loss, "value_loss": value_loss,
                                        "entropy": entropy.mean(), "approx_kl": (logp0 - logp).mean(),
                                        "clip_fraction": ((ratio - 1).abs() > self.train["clip_ratio"]).float().mean(),
                                        "explained_variance": explained, "value_mean": value_pred.mean(),
                                        "return_target_mean": ret.mean(), "actor_grad_norm": actor_grad,
                                        "value_grad_norm": value_grad,
                                        **{f"entropy_{name}": value.mean() for name, value in zip(HEADS, entropy_heads)}})
            self.update_step += 1
        self.rollout = []
        return metrics

    def state_dict(self):
        names = ("actor", "critic1", "critic2", "target1", "target2", "value", "online", "target")
        optimizers = ("actor_opt", "critic1_opt", "critic2_opt", "alpha_opt", "value_opt", "optimizer")
        result = {"networks": {n: getattr(self, n).state_dict() for n in names if hasattr(self, n)},
                  "optimizers": {n: getattr(self, n).state_dict() for n in optimizers if hasattr(self, n)},
                  "log_alpha": self.log_alpha.detach().cpu() if hasattr(self, "log_alpha") else None,
                  "replay": self.replay.state_dict() if hasattr(self, "replay") else None,
                  "rollout": getattr(self, "rollout", None), "rng": self.rng.getstate(),
                  "update_step": self.update_step}
        if self.algorithm == "sac":
            result["policy_probability_mode"] = "eval_v1"
            result["alpha_mode"] = self.alpha_mode
            result["sac_policy_structure"] = self.sac_policy_structure
            result["sac_critic_structure"] = self.sac_critic_structure
            if self.temporal_mode != "none":
                result["temporal_mode"] = self.temporal_mode
                result["temporal_window"] = self.temporal_window
            if self.temporal_enabled:
                result["online_history"] = [np.asarray(row, dtype=np.float32).copy()
                                            for row in self.online_history]
            if self.demonstrations is not None:
                result["demonstrations"] = self.demonstrations.state_dict()
        return result

    def load_state_dict(self, data):
        if self.algorithm == "sac":
            if self.demonstrations is not None and "demonstrations" not in data:
                raise ValueError("C1 checkpoint lacks demonstration source and sampler state")
            if self.demonstrations is None and "demonstrations" in data:
                raise ValueError("C0 checkpoint cannot resume with C1 demonstration replay")
            if data.get("alpha_mode", "adaptive") != self.alpha_mode:
                raise ValueError("SAC checkpoint alpha mode differs from configured mode")
            if (data.get("temporal_mode", "none") != self.temporal_mode or
                    (self.temporal_enabled and data.get("temporal_window") != self.temporal_window)):
                raise ValueError("SAC checkpoint temporal mode/window differs")
            if data.get("sac_policy_structure", "factorized") != self.sac_policy_structure or \
                    data.get("sac_critic_structure", "additive") != self.sac_critic_structure:
                raise ValueError("SAC checkpoint action structures differ from configured structures")
            mode = data.get("policy_probability_mode")
            if mode is None:
                if self.action_mask_mode == "joint_feasible":
                    raise ValueError("Masked SAC checkpoint lacks policy_probability_mode; cannot resume")
                if self.config["network"]["dropout"] > 0:
                    raise ValueError("Historical SAC dropout checkpoint uses different policy probabilities; "
                                     "resume it with its original frozen source")
            elif mode != "eval_v1":
                raise ValueError(f"Unsupported SAC policy_probability_mode: {mode}")
            if self.action_mask_mode == "joint_feasible":
                replay = data.get("replay")
                if not isinstance(replay, dict) or not isinstance(replay.get("rows"), list):
                    raise ValueError("Masked SAC checkpoint lacks replay with action masks")
                for row in replay["rows"]:
                    if (len(row) != (10 if self.temporal_enabled else 7) or row[5] is None or
                            (row[6] is None) != bool(row[4])):
                        raise ValueError("Masked SAC checkpoint replay row lacks current/next action masks")
            if self.alpha_mode == "fixed" and not math.isclose(
                    float(data["log_alpha"].exp()), float(self.train["fixed_alpha"]),
                    rel_tol=1e-6, abs_tol=1e-8):
                raise ValueError("Fixed alpha checkpoint differs from configured value")
        for name, state in data["networks"].items():
            getattr(self, name).load_state_dict(state, strict=True)
        for name, state in data["optimizers"].items():
            getattr(self, name).load_state_dict(state)
        if data["log_alpha"] is not None:
            self.log_alpha.data.copy_(data["log_alpha"].to(self.device))
        if data["replay"] is not None:
            self.replay.load_state_dict(data["replay"])
        if data["rollout"] is not None:
            self.rollout = data["rollout"]
        self.rng.setstate(data["rng"])
        self.update_step = data["update_step"]
        if self.algorithm == "sac" and self.demonstrations is not None:
            self.demonstrations.load_state_dict(data["demonstrations"])
        if self.temporal_enabled:
            history = data.get("online_history")
            if not isinstance(history, list) or len(history) > self.temporal_window:
                raise ValueError("Temporal checkpoint online history missing or too long")
            self.online_history = deque((np.asarray(row, dtype=np.float32).copy()
                                         for row in history), maxlen=self.temporal_window)


已定义 C_AGENT: src/satellite_rl/agents/workbench_algorithms.py:461-1186


### 训练器构建入口

训练器通过这个入口拿到上面的 agent，网络宽度和动作维度都来自冻结配置。这里不做参数更新，真正更新发生在 `WorkbenchAgent` 和后面的训练循环。

下格是冻结源码的完整 `build_agent` 定义，来源 `src/satellite_rl/agents/workbench_algorithms.py:1189-1190`，提交 `48c7c4feac71`；运行时逐字节核对。


In [11]:
%%project_module C_BUILD
def build_agent(config, state_dim, action_dims, device="cpu"):
    return WorkbenchAgent(config, state_dim, action_dims, device)


已定义 C_BUILD: src/satellite_rl/agents/workbench_algorithms.py:1189-1190


### 完整断点保存

输入是 agent、进度、冻结配置及环境状态，输出是可恢复的断点文件。它保存的不只是网络权重，还包括继续训练需要的状态。

下格是冻结源码的完整 `save_checkpoint` 定义，来源 `src/satellite_rl/workbench/checkpoints.py:264-278`，提交 `48c7c4feac71`；运行时逐字节核对。


In [12]:
%%project_module C_SAVE
def save_checkpoint(path, agent, progress, config, environment_state):
    """Full trusted local resume point; loaded only from the job's own directory."""
    rng = {"python": random.getstate(), "numpy": np.random.get_state(),
           "torch": torch.get_rng_state(),
           "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    from satellite_rl.envs.regions import profile_metadata
    _atomic_torch(path, {"format": "satellite_rl_workbench_checkpoint_v1", "config_hash": config_hash(config),
                         "sac_policy_structure": _sac_structures(config)[0],
                         "sac_critic_structure": _sac_structures(config)[1],
                         "experiment_modes": _experiment_modes(config),
                         "evaluation_extra_steps": config["evaluation"].get("extra_steps", []),
                         "environment_profile": profile_metadata(config["environment"].get("profile", "china_legacy")),
                         "agent": agent.state_dict(), "progress": progress,
                         "environment_state": environment_state, "rng": rng})
    return str(path)


已定义 C_SAVE: src/satellite_rl/workbench/checkpoints.py:264-278


### 完整断点恢复

恢复时核对配置身份，再把模型、优化器、回放和进度读回来。只拿最终 `policy.pt` 无法等价续训，因此这里必须看完整断点。

下格是冻结源码的完整 `load_checkpoint` 定义，来源 `src/satellite_rl/workbench/checkpoints.py:281-309`，提交 `48c7c4feac71`；运行时逐字节核对。


In [13]:
%%project_module C_LOAD
def load_checkpoint(path, agent, config):
    path = Path(path).resolve()
    if path.name != "checkpoint.pt":
        raise ValueError("Resume only accepts this job's checkpoint.pt")
    # Full optimizer/replay/RNG state requires Python objects. This path must be local and trusted.
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if payload.get("format") != "satellite_rl_workbench_checkpoint_v1" or payload.get("config_hash") != config_hash(config):
        raise ValueError("Checkpoint format/config mismatch")
    if (payload.get("sac_policy_structure", "factorized"),
            payload.get("sac_critic_structure", "additive")) != _sac_structures(config):
        raise ValueError("Checkpoint SAC action structures mismatch")
    if payload.get("experiment_modes", _experiment_modes(config)) != _experiment_modes(config):
        raise ValueError("Checkpoint temperature, temporal or graph modes mismatch")
    if payload.get("evaluation_extra_steps", []) != config["evaluation"].get("extra_steps", []):
        raise ValueError("Checkpoint evaluation steps mismatch")
    from satellite_rl.envs.regions import profile_metadata
    expected_profile = profile_metadata(config["environment"].get("profile", "china_legacy"))
    if expected_profile["id"] != "china_legacy" and "environment_profile" not in payload:
        raise ValueError("Checkpoint environment profile metadata missing")
    if payload.get("environment_profile", expected_profile) != expected_profile:
        raise ValueError("Checkpoint environment profile metadata mismatch")
    agent.load_state_dict(payload["agent"])
    rng = payload["rng"]
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"])
    if rng["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng["cuda"])
    return payload["progress"], payload["environment_state"]


已定义 C_LOAD: src/satellite_rl/workbench/checkpoints.py:281-309


### 冻结场景验证

固定验证用另一组场景和当前冻结权重，不更新网络。训练器把节点结果与断点绑定，不能事后从不同权重拼接验证成绩。

下格是冻结源码的完整 `_fixed_evaluate` 定义，来源 `src/satellite_rl/workbench/runner.py:583-707`，提交 `48c7c4feac71`；运行时逐字节核对。


In [14]:
%%project_module C_VALID
def _fixed_evaluate(config, agent, directory, group, count, seed_base, env_step, ledger,
                    scene_set=None, zero_cache=None):
    from satellite_rl.workbench.environment import WorkbenchEnv
    files = None
    if scene_set is not None:
        from satellite_rl.workbench.datasets import assert_scene_profile, scene_set_files
        assert_scene_profile(scene_set, config["environment"].get("profile", "china_legacy"))
        files = scene_set_files(scene_set)
        if len(files) != count:
            raise ValueError("冻结评价场景数量与配置不一致")

    results = []
    p19_policy_sha = (_policy_parameters_sha256(agent)
                      if config.get("experiment", {}).get("id") == "S2-P19" and group == "validation"
                      else None)
    if zero_cache is not None and (group != "validation" or env_step != 0 or
                                   config["training"].get("beam_count_mode", "none") == "none" or
                                   config["evaluation"]["repeats"] != 1):
        raise ValueError("零步案例缓存仅支持可变束数单动作验证")
    for index in range(count):
        seed = seed_base + index
        scene_path = Path(directory) / "scenes" / group / f"scene_{seed}.npz"
        cache_contract = ({"checkpoint_sha256": zero_cache["checkpoint_sha256"],
                           "config_hash": zero_cache["config_hash"],
                           "scene_index": index, "source_scene_sha256": files[index][1],
                           "action_seed": seed, "group": "validation", "env_step": 0}
                          if zero_cache is not None else None)
        cache_path = (Path(directory) / "validation_zero_cases" / f"scene_{index:05d}.json"
                      if zero_cache is not None else None)
        cached = _zero_validation_case(cache_path, cache_contract) if cache_path else None
        if cached is not None:
            result = cached
            scene_ref = result["scene"]
            for trace in result["trace"]:
                _jsonl(Path(directory) / "traces.jsonl", trace)
            _jsonl(Path(directory) / "evaluations.jsonl", {"env_step": 0, "group": group,
                "scene_id": result["scene_id"], "action_seed": seed,
                "metrics": result["metrics"], "beam_count": result["beam_count"],
                "scene_metadata": result["scene_metadata"],
                "inference_seconds": result["inference_seconds"],
                "initial_preparation_seconds": result["initial_preparation_seconds"],
                "initial_verification_seconds": result["initial_verification_seconds"],
                "initial_allocation_seconds": result["initial_allocation_seconds"],
                "beams_final": result["beams"], "service_timing": result["service_timing"],
                "scene_sha256": scene_ref["sha256"]})
            results.append(result)
            continue
        with isolated_rng(seed):
            env = WorkbenchEnv(config)
            env.reset(seed=seed, scene=_load_scene(files[index][0]) if files is not None else None)
            scene = env.scene()
            scene_ref = _save_scene(scene_path, scene)
            if files is not None:
                scene_ref.update(source_path=str(files[index][0]), source_sha256=files[index][1],
                                 source_scene_id=scene_set["id"])
        for repeat in range(config["evaluation"]["repeats"]):
            action_seed = seed + repeat * 1000000
            p19_case = (Path(directory) / "validation_cases" / f"step_{env_step:09d}" /
                        f"scene_{index:05d}_r{repeat}.json"
                        if config.get("experiment", {}).get("id") == "S2-P19" and group == "validation"
                        else None)
            p19_contract = ({"config_hash": config_hash(config), "env_step": env_step,
                             "scene_index": index, "repeat_index": repeat,
                             "source_scene_sha256": files[index][1], "action_seed": action_seed,
                             "policy_sha256": p19_policy_sha}
                            if p19_case is not None else None)
            result = (_s2p19_validation_case(p19_case, p19_contract)
                      if p19_case is not None else None)
            if result is None:
                with isolated_rng(action_seed):
                    agent_rng = agent.rng.getstate()
                    agent.rng.seed(action_seed + 7321)
                    try:
                        env = WorkbenchEnv(config)
                        result = _evaluation_episode(env, agent, scene, f"{group}_{seed}_r{repeat}",
                                                     action_seed, config["evaluation"]["action_mode"],
                                                     None if zero_cache is not None or p19_case is not None
                                                     else Path(directory) / "traces.jsonl",
                                                     env_step, scene_sha256=scene_ref["sha256"], group=group,
                                                     scene_metadata=(scene_set["files"][index] if files is not None else None))
                    finally:
                        agent.rng.setstate(agent_rng)
            result["scene"] = scene_ref
            result["action_seed"] = action_seed
            result.setdefault("beam_count", len(scene["rate"]))
            result.setdefault("scene_metadata", scene_set["files"][index] if files is not None else None)
            result.setdefault("inference_seconds", None)
            result.setdefault("initial_preparation_seconds", None)
            result.setdefault("initial_verification_seconds", None)
            result.setdefault("initial_allocation_seconds", None)
            if zero_cache is not None:
                result = _zero_validation_case(cache_path, cache_contract, result)
                for trace in result["trace"]:
                    _jsonl(Path(directory) / "traces.jsonl", trace)
            if p19_case is not None:
                result = _s2p19_validation_case(p19_case, p19_contract, result)
                for trace in result["trace"]:
                    _jsonl(Path(directory) / "traces.jsonl", trace)
            _jsonl(Path(directory) / "evaluations.jsonl", {"env_step": env_step, "group": group,
                                                            "scene_id": result["scene_id"],
                                                            "action_seed": action_seed,
                                                            "metrics": result["metrics"],
                                                            "beam_count": result["beam_count"],
                                                            "scene_metadata": result["scene_metadata"],
                                                            "inference_seconds": result["inference_seconds"],
                                                            "initial_preparation_seconds": result["initial_preparation_seconds"],
                                                            "initial_verification_seconds": result["initial_verification_seconds"],
                                                            "initial_allocation_seconds": result["initial_allocation_seconds"],
                                                            "beams_final": result["beams"],
                                                            "service_timing": result["service_timing"],
                                                            "scene_sha256": scene_ref["sha256"]})
            results.append(result)
    summary = _summarize_scenes(results)
    ledger.record(group, env_step, summary)
    if any(row.get("scene_metadata", {}).get("beam_count") for row in results if row.get("scene_metadata")):
        by_count = {}
        for row in results:
            n = row["beam_count"]
            by_count.setdefault(n, []).append(row)
        _jsonl(Path(directory) / "count_metrics.jsonl", {
            "group": group, "env_step": env_step,
            "by_beam_count": {str(n): {"scene_actions": len(rows),
                                       "metrics": _summarize_scenes(rows)}
                              for n, rows in sorted(by_count.items())}})
    return results, summary


已定义 C_VALID: src/satellite_rl/workbench/runner.py:583-707


### 逐束训练主循环和检查点

先看 `build_agent` 和场景顺序，再沿主循环找 `env.step`、`agent.observe`、`agent.update`。达到节点时保存完整断点并做固定验证，10 万步试点还有科学闸门；这格定义函数，Run All 不调用它。

下格是冻结源码的完整 `_training` 定义，来源 `src/satellite_rl/workbench/runner.py:728-1223`，提交 `48c7c4feac71`；运行时逐字节核对。


In [15]:
%%project_module C_TRAIN
def _training(directory, job, config, ledger, resume):
    from satellite_rl.workbench.config import config_summary
    from satellite_rl.workbench.environment import WorkbenchEnv

    training = config["training"]
    summary = config_summary(config)
    seed_training(training["seed"])
    agent = build_agent(config, summary["state_dim"], summary["action_dims"], "cuda")
    checkpoint = Path(directory) / "checkpoint.pt"
    if resume:
        if not checkpoint.exists():
            raise RuntimeError("No recoverable checkpoint")
        progress, restored_environment = load_checkpoint(checkpoint, agent, config)
        restored_sets = restored_environment.get("scene_sets") if restored_environment else None
        if restored_sets is not None and restored_sets != job.get("scene_sets", {}):
            raise ValueError("恢复任务的冻结场景与断点不一致")
    else:
        progress = {"env_step": 0, "episode": 0, "best_validation": None,
                    "last_validation_step": -1, "last_checkpoint_step": 0,
                    "wall_seconds": 0.0}
    env_step = progress["env_step"]
    episode = progress["episode"]
    pilot_stop = (training["pilot_stop_steps"] if config["experiment"].get("id") == "S2-P19"
                  else None)
    if pilot_stop is not None and resume and pilot_stop <= env_step < training["total_steps"]:
        if not s2p19_pilot_gate_passed(directory, job["run_id"], checkpoint, pilot_stop):
            ledger.event("s2p19_pilot_gate_pending", env_step=env_step)
            ledger.save_status(stage="stopped", env_step=env_step,
                               stop_reason="s2p19_pilot_gate_pending",
                               checkpoint_step=env_step)
            return
        ledger.status.pop("stop_reason", None)
    session_started = time.monotonic()
    session_base_steps = env_step
    session_base_updates = agent.update_step
    base_wall_seconds = float(progress.get("wall_seconds", 0.0))

    def save_progress(stage=None, **fields):
        elapsed = max(time.monotonic() - session_started, 1e-9)
        step_rate = max(0, env_step - session_base_steps) / elapsed
        update_rate = max(0, agent.update_step - session_base_updates) / elapsed
        current_stage = stage or ("warmup" if config["algorithm"] != "ppo" and
                                  env_step < training["warmup_steps"] else "training")
        if training.get("beam_count_mode", "none") != "none":
            fields.setdefault("beam_count_steps", dict(count_steps))
            fields.setdefault("beam_count_episodes", dict(count_episodes))
            fields.setdefault("beam_bucket_steps", dict(bucket_steps))
            fields.setdefault("beam_bucket_episodes", dict(bucket_episodes))
            fields.setdefault("replay_bucket_sizes", {key: len(value) for key, value in
                                                      agent.replay.bucket_indices.items()})
            fields.setdefault("replay_sampled_by_bucket", dict(agent.replay.sampled_by_bucket))
            if agent.replay.segment_indices is not None:
                # S2-P18 手段E：每桶、每进度段的配额与有放回抽样次数都必须可回读。
                fields.setdefault("progress_strata", agent.replay.progress_strata)
                fields.setdefault("replay_sampled_by_stratum", dict(agent.replay.sampled_by_stratum))
                fields.setdefault("replay_replacement_draws_by_stratum",
                                  dict(agent.replay.replacement_draws_by_stratum))
                fields.setdefault("replay_stratum_sizes", {
                    key: [len(cell) for cell in bucket]
                    for key, bucket in agent.replay.segment_indices.items()})
        ledger.save_status(stage=current_stage, env_step=env_step, episode=episode,
                           update_step=agent.update_step, budget_env_steps=training["total_steps"],
                           steps_per_second=step_rate, updates_per_second=update_rate,
                           eta_seconds=((training["total_steps"] - env_step) / step_rate
                                        if step_rate > 0 else None),
                           wall_seconds=base_wall_seconds + elapsed,
                           gpu=ledger.status["preflight"]["gpu"],
                           gpu_memory_mb=torch.cuda.memory_allocated() / 1048576,
                           **fields)

    model_dir = Path(job["model_dir"])
    model_dir.mkdir(parents=True, exist_ok=True)
    scene_sets = job.get("scene_sets", {})
    if scene_sets:
        from satellite_rl.workbench.datasets import assert_scene_profile, scene_set_files
        for record in scene_sets.values():
            assert_scene_profile(record, config["environment"].get("profile", "china_legacy"))
            scene_set_files(record)
    supplied_scenes = (scene_set_files(scene_sets["train"]) if "train" in scene_sets else
                       [(path, _sha256(path)) for path in _scene_files(job["training_scene_path"])]
                       if job.get("training_scene_path") else [])
    variable_mode = training.get("beam_count_mode", "none")
    if variable_mode != "none":
        if "train" not in scene_sets or training.get("scene_order") != "variable_beams":
            raise ValueError("Variable beam training requires a frozen train set and variable_beams order")
        scene_order = VariableBeamSceneOrder(
            variable_mode, training.get("scene_order_seed", 51001),
            [digest for _, digest in supplied_scenes], scene_sets["train"]["files"])
        if config["experiment"].get("id") == "S2-P19" and config["experiment"].get("arm") == "C1":
            demonstration = training["demonstrations"]
            if demonstration["source_set_sha256"] != scene_order.source_set_sha256:
                raise ValueError("C1示范来源与冻结训练场景集SHA不一致")
            source_hashes = {digest for _, digest in supplied_scenes}
            teacher_hashes = {row["scene_sha256"] for row in agent.demonstrations.records}
            if not teacher_hashes.issubset(source_hashes):
                raise ValueError("C1示范含不在冻结训练集中的场景")
            if len(agent.demonstrations.records) != 576 or len(teacher_hashes) != 576:
                raise ValueError("C1示范须为576个不同训练母场景各一条完整轨迹")
            source_metadata = {digest: metadata for (_, digest), metadata in zip(
                supplied_scenes, scene_sets["train"]["files"])}
            mothers = [source_metadata[digest].get("mother_scene_id") for digest in teacher_hashes]
            if None in mothers or len(set(mothers)) != 576:
                raise ValueError("C1示范的576条轨迹未覆盖576个不同训练母场景")
            _verify_c1_demonstration_provenance(job, config, scene_sets["train"], scene_order, agent)
    else:
        scene_order = SceneOrder(training.get("scene_order", "sequential"),
                                 training.get("scene_order_seed", 51001),
                                 [digest for _, digest in supplied_scenes])
    if resume:
        scene_order.verify_checkpoint((restored_environment or {}).get("scene_order"), episode)
    partial_episode = (restored_environment or {}).get("partial_episode") if resume else None
    if partial_episode is not None and variable_mode == "none":
        raise ValueError("Legacy training cannot resume a variable beam partial episode")
    order_provenance = {"mode": scene_order.mode, "seed": scene_order.seed,
                        "source_count": len(supplied_scenes),
                        "source_set_sha256": scene_order.source_set_sha256}
    if variable_mode != "none":
        order_provenance["beam_count_mode"] = variable_mode
    bucket_steps = dict(progress.get("bucket_steps", {}))
    bucket_episodes = dict(progress.get("bucket_episodes", {}))
    count_steps = dict(progress.get("beam_count_steps", {}))
    count_episodes = dict(progress.get("beam_count_episodes", {}))
    provenance = {**_job_provenance(job, config), "training_scene_order": order_provenance}
    source_manifest = Path(job["source_root"]) / "source_manifest.json"
    snapshot_provenance = {**provenance, "source_root": job["source_root"],
                           "source_manifest_sha256": _sha256(source_manifest) if source_manifest.is_file() else None,
                           "training_scene_path": job.get("training_scene_path")}

    def snapshot_artifact(step):
        path = save_learning_snapshot(directory, agent, config, snapshot_provenance, step,
                                      source_history=job.get("source_history"))
        if path is None:
            return None
        recovery_records = sorted(str(item) for item in Path(path).parent.glob(
            f"{Path(path).stem}_recovery_*.json"))
        return {"directory": str(Path(directory) / "learning_snapshots"), "latest": path,
                "format": LEARNING_SNAPSHOT_FORMAT,
                "source_recovery_records": recovery_records,
                "description": "SAC网络参数及最多64条终止、64条非终止转移；用于离线学习诊断，不可用于恢复训练"}

    def run_validation():
        save_progress("validating")
        ledger.flush()
        zero_cache = ({"checkpoint_sha256": _sha256(checkpoint),
                       "config_hash": config_hash(config)}
                      if variable_mode != "none" and env_step == 0 else None)
        _, validation = _fixed_evaluate(config, agent, directory, "validation",
                                         config["evaluation"]["validation_count"],
                                         config["evaluation"]["validation_seed"], env_step, ledger,
                                         scene_set=scene_sets.get("validation"),
                                         zero_cache=zero_cache)
        progress["last_validation_step"] = env_step
        selection = validation.get("final_si_mean")
        if selection is not None and (progress["best_validation"] is None or selection > progress["best_validation"]):
            progress["best_validation"] = selection
            export_policy(model_dir / "best_validation", agent, config,
                          {**provenance,
                           "env_step": env_step, "selection": "validation_si_mean"})
        save_progress()

    if resume:
        snapshot_record = snapshot_artifact(env_step)
        _restore_logging(ledger, directory, env_step, progress.get("last_validation_step", -1))
        if snapshot_record:
            _artifacts(directory, job, learning_snapshots=snapshot_record)
    save_progress(scene_order=scene_order.checkpoint_state(episode))
    ledger.event("training_started" if not resume else "training_resumed", env_step=env_step,
                 config_hash=config_hash(config), training_scene_order=order_provenance)
    last_saved_step = progress.get("last_checkpoint_step", env_step)
    if not resume:
        if variable_mode != "none" and env_step == 0:
            # A recoverable initial state must exist before the 144-scene node.
            save_checkpoint(checkpoint, agent, progress, config,
                            {**provenance, "boundary": "next_episode",
                             "scene_order": scene_order.checkpoint_state(episode)})
            ledger.status["checkpoint_step"] = 0
        if _validation_due(config["evaluation"], progress, env_step):
            run_validation()
        save_checkpoint(checkpoint, agent, progress, config,
                        {**provenance, "boundary": "next_episode",
                         "scene_order": scene_order.checkpoint_state(episode)})
        ledger.status["checkpoint_step"] = env_step
        snapshot_record = snapshot_artifact(env_step)
        _artifacts(directory, job, checkpoint=str(checkpoint),
                   **({"learning_snapshots": snapshot_record} if snapshot_record else {}))
    elif variable_mode != "none" and env_step == 0 and \
            _validation_due(config["evaluation"], progress, env_step):
        # The loaded checkpoint is the pre-validation state; finish node zero
        # before entering the training loop, then commit its completed marker.
        run_validation()
        save_checkpoint(checkpoint, agent, progress, config,
                        {**provenance, "boundary": "next_episode",
                         "scene_order": scene_order.checkpoint_state(episode)})
        ledger.status["checkpoint_step"] = 0
        snapshot_record = snapshot_artifact(0)
        _artifacts(directory, job, checkpoint=str(checkpoint),
                   **({"learning_snapshots": snapshot_record} if snapshot_record else {}))
    while env_step < training["total_steps"]:
        scene_seed = training["seed"] + 100000 + episode
        selection = scene_order.selection(episode)
        replay_bucket = selection.get("bucket") if variable_mode != "none" else None
        source_index = selection["source_index"]
        source_scene = supplied_scenes[source_index][0] if source_index is not None else None
        if source_scene is not None and _sha256(source_scene) != supplied_scenes[source_index][1]:
            raise ValueError("训练场景文件SHA-256与冻结场景来源不一致")
        if partial_episode is None:
            env = WorkbenchEnv(config)
            with isolated_rng(scene_seed):
                state = env.reset(seed=scene_seed, scene=_load_scene(source_scene) if source_scene else None)
                scene = env.scene()
            if variable_mode != "none" and len(scene["rate"]) != selection["beam_count"]:
                raise ValueError("Frozen scene actual beam count differs from training metadata")
            # 逐束实体编码档位：观测宽度随 N 变化，权重按束数上界建立，必须与
            # 配置的 beam_count_max 一致（否则策略输入会在训练中途改变形状）。
            if config["network"].get("context_beam_count", "fixed_100") == "actual":
                from satellite_rl.workbench.config import expected_state_dim
                if state.shape[0] != expected_state_dim(config):
                    raise ValueError(
                        "逐束实体编码要求场景束数等于配置的 beam_count_max；"
                        f"当前场景 {len(scene['rate'])} 束、配置上限 "
                        f"{config['environment']['beam_count_max']}")
            reset_history = getattr(agent, "reset_episode", None)
            if reset_history is not None:
                reset_history()
            scene_ref = _save_scene(Path(directory) / "scenes" / "train" / f"scene_{episode:07d}.npz", scene)
            episode_return, length, episode_terms = 0.0, 0, {}
            episode_start = time.monotonic()
        else:
            if (partial_episode.get("episode") != episode or
                    partial_episode.get("selection") != selection or
                    partial_episode.get("scene_sha256") != supplied_scenes[source_index][1]):
                raise ValueError("Partial episode source differs from frozen checkpoint")
            env = partial_episode["environment"]
            if not isinstance(env, WorkbenchEnv):
                raise ValueError("Partial episode environment type mismatch")
            state = partial_episode["state"]
            scene_ref = partial_episode["scene_ref"]
            if _sha256(scene_ref["path"]) != scene_ref["sha256"]:
                raise ValueError("Partial episode archived scene differs from checkpoint")
            episode_return = partial_episode["return"]
            length = partial_episode["length"]
            episode_terms = partial_episode["reward_terms"]
            episode_start = time.monotonic() - partial_episode["elapsed_seconds"]
            partial_episode = None
        # S2-P18 手段D：按本回合实际束数解析折扣，并写进环境供奖励整形使用。
        beam_count_for_gamma = (selection.get("beam_count")
                                or (len(scene["rate"]) if "rate" in scene else None)
                                or getattr(getattr(env, "raw_env", None), "beam_num", None))
        gamma_lookup = getattr(agent, "gamma_for", None)
        effective_gamma = (gamma_lookup(beam_count_for_gamma) if gamma_lookup is not None
                           else float(training["gamma"]))
        env._effective_gamma = effective_gamma
        done = False
        while not done and env_step < training["total_steps"]:
            mask = env.action_mask()
            progress_segment = None
            replay = getattr(agent, "replay", None)
            if getattr(replay, "segment_indices", None) is not None:
                # 回合内进度分位：用掩码前可见的 beam_idx/beam_num 分段。
                progress_segment = min(replay.progress_strata - 1,
                                       int(env.raw_env.beam_idx / env.raw_env.beam_num
                                           * replay.progress_strata))
            policy_options = {"action_mask": mask} if mask is not None else {}
            if getattr(agent, "variable_width", False):
                # 逐束实体档位：观测按束数上限补零，必须把真实束数传给编码器，
                # 否则补零行会参与注意力池化与 N 条件调制。
                if beam_count_for_gamma is None:
                    raise RuntimeError("逐束实体档位缺少真实束数")
                policy_options["beam_entity_count"] = int(beam_count_for_gamma)
            action = agent.act(state, env_step=env_step, **policy_options)
            next_state, reward, done, info = env.step(action)
            if not np.isfinite(reward) or not np.isfinite(next_state).all():
                raise FloatingPointError("Nonfinite environment transition; last checkpoint preserved")
            replay_options = ({"action_mask": mask,
                               "next_action_mask": None if done else env.action_mask()}
                              if mask is not None else {})
            if getattr(agent, "temporal_enabled", False):
                replay_options["actual_action"] = info["actual_action"]
            if replay_bucket is not None:
                replay_options["replay_bucket"] = replay_bucket
            if progress_segment is not None:
                replay_options["progress_segment"] = progress_segment
            if getattr(agent, "variable_width", False):
                # 逐束实体档位：真实束数随转移写入回放（自然布局观测无法还原束数）。
                replay_options["entity_count"] = int(beam_count_for_gamma)
            agent.observe(state, action, reward, next_state, done, **replay_options)
            env_step += 1
            if replay_bucket is not None:
                bucket_steps[replay_bucket] = bucket_steps.get(replay_bucket, 0) + 1
                count_key = str(selection["beam_count"])
                count_steps[count_key] = count_steps.get(count_key, 0) + 1
            length += 1
            episode_return += float(reward)
            for name, values in info.get("reward_terms", {}).items():
                if isinstance(values, dict):
                    for component in ("raw", "scaled", "weighted"):
                        key = f"{name}/{component}"
                        episode_terms[key] = episode_terms.get(key, 0.0) + float(values.get(component, 0.0))
            if agent.ready(env_step, done):
                updates = training["gradient_steps"] if config["algorithm"] != "ppo" else 1
                for _ in range(updates):
                    diagnostics = agent.update()
                    _ensure_finite_agent(agent)
                    _jsonl(Path(directory) / "updates.jsonl", {"env_step": env_step,
                                                                "update_step": agent.update_step,
                                                                "values": diagnostics})
                    if agent.update_step % training["log_interval"] == 0:
                        ledger.record("learning", env_step, diagnostics)
            if variable_mode != "none" and _validation_due(config["evaluation"], progress, env_step):
                run_validation()
            if env_step % training["log_interval"] == 0:
                elapsed = max(time.monotonic() - session_started, 1e-6)
                ledger.record("runtime", env_step, {"env_steps_per_second": (env_step - session_base_steps) / elapsed,
                                                     "updates_per_second": (agent.update_step - session_base_updates) / elapsed,
                                                     "wall_seconds": base_wall_seconds + elapsed,
                                                     "replay_size": len(agent.replay) if hasattr(agent, "replay") else len(agent.rollout),
                                                     **({"replay_bucket_sizes": {key: len(value) for key, value in agent.replay.bucket_indices.items()},
                                                         "replay_sampled_by_bucket": dict(agent.replay.sampled_by_bucket),
                                                         "replay_replacement_draws": dict(agent.replay.replacement_draws)}
                                                        if variable_mode != "none" else {}),
                                                     "cuda_allocated_mb": torch.cuda.memory_allocated() / 1048576,
                                                     "cuda_reserved_mb": torch.cuda.memory_reserved() / 1048576,
                                                     **{k: v for k, v in agent.last_action_info.items() if isinstance(v, (int, float))}})
                requested = info.get("requested_action") or {}
                actual = info.get("actual_action") or {}
                ledger.record("action", env_step, {
                    **{f"requested_{key}": requested.get(key) for key in ("freq", "slots", "power_w")},
                    **{f"actual_{key}": actual.get(key) for key in ("freq", "slots", "power_w", "charged_power_w")},
                    "clipped_slots": (requested.get("slots", 0) - actual.get("slots", 0)),
                })
                save_progress()
            _jsonl(Path(directory) / "traces.jsonl", {"env_step": env_step, "group": "training",
                                                          "episode": episode,
                                                          "step": length - 1, "scene_sha256": scene_ref["sha256"],
                                                          "beam_count": selection.get("beam_count"),
                                                          "replay_bucket": replay_bucket,
                                                          "mother_scene_id": selection.get("mother_scene_id"),
                                                          "load_multiplier": selection.get("load_multiplier"),
                                                          "beam_id": info.get("beam_id"),
                                                          "beam_service_step": info.get("beam_service_step"),
                                                          "service_timing": "after_action_before_later_interference",
                                                          "requested_action": info.get("requested_action", list(action)),
                                                          "actual_action": info.get("actual_action"),
                                                          "reward": reward, "reward_terms": info.get("reward_terms"),
                                                          "done": done})
            state = next_state
            if (variable_mode != "none" and not done and
                    env_step < training["total_steps"] and
                    env_step % training["checkpoint_interval"] == 0):
                _ensure_finite_agent(agent)
                ledger.flush()
                progress.update({"env_step": env_step, "episode": episode,
                                 "last_checkpoint_step": env_step,
                                 "wall_seconds": base_wall_seconds + time.monotonic() - session_started,
                                 "bucket_steps": bucket_steps,
                                 "beam_count_steps": count_steps})
                partial = {"episode": episode, "selection": selection,
                           "scene_sha256": supplied_scenes[source_index][1],
                           "environment": env, "state": state,
                           "scene_ref": scene_ref, "return": episode_return,
                           "length": length, "reward_terms": episode_terms,
                           "elapsed_seconds": time.monotonic() - episode_start}
                save_checkpoint(checkpoint, agent, progress, config,
                                {**provenance, "boundary": "within_episode",
                                 "scene_order": scene_order.checkpoint_state(episode),
                                 "partial_episode": partial})
                ledger.status["checkpoint_step"] = env_step
                last_saved_step = env_step
                snapshot_record = snapshot_artifact(env_step)
                _artifacts(directory, job, checkpoint=str(checkpoint),
                           **({"learning_snapshots": snapshot_record} if snapshot_record else {}))
                save_progress(checkpoint_step=env_step)
                ledger.flush()
                if pilot_stop is not None and env_step == pilot_stop:
                    export_policy(model_dir / "latest", agent, config,
                                  {**provenance, "env_step": env_step,
                                   "selection": "s2p19_pilot_gate_pending"})
                    _artifacts(directory, job, checkpoint=str(checkpoint),
                               latest_policy=str(model_dir / "latest"))
                    ledger.event("s2p19_pilot_gate_pending", env_step=env_step,
                                 checkpoint_sha256=_sha256(checkpoint))
                    save_progress("stopped", stop_reason="s2p19_pilot_gate_pending",
                                  checkpoint_step=env_step)
                    ledger.flush()
                    return
        if config["algorithm"] == "ppo" and agent.rollout:
            diagnostics = agent.update()
            _ensure_finite_agent(agent)
            _jsonl(Path(directory) / "updates.jsonl", {"env_step": env_step,
                                                        "update_step": agent.update_step, "values": diagnostics})
            ledger.record("learning", env_step, diagnostics)
        final_beams = env.beam_results() if hasattr(env, "beam_results") else []
        terminal_metrics = env.metrics(beams=final_beams) if final_beams else env.metrics()
        episode_row = {"env_step": env_step, "episode": episode, "length": length,
                       "return": episode_return, "duration_seconds": time.monotonic() - episode_start,
                       "scene_seed": scene_seed, "scene": scene_ref, "terminal": bool(done),
                       "episode_end_reason": (terminal_metrics.get("termination_reason", "complete")
                                              if done else "training_budget_exhausted"),
                       "source_scene": ({"path": str(source_scene), "sha256": supplied_scenes[source_index][1],
                                         "index": source_index,
                                         "scene_id": scene_sets["train"]["files"][source_index].get("scene_id")
                                         if "train" in scene_sets else source_scene.stem,
                                         "set_id": scene_sets["train"]["id"] if "train" in scene_sets else None}
                                        if source_scene else None),
                       "scene_order": selection,
                       "beam_count": len(final_beams), "replay_bucket": replay_bucket,
                       "scene_metadata": (scene_sets["train"]["files"][source_index]
                                          if variable_mode != "none" else None),
                       "metrics": terminal_metrics,
                       "beams_final": final_beams,
                       "service_timing": "after_last_recorded_action_including_later_interference",
                       "reward_terms": episode_terms}
        _jsonl(Path(directory) / "episodes.jsonl", episode_row)
        ledger.record("training", env_step, {"return": episode_return, "episode_length": length,
                                             **{k: v for k, v in terminal_metrics.items() if isinstance(v, (int, float))}})
        ledger.record("reward", env_step, episode_terms)
        episode += 1
        if replay_bucket is not None:
            bucket_episodes[replay_bucket] = bucket_episodes.get(replay_bucket, 0) + 1
            count_key = str(selection["beam_count"])
            count_episodes[count_key] = count_episodes.get(count_key, 0) + 1
            progress.update(bucket_steps=bucket_steps, bucket_episodes=bucket_episodes,
                            beam_count_steps=count_steps, beam_count_episodes=count_episodes)
        save_progress(scene_order=scene_order.checkpoint_state(episode))
        if variable_mode == "none" and _validation_due(config["evaluation"], progress, env_step):
            run_validation()
        stopped = (Path(directory) / "stop.request").exists()
        if stopped or env_step - last_saved_step >= training["checkpoint_interval"] or env_step >= training["total_steps"]:
            _ensure_finite_agent(agent)
            ledger.flush()
            progress.update({"env_step": env_step, "episode": episode, "last_checkpoint_step": env_step,
                             "wall_seconds": base_wall_seconds + time.monotonic() - session_started})
            save_checkpoint(checkpoint, agent, progress, config,
                            {**provenance, "boundary": "next_episode",
                             "scene_order": scene_order.checkpoint_state(episode)})
            ledger.status["checkpoint_step"] = env_step
            snapshot_record = snapshot_artifact(env_step)
            last_saved_step = env_step
            export_policy(model_dir / "latest", agent, config,
                          {**provenance, "env_step": env_step})
            _artifacts(directory, job, checkpoint=str(checkpoint), latest_policy=str(model_dir / "latest"),
                       best_validation_policy=str(model_dir / "best_validation") if (model_dir / "best_validation").exists() else None,
                       **({"learning_snapshots": snapshot_record} if snapshot_record else {}))
            save_progress(checkpoint_step=env_step, best_validation=progress["best_validation"])
            ledger.flush()
        if pilot_stop is not None and env_step == pilot_stop:
            ledger.event("s2p19_pilot_gate_pending", env_step=env_step,
                         checkpoint_sha256=_sha256(checkpoint))
            save_progress("stopped", stop_reason="s2p19_pilot_gate_pending",
                          checkpoint_step=env_step)
            ledger.flush()
            return
        if stopped:
            ledger.event("training_stopped", env_step=env_step)
            save_progress("stopped")
            return
    # The approved V0/V1 assessment is a separate frozen 440-scene job. A
    # default workbench test_count would otherwise add unapproved allocations.
    if variable_mode != "none":
        export_policy(model_dir / "final", agent, config,
                      {**provenance, "env_step": env_step,
                       "selection": "fixed_budget_final"})
        result_path = Path(directory) / "results.json"
        _atomic_json(result_path, {"summary": None, "scenes": [],
                                   "metadata": {**provenance, "group": "training_endpoint",
                                                "run_id": job["run_id"],
                                                "git_commit": job["git_commit"],
                                                "config_hash": config_hash(config),
                                                "model_selection": "fixed_budget_final",
                                                "independent_assessment_pending": True}})
        _artifacts(directory, job, final_policy=str(model_dir / "final"), results=str(result_path))
        ledger.event("training_completed_assessment_pending", env_step=env_step)
        ledger.flush()
        save_progress("completed", metrics=None, independent_assessment_pending=True)
        return
    # Final test is never used to select a model.
    save_progress("evaluating")
    test_rows, test_summary = _fixed_evaluate(config, agent, directory, "test",
                                               config["evaluation"]["test_count"],
                                               config["evaluation"]["test_seed"], env_step, ledger,
                                               scene_set=scene_sets.get("test"))
    export_policy(model_dir / "final", agent, config,
                  {**provenance, "env_step": env_step})
    _atomic_json(Path(directory) / "results.json", {"summary": test_summary, "scenes": test_rows,
                                                    "metadata": {**provenance, "group": "test", "run_id": job["run_id"],
                                                                 "git_commit": job["git_commit"],
                                                                 "config_hash": config_hash(config),
                                                                 "action_mode": config["evaluation"]["action_mode"],
                                                                 "repeats": config["evaluation"]["repeats"],
                                                                 "actual_scene_count": config["evaluation"]["test_count"],
                                                                 "repeated_actions_are_not_independent_scenes": True,
                                                                 "model_selection": "fixed_budget_final"}})
    _artifacts(directory, job, final_policy=str(model_dir / "final"), results=str(Path(directory) / "results.json"))
    ledger.event("training_completed", env_step=env_step)
    ledger.flush()
    save_progress("completed", metrics=test_summary)


已定义 C_TRAIN: src/satellite_rl/workbench/runner.py:728-1223


## I0：贪心初解上的多步改进 SAC

I0 先由非学习规则给出完整布局，再让 SAC 从最多 16 个候选修改里选一个。每步可移动、调边界、交换、改窗口、回退或 STOP；每次候选都用完整物理评价。环境保护已保存的最佳布局，只有满足约束且改进的候选才替换输出。奖励为 SI 增量，`gamma=1`，因此一回合奖励可望远镜式累加到终态改进。


### I0 参数约束

输入是准备使用的 I0 配置，输出是通过约束检查的配置。它固定 `gamma=1`、熵系数和候选/探测预算，避免奖励含义在运行中悄悄改变。

下格是冻结源码的完整 `validate_config` 定义，来源 `src/satellite_rl/sac_improvement/config.py:45-85`，提交 `62e33c1baa06`；运行时逐字节核对。


In [16]:
%%project_module I_CONFIG
def validate_config(raw):
    if not isinstance(raw, dict):
        raise ValueError("I0 config must be an object")
    extra = set(raw) - set(DEFAULT_CONFIG)
    if extra:
        raise ValueError(f"Unknown I0 controls: {sorted(extra)}")
    config = default_config()
    config.update(deepcopy(raw))
    if config["schema_version"] != DEFAULT_CONFIG["schema_version"]:
        raise ValueError("Unknown I0 schema")
    if config["profile"] != "europe_v1" or config["power_budget_w"] != 3000.0 or config["basic_fraction"] != 0.1:
        raise ValueError("I0 requires the frozen European physical domain")
    for key in ("seed", "max_decisions", "max_probes", "candidate_limit", "batch_size",
                "replay_capacity", "warmup_steps", "update_every", "gradient_steps",
                "hidden_width", "total_steps", "pilot_stop_steps", "pilot_max_episodes",
                "max_episodes", "checkpoint_interval"):
        if type(config[key]) is not int or config[key] < 0:
            raise ValueError(f"Invalid {key}")
    if config["max_decisions"] != 32 or config["max_probes"] != 32 or config["candidate_limit"] != 16:
        raise ValueError("I0 decision/probe/candidate limits are frozen at 32/32/16")
    if config["gamma"] != 1.0 or config["alpha_mode"] != "fixed":
        raise ValueError("I0 reward telescoping requires gamma=1 and frozen fixed-alpha SAC")
    if (config["total_steps"] != 400000 or config["pilot_stop_steps"] != 100000
            or config["pilot_max_episodes"] != 3125 or config["max_episodes"] != 12500):
        raise ValueError("I0 requires a frozen 100k/3125 pilot inside 400k/12500 total")
    if config["device"] not in ("cpu", "cuda"):
        raise ValueError("Invalid I0 device")
    for key in ("si_tolerance", "sgm_tolerance", "fixed_alpha", "target_entropy_ratio",
                "actor_lr", "critic_lr", "tau", "gradient_clip"):
        value = config[key]
        if type(value) not in (int, float) or not math.isfinite(value) or value < 0:
            raise ValueError(f"Invalid {key}")
    if config["si_tolerance"] != 1e-8 or config["sgm_tolerance"] != 1e-8:
        raise ValueError("I0 numerical guards are frozen")
    if not 0 < config["fixed_alpha"] or not 0 <= config["target_entropy_ratio"] <= 1:
        raise ValueError("Invalid SAC entropy settings")
    if not 0 < config["tau"] <= 1 or min(config["actor_lr"], config["critic_lr"]) <= 0:
        raise ValueError("Invalid optimizer settings")
    if config["batch_size"] < 1 or config["replay_capacity"] < config["batch_size"]:
        raise ValueError("Invalid replay capacity")
    return config


### 候选动作记录

一个候选保留动作名称、修改位置以及改后的频率起点和槽数。环境据此重新算完整布局，而不是只估计局部变化。

下格是冻结源码的完整 `Candidate` 定义，来源 `src/satellite_rl/sac_improvement/environment.py:35-41`，提交 `62e33c1baa06`；运行时逐字节核对。


In [17]:
%%project_module I_CANDIDATE
@dataclass(frozen=True)
class Candidate:
    id: str
    kind: str
    changes: tuple
    starts: tuple
    slots: tuple


### 候选生成、完整物理、保护输出和奖励

从 `reset` 看贪心初解，再看 `candidates` 如何产生最多 16 个修改和 STOP。`state`/`action_feature` 交给 SAC，`step` 用完整物理评价并给 SI 增量奖励；`output` 只交出通过保护条件的历史最佳布局。

下格是冻结源码的完整 `ImprovementEnv` 定义，来源 `src/satellite_rl/sac_improvement/environment.py:44-477`，提交 `62e33c1baa06`；运行时逐字节核对。


In [18]:
%%project_module I_ENV
class ImprovementEnv:
    """Each non-STOP decision tests exactly one complete candidate allocation.

    An invalid request consumes one decision and zero physical probes. A repeat
    consumes a decision and a real recomputation/probe. STOP consumes one
    decision and zero probes. At decision 32 or probe 32 the
    episode ends. Only a candidate passing all three baseline guards can replace
    the saved output; exploratory current layouts may be worse.
    """

    def __init__(self, config):
        self.config = validate_config(config)
        self.scene = None
        self.initial_beams = None
        self.current_starts = None
        self.current_slots = None
        self.current_eval = None
        self.best_starts = None
        self.best_slots = None
        self.best_eval = None
        self.base_eval = None
        self.power = None
        self.demand = None
        self.base_basic = None
        self.decision_count = 0
        self.probe_count = 0
        self.invalid_count = 0
        self.repeat_count = 0
        self.proposals_generated = 0
        self.done = False
        self.visited = set()
        self.trace = []
        self._candidate_cache = None
        self.initialization_seconds = 0.0
        self.candidate_generation_seconds = 0.0
        self.probe_seconds = 0.0

    def reset(self, scene, seed=None):
        """Generate B1 by executing the existing workbench baseline, not a copy."""
        started = time.perf_counter()
        wc = workbench_default_config()
        wc["environment"].update(profile=self.config["profile"], beam_count_max=200,
                                 power_budget_w=self.config["power_budget_w"],
                                 basic_fraction=self.config["basic_fraction"],
                                 action_mode="legacy_clip", action_mask="joint_feasible")
        env = WorkbenchEnv(wc)
        state = env.reset(scene=scene)
        policy = BaselinePolicy("greedy", env)
        complete = False
        while not complete:
            choice = policy.act(state, deterministic=True, action_mask=env.action_mask())
            state, _, complete, _ = env.step(choice)
        metrics = env.metrics()
        if not metrics["complete"] or metrics["overload"] or metrics["clipped_count"]:
            raise RuntimeError("B1 did not produce a complete, unprojected feasible allocation")
        self.scene = env.scene()
        self.initial_beams = env.beam_results()
        n = len(self.initial_beams)
        self.demand = np.asarray(self.scene["rate"], dtype=float).reshape(n)
        self.power = np.asarray([row["power_w"] for row in self.initial_beams], dtype=float)
        self.current_starts = np.asarray([row["freq"] for row in self.initial_beams], dtype=int)
        self.current_slots = np.asarray([row["slots"] for row in self.initial_beams], dtype=int)
        self.current_eval = self._evaluate(self.current_starts, self.current_slots)
        actual = np.asarray([row["rate_bps"] for row in self.initial_beams])
        if not np.allclose(self.current_eval.capacity_bps, actual, rtol=1e-8, atol=1e-3):
            raise RuntimeError("I0 physics did not reproduce the actual greedy terminal capacities")
        self.base_eval = self.current_eval
        self.base_basic = self.base_eval.capacity_bps >= self.config["basic_fraction"] * self.demand
        self.best_starts = self.current_starts.copy()
        self.best_slots = self.current_slots.copy()
        self.best_eval = self.current_eval
        self.decision_count = self.probe_count = self.invalid_count = self.repeat_count = 0
        self.proposals_generated = 0
        self.done = False
        self.visited = {_layout_key(self.current_starts, self.current_slots)}
        self.trace = []
        self._candidate_cache = None
        self.initialization_seconds = time.perf_counter() - started
        self.candidate_generation_seconds = 0.0
        self.probe_seconds = 0.0
        return self.state()

    def _rows(self, starts, slots):
        return [{**row, "freq": int(starts[i]), "slots": int(slots[i])}
                for i, row in enumerate(self.initial_beams)]

    def _evaluate(self, starts, slots):
        physics = FixedAllocationPhysics.from_scene(
            self.scene, {"environment": {"profile": self.config["profile"],
                                         "power_budget_w": self.config["power_budget_w"]}},
            self._rows(starts, slots))
        return physics.evaluate(self.power)

    def _legal(self, starts, slots):
        if (len(starts) != len(self.demand) or len(slots) != len(self.demand)
                or np.any(slots < 1) or np.any(slots > 10)
                or np.any(starts < 0) or np.any(starts + slots > 100)
                or self.power.sum() > self.config["power_budget_w"] + 1e-6):
            return False
        for group in range(8):
            occupied = np.zeros(100, dtype=bool)
            for i in range(group, len(slots), 8):
                block = slice(int(starts[i]), int(starts[i] + slots[i]))
                if occupied[block].any():
                    return False
                occupied[block] = True
        return True

    def _protected(self, evaluation):
        return (evaluation.si + self.config["si_tolerance"] >= self.base_eval.si
                and evaluation.sgm + self.config["sgm_tolerance"] >= self.base_eval.sgm
                and bool(np.all(evaluation.capacity_bps[self.base_basic]
                                >= self.config["basic_fraction"] * self.demand[self.base_basic])))

    def _candidate(self, kind, changes):
        starts, slots = self.current_starts.copy(), self.current_slots.copy()
        for i, start, length in changes:
            starts[i], slots[i] = int(start), int(length)
        if not self._legal(starts, slots):
            return None
        key = _layout_key(starts, slots)
        if key == _layout_key(self.current_starts, self.current_slots):
            return None
        digest = _sha_layout(starts, slots)
        return Candidate(digest, kind, tuple((int(i), int(f), int(k)) for i, f, k in changes),
                         tuple(map(int, starts)), tuple(map(int, slots)))

    def candidates(self):
        if self.scene is None or self.done:
            return []
        if self._candidate_cache is not None:
            return self._candidate_cache
        started = time.perf_counter()
        n = len(self.demand)
        priority = sorted(range(n), key=lambda i: (float(self.current_eval.beam_si[i]),
                                                   -float(self.demand[i]), i))
        by_kind = {kind: [] for kind in KIND_NAMES}
        seen = set()

        def add(kind, changes):
            self.proposals_generated += 1
            candidate = self._candidate(kind, changes)
            if candidate is not None and candidate.id not in seen:
                seen.add(candidate.id)
                by_kind[kind].append(candidate)

        for i in priority[:min(n, 32)]:
            start, count = int(self.current_starts[i]), int(self.current_slots[i])
            for delta in (-5, -1, 1, 5):
                add("move", ((i, start + delta, count),))
            if len(by_kind["move"]) >= 8:
                break

        # Adjacent boundaries transfer one slot without leaving a gap.
        for group in range(8):
            ordered = sorted(range(group, n, 8), key=lambda i: int(self.current_starts[i]))
            for left, right in zip(ordered, ordered[1:]):
                a, b = int(self.current_starts[left]), int(self.current_starts[right])
                x, y = int(self.current_slots[left]), int(self.current_slots[right])
                if a + x == b:
                    add("boundary", ((left, a, x + 1), (right, b + 1, y - 1)))
                    add("boundary", ((left, a, x - 1), (right, b - 1, y + 1)))
            if len(by_kind["boundary"]) >= 8:
                break

        # Swap occupied intervals as atomic assignments: works even at 800/800 slots.
        for i in priority[:min(n, 20)]:
            peers = sorted((j for j in range(i % 8, n, 8) if j != i),
                           key=lambda j: (abs(int(self.current_starts[j] - self.current_starts[i])), j))
            for j in peers[:4]:
                add("swap", ((i, int(self.current_starts[j]), int(self.current_slots[j])),
                             (j, int(self.current_starts[i]), int(self.current_slots[i]))))
            if len(by_kind["swap"]) >= 8:
                break

        for group in range(8):
            ordered = sorted(range(group, n, 8), key=lambda i: int(self.current_starts[i]))
            for width in (3, 4):
                for offset in range(len(ordered) - width + 1):
                    window = ordered[offset:offset + width]
                    rotated = window[1:] + window[:1]
                    add("window", tuple((i, int(self.current_starts[j]), int(self.current_slots[j]))
                                        for i, j in zip(window, rotated)))
                    if len(by_kind["window"]) >= 8:
                        break
                if len(by_kind["window"]) >= 8:
                    break
            if len(by_kind["window"]) >= 8:
                break

        if _layout_key(self.best_starts, self.best_slots) != _layout_key(
                self.current_starts, self.current_slots):
            add("revert", tuple((i, int(self.best_starts[i]), int(self.best_slots[i]))
                                for i in range(n) if self.best_starts[i] != self.current_starts[i]
                                or self.best_slots[i] != self.current_slots[i]))

        limit = self.config["candidate_limit"]
        selected = []
        # Round-robin prevents move-only pools from suppressing full-slot swaps.
        while len(selected) < limit and any(by_kind.values()):
            for kind in KIND_NAMES:
                if by_kind[kind] and len(selected) < limit:
                    selected.append(by_kind[kind].pop(0))
        self._candidate_cache = selected
        self.candidate_generation_seconds += time.perf_counter() - started
        return selected

    def state(self):
        if self.scene is None:
            raise RuntimeError("reset before reading I0 state")
        n = len(self.demand)
        result = np.zeros((MAX_BEAMS, 14), dtype=np.float32)
        locations = np.asarray(self.scene["beam_location"], dtype=float)
        result[:n, 0] = 1
        result[:n, 1] = locations[:, 0] / 90
        result[:n, 2] = locations[:, 1] / 180
        result[:n, 3] = self.demand / 1e9
        result[:n, 4] = np.arange(n) % 8 / 7
        result[:n, 5] = self.current_starts / 100
        result[:n, 6] = self.current_slots / 10
        result[:n, 7] = self.power / 50
        result[:n, 8] = self.current_eval.capacity_bps / self.demand
        result[:n, 9] = self.best_starts / 100
        result[:n, 10] = self.best_slots / 10
        result[:n, 11] = self.best_eval.capacity_bps / self.demand
        result[:n, 12] = self.base_basic.astype(float)
        result[:n, 13] = np.arange(n) / max(n - 1, 1)
        global_part = np.asarray([n / 200, self.decision_count / 32, self.probe_count / 32,
                                  (32 - self.decision_count) / 32, (32 - self.probe_count) / 32,
                                  self.current_eval.si, self.current_eval.sgm,
                                  self.best_eval.si, self.best_eval.sgm,
                                  self.base_eval.si, self.base_eval.sgm,
                                  self.power.sum() / self.config["power_budget_w"]],
                                 dtype=np.float32)
        return np.concatenate((result.reshape(-1), global_part))

    def action_feature(self, candidate):
        result = np.zeros(ACTION_WIDTH, dtype=np.float32)
        if candidate is None:  # STOP
            result[-1] = 1
            return result
        for offset, (i, start, count) in enumerate(candidate.changes[:4]):
            at = 5 * offset
            result[at:at + 5] = (i / 200, start / 100, count / 10,
                                  (start - self.current_starts[i]) / 100,
                                  (count - self.current_slots[i]) / 10)
        result[20 + KIND_NAMES.index(candidate.kind)] = 1
        result[-2] = min(len(candidate.changes), 4) / 4
        return result

    def step(self, candidate_id):
        if self.done:
            raise RuntimeError("I0 episode already complete")
        candidates = {candidate.id: candidate for candidate in self.candidates()}
        candidate = candidates.get(candidate_id)
        before_starts, before_slots = self.current_starts.copy(), self.current_slots.copy()
        requested_changes = ([{"beam_id": int(i), "group": int(i % 8),
                               "from_freq": int(before_starts[i]), "to_freq": int(start),
                               "from_slots": int(before_slots[i]), "to_slots": int(length),
                               "power_w": float(self.power[i])}
                              for i, start, length in candidate.changes]
                             if candidate is not None else [] if candidate_id == "STOP" else None)
        requested_action = {"candidate_id": candidate_id,
                            "kind": candidate.kind if candidate is not None else
                                    "STOP" if candidate_id == "STOP" else "unknown",
                            "changes": requested_changes}
        self.decision_count += 1
        reward = 0.0
        reason = "candidate"
        actual_changes = []
        changed_beam_service = []
        if candidate_id == "STOP":
            reason = "stop"
            self.done = True
        elif candidate is None:
            reason = "invalid_or_stale"
            self.invalid_count += 1
        else:
            starts = np.asarray(candidate.starts, dtype=int)
            slots = np.asarray(candidate.slots, dtype=int)
            key = _layout_key(starts, slots)
            if key in self.visited:
                reason = "repeat"
                self.repeat_count += 1
            started = time.perf_counter()
            evaluated = self._evaluate(starts, slots)
            self.probe_seconds += time.perf_counter() - started
            self.probe_count += 1
            self.visited.add(key)
            self.current_starts, self.current_slots, self.current_eval = starts, slots, evaluated
            actual_changes = list(requested_changes)
            for change in actual_changes:
                i = change["beam_id"]
                changed_beam_service.append({
                    "beam_id": i, "group": change["group"],
                    **beam_service_fields(float(evaluated.capacity_bps[i]),
                                          float(self.demand[i]),
                                          float(evaluated.beam_sgm[i])),
                    "service_timing": "after_this_macro_action_and_full_interference_recalculation",
                    "source": "FixedAllocationPhysics.from_scene.evaluate",
                    "units": {"required_bps": "bit/s", "capacity_bps": "bit/s",
                              "effective_served_bps": "bit/s", "capacity_demand_ratio": "ratio",
                              "si": "ratio", "sgm": "ratio"}})
            if self._protected(evaluated) and evaluated.si > self.best_eval.si + 1e-12:
                reward = 100.0 * (evaluated.si - self.best_eval.si)
                self.best_starts, self.best_slots, self.best_eval = starts.copy(), slots.copy(), evaluated
            self._candidate_cache = None
        if self.decision_count >= self.config["max_decisions"] or self.probe_count >= self.config["max_probes"]:
            self.done = True
            if reason == "candidate":
                reason = "decision_or_probe_limit"
        info = {"reason": reason, "candidate_id": candidate_id, "decisions": self.decision_count,
                "probes": self.probe_count, "current_si": self.current_eval.si,
                "best_si": self.best_eval.si, "best_sgm": self.best_eval.sgm,
                "protected_output": self._protected(self.best_eval),
                "requested_action": requested_action,
                "actual_action": {"accepted": candidate is not None and candidate_id != "STOP",
                                  "changes": actual_changes, "rejection_reason":
                                  reason if candidate is None and candidate_id != "STOP" else None},
                "changed_beam_service": changed_beam_service,
                "service_scope": "changed_beams_only_after_full_all_beam_physics_recalculation",
                "all_beams_recomputed": bool(actual_changes),
                "service_units": {"demand": "bit/s", "capacity": "bit/s",
                                  "effective_served": "bit/s", "rho": "ratio",
                                  "si": "ratio", "sgm": "ratio"}}
        self.trace.append(info)
        return self.state(), reward, self.done, info

    def _beam_rows(self, starts, slots, evaluation):
        beams = []
        for i, old in enumerate(self.initial_beams):
            capacity = float(evaluation.capacity_bps[i])
            matching = float(evaluation.beam_sgm[i])
            row = dict(old, freq=int(starts[i]), slots=int(slots[i]),
                       **beam_service_fields(capacity, float(self.demand[i]), matching))
            row["basic_met"] = bool(capacity >= self.config["basic_fraction"] * self.demand[i])
            beams.append(row)
        return beams

    def terminal_snapshot(self):
        """Return both real last explored layout and independently saved best."""
        if self.scene is None:
            raise RuntimeError("I0 has no complete initial allocation")
        current = self._beam_rows(self.current_starts, self.current_slots, self.current_eval)
        best = self._beam_rows(self.best_starts, self.best_slots, self.best_eval)
        def view(label, beams, evaluation, starts, slots):
            return {"state_role": label, "beams": beams,
                    "layout_sha256": _sha_layout(starts, slots),
                    "metrics": {"si": float(evaluation.si), "sgm": float(evaluation.sgm),
                                "basic_met": sum(row["basic_met"] for row in beams),
                                "power_w": float(self.power.sum()),
                                "slots_total": int(slots.sum())}}
        return {"last_exploration_state": view("last_exploration_state", current,
                                               self.current_eval, self.current_starts,
                                               self.current_slots),
                "selected_best_output": view("selected_best_output", best,
                                             self.best_eval, self.best_starts,
                                             self.best_slots),
                "beam_result_source": "FixedAllocationPhysics.from_scene.evaluate",
                "service_timing": "after_last_recorded_macro_action_including_all_interference",
                "units": {"required_bps": "bit/s", "capacity_bps": "bit/s",
                          "effective_served_bps": "bit/s", "capacity_demand_ratio": "ratio",
                          "si": "ratio", "sgm": "ratio", "power_w": "W"}}

    def output(self):
        if self.scene is None:
            raise RuntimeError("I0 has no complete allocation")
        beams = self._beam_rows(self.best_starts, self.best_slots, self.best_eval)
        return {"beams": beams, "metrics": {"si": self.best_eval.si, "sgm": self.best_eval.sgm,
                                            "initial_si": self.base_eval.si,
                                            "initial_sgm": self.base_eval.sgm,
                                            "basic_met": sum(row["basic_met"] for row in beams),
                                            "power_w": float(self.power.sum()),
                                            "slots_total": int(self.best_slots.sum()),
                                            "decisions": self.decision_count,
                                            "probes": self.probe_count,
                                            "invalid": self.invalid_count,
                                            "repeats": self.repeat_count,
                                            "initialization_seconds": self.initialization_seconds,
                                            "candidate_generation_seconds": self.candidate_generation_seconds,
                                            "probe_seconds": self.probe_seconds,
                                            "proposals_generated": self.proposals_generated},
                "trace": list(self.trace), "layout_sha256": _sha_layout(self.best_starts, self.best_slots)}

    def checkpoint_state(self):
        if self.scene is None:
            raise RuntimeError("Cannot save an uninitialized I0 episode")
        return {"version": 1, "initial_layout_sha256": _sha_layout(
                    [row["freq"] for row in self.initial_beams],
                    [row["slots"] for row in self.initial_beams]),
                "current_starts": self.current_starts.copy(), "current_slots": self.current_slots.copy(),
                "best_starts": self.best_starts.copy(), "best_slots": self.best_slots.copy(),
                "decision_count": self.decision_count, "probe_count": self.probe_count,
                "invalid_count": self.invalid_count, "repeat_count": self.repeat_count,
                "proposals_generated": self.proposals_generated,
                "initialization_seconds": self.initialization_seconds,
                "candidate_generation_seconds": self.candidate_generation_seconds,
                "probe_seconds": self.probe_seconds,
                "visited": list(self.visited), "trace": list(self.trace), "done": self.done}

    def load_checkpoint_state(self, saved):
        """Call after reset on the exact frozen scene; recomputation is audited."""
        if saved["version"] != 1 or saved["initial_layout_sha256"] != _sha_layout(
                [row["freq"] for row in self.initial_beams],
                [row["slots"] for row in self.initial_beams]):
            raise ValueError("I0 resume initial allocation differs")
        current_starts = np.asarray(saved["current_starts"], dtype=int)
        current_slots = np.asarray(saved["current_slots"], dtype=int)
        best_starts = np.asarray(saved["best_starts"], dtype=int)
        best_slots = np.asarray(saved["best_slots"], dtype=int)
        if not self._legal(current_starts, current_slots) or not self._legal(best_starts, best_slots):
            raise ValueError("I0 resume contains an illegal layout")
        current_eval = self._evaluate(current_starts, current_slots)
        best_eval = self._evaluate(best_starts, best_slots)
        if not self._protected(best_eval) or best_eval.si + 1e-12 < self.base_eval.si:
            raise ValueError("I0 resume best output violates baseline guards")
        self.current_starts, self.current_slots, self.current_eval = current_starts, current_slots, current_eval
        self.best_starts, self.best_slots, self.best_eval = best_starts, best_slots, best_eval
        self.decision_count = int(saved["decision_count"])
        self.probe_count = int(saved["probe_count"])
        self.invalid_count = int(saved["invalid_count"])
        self.repeat_count = int(saved["repeat_count"])
        self.proposals_generated = int(saved["proposals_generated"])
        self.initialization_seconds = float(saved["initialization_seconds"])
        self.candidate_generation_seconds = float(saved["candidate_generation_seconds"])
        self.probe_seconds = float(saved["probe_seconds"])
        self.visited = {tuple((int(f), int(k)) for f, k in row) for row in saved["visited"]}
        self.trace = list(saved["trace"])
        self.done = bool(saved["done"])
        if (not 0 <= self.probe_count <= self.decision_count <= self.config["max_decisions"]
                or self.probe_count > self.config["max_probes"]
                or _layout_key(current_starts, current_slots) not in self.visited):
            raise ValueError("I0 resume resource ledger differs")
        self._candidate_cache = None


### 候选动作打分网络

输入是整体布局状态和每个候选的特征，输出每个候选的分数。actor 和两个 Q 都用这类网络，但参数互不共享。

下格是冻结源码的完整 `CandidateNetwork` 定义，来源 `src/satellite_rl/sac_improvement/agent.py:21-34`，提交 `62e33c1baa06`；运行时逐字节核对。


In [19]:
%%project_module I_NET
class CandidateNetwork(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.state = nn.Sequential(nn.Linear(STATE_WIDTH, width), nn.LayerNorm(width), nn.ReLU(),
                                   nn.Linear(width, width), nn.ReLU())
        self.action = nn.Sequential(nn.Linear(ACTION_WIDTH, width // 2), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(width + width // 2, width), nn.ReLU(), nn.Linear(width, 1))

    def forward(self, state, features):
        if state.ndim != 2 or features.ndim != 3 or features.shape[0] != state.shape[0]:
            raise ValueError("Candidate network input dimensions differ")
        common = self.state(state).unsqueeze(1).expand(-1, features.shape[1], -1)
        encoded = self.action(features)
        return self.head(torch.cat((common, encoded), dim=-1)).squeeze(-1)


### 动作候选打包

把可变数量的候选补齐到固定张量并生成掩码。更新时掩码阻止网络把未生成的候选当作合法动作。

下格是冻结源码的完整 `pack_actions` 定义，来源 `src/satellite_rl/sac_improvement/agent.py:37-53`，提交 `62e33c1baa06`；运行时逐字节核对。


In [20]:
%%project_module I_PACK
def pack_actions(environment):
    """Return identity-bound features and validity; STOP is always last."""
    choices = environment.candidates()
    if len(choices) > MAX_ACTIONS - 1:
        raise ValueError("I0 candidate generator exceeded the frozen action limit")
    features = np.zeros((MAX_ACTIONS, ACTION_WIDTH), dtype=np.float32)
    mask = np.zeros(MAX_ACTIONS, dtype=bool)
    identifiers = []
    for index, choice in enumerate(choices):
        features[index] = environment.action_feature(choice)
        mask[index] = True
        identifiers.append(choice.id)
    features[-1] = environment.action_feature(None)
    mask[-1] = True
    identifiers.extend([None] * (MAX_ACTIONS - 1 - len(choices)))
    identifiers.append("STOP")
    return features, mask, tuple(identifiers)


### I0 回放

每条记录存当前候选、所选动作、奖励和下一状态候选。抽批后给两个 Q 与 actor 更新；随机采样器的状态也随断点保存。

下格是冻结源码的完整 `Replay` 定义，来源 `src/satellite_rl/sac_improvement/agent.py:66-104`，提交 `62e33c1baa06`；运行时逐字节核对。


In [21]:
%%project_module I_REPLAY
class Replay:
    def __init__(self, capacity, seed):
        self.rows = deque(maxlen=capacity)
        self.rng = random.Random(seed)

    def add(self, state, features, mask, identifiers, index, reward,
            next_state, next_features, next_mask, done):
        if (len(identifiers) != MAX_ACTIONS or not mask[index]
                or identifiers[index] is None or mask[-1] is not True and not bool(mask[-1])):
            raise ValueError("Invalid candidate identity or action")
        row = (np.asarray(state, np.float32).copy(), np.asarray(features, np.float32).copy(),
               np.asarray(mask, bool).copy(), tuple(identifiers), int(index), float(reward),
               np.asarray(next_state, np.float32).copy(), np.asarray(next_features, np.float32).copy(),
               np.asarray(next_mask, bool).copy(), bool(done))
        self.rows.append(row)

    def sample(self, count, device):
        rows = self.rng.sample(self.rows, count)
        def tensor(values, dtype):
            return torch.as_tensor(np.stack(values), dtype=dtype, device=device)
        return (tensor([r[0] for r in rows], torch.float32),
                tensor([r[1] for r in rows], torch.float32),
                tensor([r[2] for r in rows], torch.bool),
                torch.as_tensor([r[4] for r in rows], dtype=torch.long, device=device),
                torch.as_tensor([r[5] for r in rows], dtype=torch.float32, device=device),
                tensor([r[6] for r in rows], torch.float32),
                tensor([r[7] for r in rows], torch.float32),
                tensor([r[8] for r in rows], torch.bool),
                torch.as_tensor([r[9] for r in rows], dtype=torch.float32, device=device))

    def state_dict(self):
        return {"rows": list(self.rows), "rng_state": self.rng.getstate(),
                "capacity": self.rows.maxlen}

    def load_state_dict(self, state):
        if state["capacity"] != self.rows.maxlen:
            raise ValueError("Replay capacity changed across resume")
        self.rows = deque(state["rows"], maxlen=self.rows.maxlen)
        self.rng.setstate(state["rng_state"])


### 双 Q 和 actor 参数更新

先看 `choose` 怎样在掩码内选候选，再看 `update`：两个目标 Q 的较小值构成目标，两个在线 Q 拟合目标，actor 优先选价值高且保留熵的动作。最后看 `state_dict`，恢复时网络、优化器和回放都要对上。

下格是冻结源码的完整 `ImprovementSAC` 定义，来源 `src/satellite_rl/sac_improvement/agent.py:107-198`，提交 `62e33c1baa06`；运行时逐字节核对。


In [22]:
%%project_module I_AGENT
class ImprovementSAC:
    def __init__(self, config):
        self.config = config
        self.device = torch.device(config["device"])
        width = config["hidden_width"]
        self.actor = CandidateNetwork(width).to(self.device)
        self.critic1 = CandidateNetwork(width).to(self.device)
        self.critic2 = CandidateNetwork(width).to(self.device)
        self.target1 = copy.deepcopy(self.critic1).eval()
        self.target2 = copy.deepcopy(self.critic2).eval()
        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=config["actor_lr"])
        self.critic1_opt = torch.optim.Adam(self.critic1.parameters(), lr=config["critic_lr"])
        self.critic2_opt = torch.optim.Adam(self.critic2.parameters(), lr=config["critic_lr"])
        self.replay = Replay(config["replay_capacity"], config["seed"] + 931)
        self.updates = 0

    def choose(self, state, features, mask, deterministic=False):
        with torch.no_grad():
            state_t = torch.as_tensor(state[None], device=self.device, dtype=torch.float32)
            feat_t = torch.as_tensor(features[None], device=self.device, dtype=torch.float32)
            mask_t = torch.as_tensor(mask[None], device=self.device, dtype=torch.bool)
            logits = self.actor(state_t, feat_t)
            probability, _, _ = _distribution(logits, mask_t)
            if deterministic:
                return int(probability.argmax(-1).item())
            return int(torch.multinomial(probability[0], 1).item())

    def update(self):
        batch_size = self.config["batch_size"]
        if len(self.replay.rows) < batch_size:
            return None
        s, f, m, action, reward, ns, nf, nm, done = self.replay.sample(batch_size, self.device)
        alpha = self.config["fixed_alpha"]
        with torch.no_grad():
            next_p, next_logp, _ = _distribution(self.actor(ns, nf), nm)
            next_q = torch.minimum(self.target1(ns, nf), self.target2(ns, nf))
            next_value = (next_p * (next_q - alpha * next_logp)).sum(-1)
            target = reward + self.config["gamma"] * (1 - done) * next_value
        q1 = self.critic1(s, f).gather(1, action[:, None]).squeeze(1)
        q2 = self.critic2(s, f).gather(1, action[:, None]).squeeze(1)
        loss1, loss2 = F.mse_loss(q1, target), F.mse_loss(q2, target)
        for optimizer, loss, network in ((self.critic1_opt, loss1, self.critic1),
                                         (self.critic2_opt, loss2, self.critic2)):
            if not bool(torch.isfinite(loss)):
                raise RuntimeError("I0 critic loss is not finite")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), self.config["gradient_clip"])
            optimizer.step()
        p, logp, entropy = _distribution(self.actor(s, f), m)
        with torch.no_grad():
            q = torch.minimum(self.critic1(s, f), self.critic2(s, f))
        actor_loss = (p * (alpha * logp - q)).sum(-1).mean()
        if not bool(torch.isfinite(actor_loss)):
            raise RuntimeError("I0 actor loss is not finite")
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), self.config["gradient_clip"])
        self.actor_opt.step()
        tau = self.config["tau"]
        with torch.no_grad():
            for target_net, net in ((self.target1, self.critic1), (self.target2, self.critic2)):
                for target_param, source_param in zip(target_net.parameters(), net.parameters()):
                    target_param.lerp_(source_param, tau)
        self.updates += 1
        valid_count = m.sum(-1).float()
        target_entropy = self.config["target_entropy_ratio"] * valid_count.log()
        result = {"critic1_loss": float(loss1.item()), "critic2_loss": float(loss2.item()),
                  "actor_loss": float(actor_loss.item()), "alpha": float(alpha),
                  "entropy": float(entropy.mean().item()),
                  "target_entropy": float(target_entropy.mean().item()),
                  "q1": float(q1.mean().item()), "target_q": float(target.mean().item()),
                  "valid_actions": float(valid_count.mean().item()), "updates": self.updates}
        if not all(math.isfinite(v) for v in result.values()):
            raise RuntimeError("I0 update diagnostics are not finite")
        return result

    def state_dict(self):
        return {"actor": self.actor.state_dict(), "critic1": self.critic1.state_dict(),
                "critic2": self.critic2.state_dict(), "target1": self.target1.state_dict(),
                "target2": self.target2.state_dict(), "actor_opt": self.actor_opt.state_dict(),
                "critic1_opt": self.critic1_opt.state_dict(),
                "critic2_opt": self.critic2_opt.state_dict(),
                "replay": self.replay.state_dict(), "updates": self.updates}

    def load_state_dict(self, state):
        for name in ("actor", "critic1", "critic2", "target1", "target2"):
            getattr(self, name).load_state_dict(state[name], strict=True)
        for name in ("actor_opt", "critic1_opt", "critic2_opt"):
            getattr(self, name).load_state_dict(state[name])
        self.replay.load_state_dict(state["replay"])
        self.updates = int(state["updates"])


### 训练状态与环境断点

输入包括 agent、步数、回合、随机状态和当前环境，输出完整 `checkpoint.pt`。I0 可在回合中断点恢复，所以当前候选与已探测布局也要保存。

下格是冻结源码的完整 `_checkpoint` 定义，来源 `src/satellite_rl/sac_improvement/runner.py:147-160`，提交 `62e33c1baa06`；运行时逐字节核对。


In [23]:
%%project_module I_SAVE
def _checkpoint(directory, job, config, frozen_config_sha, agent, steps, episodes,
                scene_index, environment, pilot_gate_passed=False):
    state = {"version": 1, "run_id": job["run_id"],
             "frozen_config_sha256": frozen_config_sha,
             "i0_config": config, "env_step": steps, "episodes": episodes,
             "scene_index": scene_index,
             "pilot_gate_passed": bool(pilot_gate_passed),
             "environment": environment.checkpoint_state() if environment is not None else None,
             "agent": agent.state_dict(), "rng": _rng_state()}
    path = directory / "checkpoint.pt"
    temp = path.with_name(path.name + f".{os.getpid()}.tmp")
    torch.save(state, temp)
    os.replace(temp, path)
    return _sha(path)


### 断点读取及身份核对

读取时核 run、配置和源码身份；不匹配就拒绝恢复。这样旧试点的学习状态不会被接到不同实验上。

下格是冻结源码的完整 `_read_checkpoint` 定义，来源 `src/satellite_rl/sac_improvement/runner.py:163-172`，提交 `62e33c1baa06`；运行时逐字节核对。


In [24]:
%%project_module I_LOAD
def _read_checkpoint(directory, job, config, frozen_config_sha):
    path = directory / "checkpoint.pt"
    if not path.is_file():
        raise ValueError("I0 resume lacks complete checkpoint")
    state = torch.load(path, map_location="cpu", weights_only=False)
    if (state.get("version") != 1 or state.get("run_id") != job["run_id"]
            or state.get("frozen_config_sha256") != frozen_config_sha
            or state.get("i0_config") != config):
        raise ValueError("I0 checkpoint identity/config/source differs")
    return state, _sha(path)


### 固定节点验证

在冻结验证景上执行当前策略，保存逐景输出和对应断点 SHA。验证不做梯度更新，也不替代最终独立考核。

下格是冻结源码的完整 `_validate_node` 定义，来源 `src/satellite_rl/sac_improvement/runner.py:497-568`，提交 `62e33c1baa06`；运行时逐字节核对。


In [25]:
%%project_module I_VALID
def _validate_node(directory, job, config, agent, step, checkpoint_sha, validation):
    record, files = validation
    node_dir = directory / "validation" / f"step_{step:09d}"
    node_dir.mkdir(parents=True, exist_ok=True)
    summary_path = node_dir / "summary.json"
    previous = _read(summary_path)
    if previous is not None:
        if previous.get("checkpoint_sha256") != checkpoint_sha or previous.get("count") != len(files):
            raise ValueError("I0 fixed validation node provenance differs")
        return True
    saved_rng = _rng_state()
    actor_was_training = agent.actor.training
    agent.actor.eval()
    rows = []
    try:
        for index, (path, digest) in enumerate(files):
            case_id = f"step_{step:09d}_scene_{index:05d}"
            case_path = node_dir / f"scene_{index:05d}.json"
            prior = _read(case_path)
            if prior is not None:
                if (prior.get("scene_sha256") != digest or
                        prior.get("checkpoint_sha256") != checkpoint_sha):
                    raise ValueError("I0 validation case/checkpoint source differs")
                pending_path = directory / "case_attempt.json"
                pending = _read(pending_path)
                if pending is not None and pending.get("case_id") == case_id:
                    _jsonl(directory / "interrupted_attempts.jsonl", {
                        "case_id": case_id, "scene_sha256": digest,
                        "started_at": pending.get("started_at"), "detected_at": _now(),
                        "exact_elapsed_seconds": None,
                        "reason": "validation case saved before interruption; timing receipt missing"})
                    pending_path.unlink(missing_ok=True)
                rows.append(prior)
                continue
            if (directory / "stop.request").exists():
                return False
            seed = int(job.get("action_seed", 53001))
            _seed(seed)
            started = _attempt_start(directory, case_id, digest)
            env = _model_episode(agent, _load_scene(path), config, seed, deterministic=False)
            result = _episode_result(env, record["files"][index], digest, seed, "I0_validation")
            result.update(checkpoint_sha256=checkpoint_sha, env_step=step, repeat_index=0)
            result["metrics"]["total_seconds"] = env.total_seconds
            _atomic_json(case_path, result)
            _attempt_complete(directory, case_id, started)
            rows.append(result)
    finally:
        agent.actor.train(actor_was_training)
        _load_rng(saved_rng)
    summary = {"env_step": step, "checkpoint_sha256": checkpoint_sha,
               "source_set_id": record.get("id"), "count": len(rows),
               "mean_si": float(np.mean([r["metrics"]["si"] for r in rows])),
               "mean_sgm": float(np.mean([r["metrics"]["sgm"] for r in rows])),
               "mean_si_delta_vs_b1": float(np.mean([
                   r["metrics"]["si"] - r["metrics"]["initial_si"] for r in rows])),
               "cases": [{"path": str(node_dir / f"scene_{i:05d}.json"),
                          "sha256": _sha(node_dir / f"scene_{i:05d}.json")}
                         for i in range(len(rows))], "completed_at": _now()}
    _atomic_json(summary_path, summary)
    # Rebuild from immutable node summaries, avoiding duplicates after a crash.
    summaries = [_read(directory / "validation" / f"step_{node:09d}" / "summary.json")
                 for node in VALIDATION_NODES if
                 (directory / "validation" / f"step_{node:09d}" / "summary.json").is_file()]
    extra = [p for p in (directory / "validation").glob("step_*/summary.json")
             if int(p.parent.name.rsplit("_", 1)[-1]) not in VALIDATION_NODES]
    summaries.extend(_read(p) for p in sorted(extra))
    text = "".join(json.dumps(_jsonable(row), ensure_ascii=False, allow_nan=False) + "\n"
                   for row in sorted(summaries, key=lambda item: item["env_step"]))
    temp = directory / f"evaluations.jsonl.{os.getpid()}.tmp"
    temp.write_text(text, encoding="utf-8")
    os.replace(temp, directory / "evaluations.jsonl")
    return True


### 宏动作训练主循环

主循环先取 `env.state()` 与候选，再由 agent 选动作、调用 `env.step`、把转移放入回放并更新。节点触发断点和验证，回合或预算结束后保存最终权重；实际任务因 12500 回合上限停在 397112 宏步。

下格是冻结源码的完整 `_train_job` 定义，来源 `src/satellite_rl/sac_improvement/runner.py:652-871`，提交 `62e33c1baa06`；运行时逐字节核对。


In [26]:
%%project_module I_TRAIN
def _train_job(directory, job, config, resume, frozen_config_sha):
    record, files = _scene_set(job, "train")
    if len(files) != 8064:
        raise ValueError("I0 training requires the frozen 8064 variable-beam derivatives")
    validation = _scene_set(job, "validation")
    if len(validation[1]) != 72:
        raise ValueError("I0 fixed validation requires 72 frozen derivatives")
    order = VariableBeamSceneOrder("mixed_four_buckets", config["seed"],
                                   [digest for _, digest in files], record["files"])
    previous = _read(directory / "status.json", {})
    _seed(config["seed"])
    agent = ImprovementSAC(config)
    if resume:
        saved, digest = _read_checkpoint(directory, job, config, frozen_config_sha)
        gate_passed = _gate_on_resume(directory, job, previous, digest, saved, config)
        agent.load_state_dict(saved["agent"])
        steps, episodes = int(saved["env_step"]), int(saved["episodes"])
        scene_index = saved["scene_index"]
        current = saved["environment"]
        if scene_index is not None:
            selection = order.selection(episodes)
            if selection["source_index"] != scene_index:
                raise ValueError("I0 resume scene order differs")
            env = ImprovementEnv(config)
            env.reset(_load_scene(files[scene_index][0]))
            env.load_checkpoint_state(current)
        else:
            env = None
        _load_rng(saved["rng"])
    else:
        if (directory / "checkpoint.pt").exists():
            raise ValueError("I0 checkpoint exists; resume required")
        gate_passed = False
        steps = episodes = 0
        env = None
        scene_index = None
    _status(directory, job, config, "training", step=steps, episodes=episodes,
            pilot_gate_passed=gate_passed, updates=agent.updates)
    last_checkpoint_step = steps
    if steps == 0 and not (directory / "checkpoint.pt").is_file():
        digest = _checkpoint(directory, job, config, frozen_config_sha, agent,
                             steps, episodes, scene_index, env, pilot_gate_passed=gate_passed)
    else:
        digest = _sha(directory / "checkpoint.pt")
    _artifacts(directory, job, checkpoint=str(directory / "checkpoint.pt"),
               validation_directory=str(directory / "validation"))
    initial_terminal = steps >= config["total_steps"] or episodes >= config["max_episodes"]
    saved_nodes = _saved_validation_nodes(directory)
    if (len(saved_nodes) >= VALIDATION_NODE_BUDGET and
            not (initial_terminal and steps in saved_nodes)):
        raise ValueError("I0 direct validation budget was exhausted before the actual endpoint")
    initial_pilot = (not gate_passed and
                     (steps >= config["pilot_stop_steps"] or
                      episodes >= config["pilot_max_episodes"]))
    if steps in VALIDATION_NODES or initial_pilot or initial_terminal:
        if not _scheduled_validation(directory, job, config, agent, steps, digest,
                                     validation, terminal=initial_terminal,
                                     pilot_reached=initial_pilot):
            _status(directory, job, config, "stopped", step=steps, episodes=episodes,
                    stop_reason="user_stop_during_validation", checkpoint_step=steps,
                    checkpoint_sha256=digest, pilot_gate_passed=gate_passed)
            return
    if not gate_passed and (steps >= config["pilot_stop_steps"] or
                            episodes >= config["pilot_max_episodes"]):
        _status(directory, job, config, "stopped", step=steps, episodes=episodes,
                stop_reason="s2p19_pilot_gate_pending", checkpoint_step=steps,
                checkpoint_sha256=digest, pilot_gate_passed=False,
                updates=agent.updates)
        return
    while steps < config["total_steps"] and episodes < config["max_episodes"]:
        if env is None:
            scene_index = order.selection(episodes)["source_index"]
            env = ImprovementEnv(config)
            env.reset(_load_scene(files[scene_index][0]))
        state = env.state()
        features, mask, identifiers = pack_actions(env)
        index = agent.choose(state, features, mask, deterministic=False)
        chosen = identifiers[index]
        next_state, reward, done, info = env.step(chosen)
        next_features, next_mask, _ = pack_actions(env) if not done else (
            np.zeros_like(features), np.eye(1, MAX_ACTIONS, MAX_ACTIONS - 1, dtype=bool)[0], None)
        agent.replay.add(state, features, mask, identifiers, index, reward,
                         next_state, next_features, next_mask, done)
        steps += 1
        diagnostic = None
        if (steps >= config["warmup_steps"] and steps % config["update_every"] == 0):
            for _ in range(config["gradient_steps"]):
                diagnostic = agent.update()
        if diagnostic is not None and steps % 50 == 0:
            _jsonl(directory / "updates.jsonl", {"env_step": steps, **diagnostic})
        _jsonl(directory / "traces.jsonl", {"env_step": steps, "episode": episodes,
                                              "scene_index": scene_index, "action_index": index,
                                              "candidate_id": chosen, "reward": reward,
                                               "done": done, "episode_step": env.decision_count,
                                               "scene_sha256": files[scene_index][1],
                                               "mother_scene_id": record["files"][scene_index].get("mother_scene_id"),
                                               "selected_time_utc": record["files"][scene_index].get("selected_time_utc"),
                                               "beam_count": len(env.demand),
                                               "load_multiplier": record["files"][scene_index].get("load_multiplier"),
                                               "source": {"run_id": job["run_id"],
                                                          "git_commit": job.get("git_commit"),
                                                          "physics": "FixedAllocationPhysics.from_scene.evaluate"},
                                               **info})
        if done:
            output = env.output()
            terminal_reason = ("policy_stop" if info["reason"] == "stop" else
                               "decision_or_probe_limit" if env.decision_count >= config["max_decisions"]
                               or env.probe_count >= config["max_probes"] else info["reason"])
            _jsonl(directory / "episodes.jsonl", {"episode": episodes, "env_step": steps,
                                                   "scene_index": scene_index,
                                                   "scene_sha256": files[scene_index][1],
                                                    "mother_scene_id": record["files"][scene_index].get("mother_scene_id"),
                                                    "beam_count": len(env.demand),
                                                    "load_multiplier": record["files"][scene_index].get("load_multiplier"),
                                                   "metrics": output["metrics"],
                                                   "reward_total": 100 * (output["metrics"]["si"] -
                                                                           output["metrics"]["initial_si"]),
                                                    "terminal_snapshot": _training_beam_snapshot(
                                                        env, job, record["files"][scene_index],
                                                        files[scene_index][1], episodes, steps,
                                                        terminal_reason, True)})
            episodes += 1
            env = None
            scene_index = None
        pilot_reached = (not gate_passed and
                         (steps >= config["pilot_stop_steps"] or
                          episodes >= config["pilot_max_episodes"]))
        requested_stop = (directory / "stop.request").exists()
        checkpoint_due = _training_checkpoint_due(
            steps, last_checkpoint_step, episodes, config, pilot_reached, requested_stop)
        if checkpoint_due:
            digest = _checkpoint(directory, job, config, frozen_config_sha, agent, steps,
                                 episodes, scene_index, env, pilot_gate_passed=gate_passed)
            _artifacts(directory, job, checkpoint=str(directory / "checkpoint.pt"),
                       validation_directory=str(directory / "validation"))
            last_checkpoint_step = steps
            if env is not None and (pilot_reached or requested_stop or
                                    steps >= config["total_steps"]):
                pause_reason = ("s2p19_pilot_gate_pending" if pilot_reached else
                                "user_stop" if requested_stop else "training_budget_exhausted")
                partial = _partial_episode_row(
                    env, job, record["files"][scene_index], scene_index,
                    files[scene_index][1], episodes, steps, pause_reason)
                _jsonl(directory / ("episodes.jsonl" if
                                    steps >= config["total_steps"] and not requested_stop and not pilot_reached
                                    else "episode_snapshots.jsonl"), partial)
            terminal = steps >= config["total_steps"] or episodes >= config["max_episodes"]
            if steps in VALIDATION_NODES or pilot_reached or terminal:
                if not _scheduled_validation(directory, job, config, agent, steps, digest,
                                             validation, terminal=terminal,
                                             pilot_reached=pilot_reached):
                    if env is not None and not (pilot_reached or requested_stop or
                                                steps >= config["total_steps"]):
                        _jsonl(directory / "episode_snapshots.jsonl", _partial_episode_row(
                            env, job, record["files"][scene_index], scene_index,
                            files[scene_index][1], episodes, steps,
                            "user_stop_during_validation"))
                    _status(directory, job, config, "stopped", step=steps, episodes=episodes,
                            stop_reason="user_stop_during_validation", checkpoint_step=steps,
                            checkpoint_sha256=digest, pilot_gate_passed=gate_passed,
                            updates=agent.updates)
                    return
            if pilot_reached or requested_stop:
                reason = "s2p19_pilot_gate_pending" if pilot_reached else "user_stop"
                _status(directory, job, config, "stopped", step=steps, episodes=episodes,
                        stop_reason=reason, checkpoint_step=steps,
                        checkpoint_sha256=digest, pilot_gate_passed=gate_passed,
                        updates=agent.updates)
                return
            _status(directory, job, config, "training", step=steps, episodes=episodes,
                    checkpoint_step=steps, checkpoint_sha256=digest,
                    pilot_gate_passed=gate_passed, updates=agent.updates)
    # A failed terminal-log append may have left a valid final checkpoint but
    # no terminal row. Refuse to publish a model until that row is durable.
    if env is not None:
        expected = (episodes, steps, "training_budget_exhausted")
        last = _last_jsonl(directory / "episodes.jsonl") or {}
        observed = (last.get("episode"), last.get("env_step"),
                    (last.get("terminal_snapshot") or {}).get("terminal_reason"))
        if observed != expected:
            _jsonl(directory / "episodes.jsonl", _partial_episode_row(
                env, job, record["files"][scene_index], scene_index,
                files[scene_index][1], episodes, steps,
                "training_budget_exhausted"))
    # The fixed endpoint is the final actor, even if the episode cap stops early.
    model_root = Path(job.get("model_dir", directory / "models"))
    model_dir = model_root / "final"
    model_dir.mkdir(parents=True, exist_ok=True)
    model = {"run_id": job["run_id"], "i0_config": config, "actor": agent.actor.state_dict(),
             "env_step": steps, "episodes": episodes, "updates": agent.updates,
             "provenance": {"run_id": job["run_id"], "env_step": steps,
                            "episode": episodes, "algorithm": "s2p19_i0_candidate_sac"},
             "git_commit": job["git_commit"], "source_manifest_sha256":
             _sha(Path(job["source_root"]) / "source_manifest.json")}
    model_path = model_dir / "policy.pt"
    temp = model_dir / f"policy.pt.{os.getpid()}.tmp"
    torch.save(model, temp)
    os.replace(temp, model_path)
    frozen_model_config_path = directory / "config.json"
    if _sha(frozen_model_config_path) != frozen_config_sha:
        raise ValueError("I0 final model config changed after training checkpoint")
    frozen_model_config = _read(frozen_model_config_path)
    _atomic_json(model_dir / "model.json", {"run_id": job["run_id"], "env_step": steps,
                                              "episodes": episodes, "updates": agent.updates,
                                              "format": "satellite_rl_s2p19_improvement_v1",
                                              "algorithm": "sac", "state_dim": STATE_WIDTH,
                                              "parameter_count": sum(p.numel() for p in agent.actor.parameters()),
                                              "config": frozen_model_config,
                                              "policy_sha256": _sha(model_path),
                                              "config_sha256": frozen_config_sha,
                                              "git_commit": job["git_commit"],
                                              "provenance": model["provenance"],
                                              "policy_algorithm": "s2p19_i0_candidate_sac"})
    _artifacts(directory, job, final_policy=str(model_dir),
               checkpoint=str(directory / "checkpoint.pt"),
               validation_directory=str(directory / "validation"))
    _status(directory, job, config, "completed", step=steps, episodes=episodes,
            completed_due_to="env_steps" if steps == config["total_steps"] else "episode_cap",
            model_path=str(model_path), model_sha256=_sha(model_path),
            checkpoint_sha256=_sha(directory / "checkpoint.pt"), updates=agent.updates)


## 只读核对与容易误读的地方

下面只打印所选路线已定义的关键入口，不构造环境、不读正式权重、不调用训练。C0/C1 的工作台配置差异由冻结任务配置决定；不能只改 `ROUTE` 就复现已结案训练。I0 有 40 万宏步上限和 12500 回合上限，实际完成 397112 宏步；不能写成跑满 40 万。C1 的教师生成与训练在线交互分开计账。若要做新正式实验，需使用工作台任务流程，并另行核对当前授权与资源。


In [27]:
if ROUTE in {"C0", "C1"}:
    from satellite_rl.agents.workbench_algorithms import WorkbenchAgent, Replay
    from satellite_rl.workbench.environment import WorkbenchEnv
    from satellite_rl.workbench.runner import _training
    print("C0/C1 定义可用:", WorkbenchEnv.__name__, Replay.__name__, WorkbenchAgent.__name__, _training.__name__)
else:
    from satellite_rl.sac_improvement.environment import ImprovementEnv
    from satellite_rl.sac_improvement.agent import ImprovementSAC
    from satellite_rl.sac_improvement.runner import _train_job
    print("I0 定义可用:", ImprovementEnv.__name__, ImprovementSAC.__name__, _train_job.__name__)
print("只完成定义与源代码核对；没有启动训练。")


C0/C1 定义可用: WorkbenchEnv Replay WorkbenchAgent _training
只完成定义与源代码核对；没有启动训练。
